# Capítulo 11: Seleção de Modelos e Regularização

**Bases 5 — Ciência de Dados** · notebook de aula

Cada célula de código é a mesma do livro e roda na ordem em que aparece — execute de cima para baixo. Versão publicada deste capítulo: [https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap11/index.html](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap11/index.html)

> **Gerado automaticamente a partir dos `.qmd` do livro por `scripts/gerar-notebooks.py`.** Edições feitas aqui se perdem no próximo `make notebooks`; para mudar o conteúdo, edite o `.qmd`.

In [ ]:
# Põe o diretório de trabalho na raiz do projeto — é o que faz
# `from scratch...` e os caminhos `dados/...` funcionarem. No livro isso vem
# do `execute-dir: project` do Quarto; aqui é feito à mão.
#
# No Colab não existe cópia do projeto, então esta célula clona uma. É rápido
# (clone raso) e acontece só na primeira execução da sessão.
import os
import subprocess
import sys

REPO = "https://github.com/BragaD/UnDF-Bases5-CienciaDeDados-202602.git"


def raiz_do_projeto(inicio="."):
    """Sobe os diretórios até achar o `_quarto.yml`. None se não houver."""
    atual = os.path.abspath(inicio)
    while not os.path.exists(os.path.join(atual, "_quarto.yml")):
        pai = os.path.dirname(atual)
        if pai == atual:
            return None
        atual = pai
    return atual


raiz = raiz_do_projeto()
if raiz is None:
    destino = "/content/bases5" if os.path.isdir("/content") else "bases5"
    if not os.path.isdir(destino):
        print("baixando o material da disciplina...")
        subprocess.run(["git", "clone", "--depth", "1", REPO, destino], check=True)
    raiz = raiz_do_projeto(destino)

os.chdir(raiz)
if raiz not in sys.path:
    sys.path.insert(0, raiz)

%matplotlib inline
print("diretório de trabalho:", os.getcwd())

> **📌 Nota**
>
> Este capítulo corresponde ao capítulo 6 de James et al. (2023).

Uma regressão por mínimos quadrados com todas as colunas disponíveis dá um coeficiente a cada uma, inclusive às que não têm relação com a resposta, e esses coeficientes variam mais de uma amostra para outra à medida que o número de colunas se aproxima do número de observações.

**Escolher colunas.** Em `Credit`, para prever `saldo`, a dívida no cartão, a busca completa ajusta todas as combinações das 11 colunas candidatas e fica com o melhor modelo de cada tamanho; a seleção passo a passo custa muito menos e não garante esse melhor modelo. Para decidir o tamanho, ou se corrige o erro de treino pelo número de colunas, com $C_p$ (que, com erros normais, escolhe o mesmo modelo que o AIC na regressão linear), BIC ou $R^2$ ajustado, ou se mede o erro fora do treino com validação cruzada.

**Encolher os coeficientes.** A regularização mantém todas as colunas e encolhe os coeficientes. A regressão ridge soma ao RSS uma penalidade sobre o tamanho dos coeficientes, e como essa penalidade depende da unidade de cada coluna, a ridge deve vir depois da padronização. O lasso troca, na penalidade, o quadrado de cada coeficiente pelo módulo, e com isso põe coeficientes exatamente em zero: também escolhe colunas. Num simulador em que a $f$ verdadeira é conhecida, a ridge aceita um pouco de viés para cortar muita variância e prevê melhor que os mínimos quadrados, e entre ela e o lasso nenhum ganha sempre: depende de quantas colunas importam de fato.

O parâmetro $\lambda$ decide quanto encolher e é escolhido por validação cruzada. Em `Credit`, com muitas linhas para poucas colunas, o $\lambda$ de menor erro encolhe pouco e quase não baixa o erro de validação cruzada dos mínimos quadrados; o que o lasso oferece ali é um modelo com seis das 11 colunas, escolhido pela regra do um desvio (seção 11.2), que prefere o modelo mais simples entre os de erro parecido. Com quase tantas colunas quanto linhas e só duas que importam, o lasso de menor erro dá a essas duas os maiores coeficientes, o que os mínimos quadrados não fazem.

**Combinar as colunas.** A regressão por componentes principais resume as colunas em poucas direções, as componentes principais, e ajusta os mínimos quadrados sobre elas, apostando que as direções em que as colunas mais variam são as que carregam a resposta. Quando as colunas vêm de poucos fatores escondidos e a resposta depende deles, ganha até da ridge; no simulador da ridge e do lasso, fica atrás do melhor entre a ridge e o lasso. Em `Credit`, onde a resposta depende de uma direção de pouca variância, só alcança o erro de validação cruzada dos mínimos quadrados quando usa todas as componentes, e a redução de dimensão não acontece.

Com tantas colunas quanto observações, ou mais, os mínimos quadrados passam por todos os pontos de treino, o $R^2$ de treino chega a 1 com colunas de puro ruído, e $C_p$, BIC e $R^2$ ajustado deixam de valer; a validação cruzada continua valendo. O lasso ainda erra menos que os mínimos quadrados, mas, com muito mais colunas de ruído que observações, fica longe do piso do erro de teste, a variância do ruído, e escolhe colunas que mudam de uma amostra para outra. Escolher, encolher e combinar são três jeitos de restringir os mínimos quadrados, e nenhum se julga pelo erro de treino: cada um vale o que acerta fora do treino.

Ao final deste capítulo, você será capaz de:

- Aplicar a busca completa, feita com um `for`, e a seleção passo a passo para frente e para trás com `SequentialFeatureSelector`, e comparar o custo de cada uma e a garantia de achar o melhor modelo de cada tamanho
- Escolher o tamanho do modelo por $C_p$ (ou AIC), BIC, $R^2$ ajustado ou validação cruzada, refazendo a seleção dentro de cada grupo, e aplicar a regra do um desvio entre tamanhos quase empatados
- Ajustar a regressão ridge num `Pipeline` com `StandardScaler`, ler o caminho dos coeficientes ao longo de $\lambda$ e explicar, pela troca entre viés e variância, por que encolher os coeficientes pode baixar o erro de teste
- Ajustar o lasso, convertendo $\lambda$ no `alpha` do `scikit-learn`, explicar pela forma da região de orçamento por que ele zera coeficientes e a ridge não, e reconhecer em que situação cada um tende a prever melhor
- Escolher $\lambda$ por validação cruzada com `GridSearchCV`, pelo menor erro ou pela regra do um desvio, e julgar quanto o encolhimento ganha sobre os mínimos quadrados conforme a proporção entre linhas e colunas
- Ajustar a regressão por componentes principais com `PCA` dentro de um `Pipeline`, escolher o número de componentes por validação cruzada e reconhecer quando a aposta nas direções de maior variância falha
- Identificar os sintomas de alta dimensão e avaliar um modelo nesse cenário pelo erro fora do treino, sem tomar as colunas escolhidas como prova de relação com a resposta

## Seções

| Seção | Tópico |
|---|---|
| [11.1](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap11/01-selecao-de-subconjuntos.html) | Seleção de Subconjuntos |
| [11.2](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap11/02-escolhendo-o-modelo.html) | Escolhendo o Modelo: Cp, AIC, BIC e Validação |
| [11.3](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap11/03-regressao-ridge.html) | Regressão Ridge |
| [11.4](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap11/04-o-lasso.html) | O Lasso |
| [11.5](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap11/05-escolhendo-o-parametro-de-regularizacao.html) | Escolhendo o Parâmetro de Regularização |
| [11.6](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap11/06-regressao-por-componentes-principais.html) | Regressão por Componentes Principais |
| [11.7](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap11/07-o-que-muda-em-alta-dimensao.html) | O que Muda em Alta Dimensão |

## Seleção de Subconjuntos

> **📌 Nota**
>
> Esta seção corresponde às seções 6.1.1 e 6.1.2 de James et al. (2023).

`Credit` traz a dívida no cartão de cada cliente, `saldo`, e várias colunas que podem ajudar a prevê-la. Uma regressão por mínimos quadrados com todas elas não descarta nenhuma: o ajuste praticamente nunca estima um coeficiente exatamente igual a zero. Isso tem dois custos. Um modelo com colunas que não têm relação com a resposta é mais difícil de ler; e, quando o número de preditores $p$ (as colunas candidatas) se aproxima do número de observações $n$, os coeficientes passam a variar muito de uma amostra para outra e o erro de teste sobe, como a seção 11.7 vai mostrar. Como escolher quais preditores entram?

In [ ]:
import itertools
import math

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.linear_model import LinearRegression

plt.style.use("estilo-figuras.mplstyle")

A resposta mais direta é ficar com algumas colunas e ajustar mínimos quadrados só com elas, o que se chama **seleção de subconjuntos**.

### Todos os modelos

In [ ]:
credit = pd.read_csv("dados/Credit.csv")
X = pd.get_dummies(credit.drop(columns="saldo"), drop_first=True, dtype=float)
y = credit["saldo"]

X.shape, X.columns.tolist()

> **🔧 Função**
>
> **`pd.get_dummies(df, drop_first, dtype)`**: troca cada coluna categórica de `df` por uma indicadora por categoria. `drop_first=True` descarta a da primeira categoria em ordem alfabética, que vira a base, e `dtype=float` grava as indicadoras como 0 e 1 em vez de `False` e `True`.
>
> **`df.columns.tolist()`**: devolve os nomes das colunas como uma lista comum do Python.

São 400 clientes e $p = 11$ colunas candidatas. `saldo` é a dívida média no cartão, em dólares. Entre as numéricas, `renda` está em milhares de dólares, `limite` é o limite de crédito do cliente, em dólares, `pontuacao` é a nota de crédito dele, `cartoes` é o número de cartões, e `idade` e `escolaridade` estão em anos (esta, anos de estudo). As quatro colunas categóricas viram indicadoras, como na seção 8.4: `imovel_proprio`, `estudante` e `casado` ganham uma cada, com `não` como base, e `regiao`, que tem três categorias, ganha duas, com `Leste` como base.

O primeiro jeito de escolher é não escolher nada de antemão: ajustar um modelo para cada combinação possível de colunas e comparar todos.

> **🔷 Conceito**
>
> A **melhor seleção de subconjuntos** (em inglês, *best subset selection*), ou **busca completa**, ajusta uma regressão por mínimos quadrados para cada combinação dos $p$ preditores, em três passos:
>
> 1. $\mathcal{M}_0$ é o **modelo nulo**, sem preditor nenhum: ele prevê a média de $y$ para todas as observações.
> 2. Para cada tamanho $k = 1, 2, \dots, p$, ajuste os $\binom{p}{k}$ modelos com exatamente $k$ preditores e chame de $\mathcal{M}_k$ o que tem o menor RSS (ou, o que dá no mesmo, o maior $R^2$).
> 3. Escolha um só entre $\mathcal{M}_0, \mathcal{M}_1, \dots, \mathcal{M}_p$ por um critério que estime o erro de teste.

$\binom{p}{k}$ é o número de maneiras de escolher $k$ colunas entre $p$; com $p = 11$ e $k = 2$, por exemplo, são $11 \cdot 10 / 2 = 55$ pares. Somando todos os tamanhos, de nenhuma coluna a todas, são $2^p$ modelos: cada coluna entra ou não entra.

In [ ]:
def rss(preditores):
    Xk = X[list(preditores)]
    previsao = LinearRegression().fit(Xk, y).predict(Xk)
    return float(((y - previsao) ** 2).sum())


colunas = X.columns.tolist()
p = len(colunas)
tss = float(((y - y.mean()) ** 2).sum())

linhas = [{"k": 0, "preditores": (), "rss": tss}]
for k in range(1, p + 1):
    for preditores in itertools.combinations(colunas, k):
        linhas.append({"k": k, "preditores": preditores, "rss": rss(preditores)})

modelos = pd.DataFrame(linhas)
modelos["r2"] = 1 - modelos["rss"] / tss

len(modelos), 2**p

> **🔧 Função**
>
> **`itertools.combinations(itens, k)`**: devolve, uma a uma, todas as combinações de `k` elementos de `itens`, sem repetir elemento e sem considerar a ordem. Com as 11 colunas e `k=2`, são os 55 pares.
>
> **`pd.DataFrame(linhas)`**: monta uma tabela a partir de uma lista de dicionários, uma linha por dicionário e uma coluna por chave.
>
> **`LinearRegression().fit(X, y).predict(X)`**: ajusta a regressão por mínimos quadrados com as colunas de `X` e devolve as previsões $\hat y$ para as mesmas linhas, de onde a função `rss` tira o RSS do modelo.

O modelo nulo prevê $\bar y$ para todos, então o RSS dele é a soma dos quadrados de $y_i - \bar y$: o TSS da seção 8.2. Por isso o $R^2 = 1 - \mathrm{RSS}/\mathrm{TSS}$ de cada modelo sai direto do RSS. Fora o modelo nulo, que entra pelo TSS sem ajuste nenhum, os outros 2.047 modelos saem de um laço: para cada tamanho, cada combinação de colunas ajusta um `LinearRegression`.

O passo 2 fica com o melhor de cada tamanho:

In [ ]:
melhores = modelos.loc[modelos.groupby("k")["rss"].idxmin()].set_index("k")
nomes = melhores["preditores"].map(lambda t: ", ".join(sorted(t)) or "(nenhum)")
tabela_melhores = pd.DataFrame({"R²": melhores["r2"].round(4), "preditores": nomes})
tabela_melhores.head(6)

> **🔧 Função**
>
> **`df.groupby("k")["rss"].idxmin()`**: separa as linhas pelo valor de `k` e, em cada grupo, devolve o rótulo da linha de menor `rss`. Usados em `.loc`, esses rótulos recuperam as linhas inteiras, uma por tamanho. **`df.set_index("k")`** faz da coluna `k` o índice da tabela.
>
> **`serie.map(funcao)`**: aplica `funcao` a cada elemento da série. Aqui, cada tupla de nomes de colunas vira um texto só, em ordem alfabética, e a tupla vazia do modelo nulo vira `(nenhum)`.

A tabela mostra os seis primeiros $\mathcal{M}_k$, do modelo nulo ao de cinco preditores. O melhor modelo com uma coluna usa `pontuacao`; o de duas acrescenta `renda`, e o de três, `estudante_sim`. A figura mostra os 2.048 modelos de uma vez, com os $\mathcal{M}_k$ ligados por uma linha.

In [ ]:
# Figura: 'RSS (em cima) e $R^2$ (embaixo) de cada um dos 2.048 modelos de mínimos quadrados que usam um subconjunto das 11 colunas de `Credit`, contra o número de preditores. A linha laranja liga o melhor modelo de cada tamanho, os $\mathcal{M}_k$ do passo 2.'
fig, (ax_rss, ax_r2) = plt.subplots(2, 1, figsize=(6.4, 6.4), sharex=True)
paineis = [
    (ax_rss, modelos["rss"] / 1e6, melhores["rss"] / 1e6,
     "RSS (milhões de dólares²)"),
    (ax_r2, modelos["r2"], melhores["r2"], "$R^2$"),
]
for ax, todos, fronteira, rotulo in paineis:
    ax.scatter(modelos["k"], todos, s=10, color="C0", alpha=0.7, linewidths=0,
               label="cada modelo")
    ax.plot(melhores.index, fronteira, color="C1", marker="o", markersize=4,
            label="melhor de cada tamanho")
    ax.set_ylabel(rotulo)
    ax.set_ylim(bottom=0)
ax_r2.set_ylim(-0.04, 1.02)
ax_r2.set_xticks(range(0, p + 1))
ax_r2.set_xlim(-0.5, p + 0.5)
ax_r2.set_xlabel("número de preditores")
ax_rss.legend(loc="center right", bbox_to_anchor=(1, 0.42))
fig.tight_layout()
plt.show()

> **🔧 Função**
>
> **`plt.subplots(2, 1, sharex=True)`**: dois painéis empilhados que dividem o mesmo eixo horizontal. **`ax.scatter(..., s, alpha, linewidths)`**: `s` é a área de cada ponto, `alpha` a opacidade (pontos sobrepostos escurecem) e `linewidths=0` tira a borda dos pontos. **`ax.legend(loc, bbox_to_anchor)`**: `bbox_to_anchor=(1, 0.42)` é o ponto do painel, em frações da largura e da altura, onde se prende o canto da legenda indicado por `loc`. **`ax.set_ylabel(texto)`** e **`ax.set_xlabel(texto)`** escrevem o rótulo de cada eixo; **`ax.set_ylim(inferior, superior)`** e **`ax.set_xlim(inferior, superior)`** fixam o intervalo mostrado; **`ax.set_xticks(posicoes)`** põe as marcas do eixo horizontal só nas posições dadas, aqui um número inteiro de preditores por marca.

A figura mostra duas coisas. A linha laranja cai depressa até três preditores e quase achata dali em diante: passar do melhor modelo de três colunas para os melhores de quatro ou mais quase não melhora o ajuste. E a nuvem azul se espalha na vertical: com o mesmo número de colunas, há modelos muito melhores que outros, e é essa escolha dentro de cada tamanho que o passo 2 resolve. Os números abaixo medem as duas observações.

In [ ]:
r2_melhor = melhores["r2"]
r2_com_uma = modelos.loc[modelos["k"] == 1, "r2"]
{
    "RSS do melhor nunca sobe": bool((melhores["rss"].diff().dropna() <= 0).all()),
    "ganho de R², 0 a 3 colunas": round(float(r2_melhor[3] - r2_melhor[0]), 4),
    "ganho de R², 3 a 11 colunas": round(float(r2_melhor[p] - r2_melhor[3]), 4),
    "R², 1 coluna: pior, melhor": (round(float(r2_com_uma.min()), 4),
                                   round(float(r2_com_uma.max()), 4)),
}

> **🔧 Função**
>
> **`serie.diff()`**: a diferença entre cada elemento e o anterior; o primeiro fica sem anterior e vira `NaN`, que **`.dropna()`** descarta. Todas as diferenças menores ou iguais a zero querem dizer que a série nunca sobe.

O `True` confirma que a linha laranja do RSS não sobe em nenhum passo. Não poderia ser diferente: entre os modelos com $k + 1$ colunas está $\mathcal{M}_k$ mais uma coluna qualquer, e esse modelo pode, no pior caso, dar coeficiente zero à coluna nova e repetir o ajuste de $\mathcal{M}_k$. O RSS desse modelo é, portanto, no máximo o de $\mathcal{M}_k$, e o RSS de $\mathcal{M}_{k+1}$, o menor entre todos os modelos com $k + 1$ colunas, também é.

O ganho, porém, se concentra no começo: o $R^2$ do melhor modelo sobe 0,9499 do modelo nulo até três preditores, e só mais 0,0052 de três preditores até as 11 colunas. Já a nuvem azul, com uma coluna só, vai de um $R^2$ de 0,0 no pior modelo (zero até a quarta casa decimal) a 0,7458 no melhor, com `pontuacao`.

A nuvem azul também não se espalha por igual: forma faixas, separadas por vãos. O vão mais largo, no painel do $R^2$, separa as faixas abaixo de 0,3 das faixas acima de 0,7; no painel do RSS é o mesmo vão, com os lados trocados, porque RSS alto é $R^2$ baixo. O que decide de que lado do vão cai um modelo é conter ou não ao menos uma das colunas `limite` e `pontuacao`:

In [ ]:
tem_uma = modelos["preditores"].map(lambda t: "limite" in t or "pontuacao" in t)
r2_sem, r2_com = modelos.loc[~tem_uma, "r2"], modelos.loc[tem_uma, "r2"]
{
    "R² máximo sem as duas": round(float(r2_sem.max()), 4),
    "R² mínimo com ao menos uma": round(float(r2_com.min()), 4),
    "faixas separadas": bool(r2_sem.max() < r2_com.min()),
    "tamanhos com as duas faixas": sorted(
        set(modelos.loc[~tem_uma, "k"]) & set(modelos.loc[tem_uma, "k"])
    ),
}

Nenhum modelo sem `limite` e sem `pontuacao` passa de um $R^2$ de 0,2945, e nenhum com ao menos uma delas fica abaixo de 0,7425. Os vãos menores, de cada lado do largo, têm outra coluna por trás, `renda`:

In [ ]:
tem_renda = modelos["preditores"].map(lambda t: "renda" in t)
modelos.groupby(
    [tem_uma.rename("limite ou pontuacao"), tem_renda.rename("renda")]
)["r2"].agg(["min", "max"]).round(4)

> **🔧 Função**
>
> **`df.groupby([serie1, serie2])`**: agrupa as linhas pelas combinações de valores de duas séries do mesmo tamanho que a tabela, aqui dois booleanos; **`serie.rename(nome)`** dá a cada uma o nome que aparece no índice. **`.agg(["min", "max"])`** calcula as duas estatísticas em cada grupo, uma coluna por estatística.

Dos dois lados do vão largo, `renda` separa duas faixas que não se tocam: sem `limite` e sem `pontuacao`, o maior $R^2$ sem `renda` é 0,0774 e o menor com ela, 0,2150; com ao menos uma das duas, 0,8267 sem `renda` e 0,8711 com ela. As colunas `limite` e `pontuacao` vão voltar adiante, quando os métodos discordarem.

O RSS e o $R^2$ de treino servem no passo 2, que compara modelos **do mesmo tamanho**: entre dois modelos com três colunas, o de menor RSS ajusta melhor o treino com a mesma liberdade. No passo 3 não servem, porque escolheriam sempre $\mathcal{M}_p$, aqui $\mathcal{M}_{11}$, o modelo com todas as colunas. O que se quer é o menor erro de **teste**, e o RSS de treino do melhor modelo é justamente o número que nunca sobe quando o modelo ganha flexibilidade, como o MSE de treino da seção 7.6, enquanto o de teste desenhava um U. O passo 3 pede outra régua: o erro de um conjunto de validação ou da validação cruzada (capítulo 10), ou critérios como $C_p$, AIC e BIC (seção 11.2).

Com validação cruzada, o passo 2 também entra na conta. Cada rodada separa um dos grupos para validação, e os $\mathcal{M}_k$ daquela rodada são escolhidos de novo só com as linhas de treino dela; com cinco grupos, saem cinco listas $\mathcal{M}_0, \dots, \mathcal{M}_p$, uma por rodada. Escolher as colunas uma vez só, com todas as linhas, deixaria as linhas de validação influenciarem a escolha: o vazamento da seção 10.6.

### O custo de olhar tudo

Cada coluna nova dobra o número de modelos. Com 11 colunas foram 2.048 modelos; a conta muda de figura quando $p$ cresce.

In [ ]:
def milhar(v):
    return f"{v:,}".replace(",", ".")


tamanhos = [10, 20, 40]
custo = pd.DataFrame(
    {"busca completa": [milhar(2**q) for q in tamanhos]},
    index=pd.Index(tamanhos, name="p"),
)
custo

> **🔧 Função**
>
> **`pd.Index(valores, name)`**: um índice de tabela com os `valores` dados e um nome, que aparece no cabeçalho, acima dos rótulos das linhas. Aqui, `p`.

Com $p = 20$, a busca completa passa de um milhão de ajustes; com $p = 40$, de um trilhão. Quantas vezes isso é a busca que `Credit` acabou de pedir?

In [ ]:
milhar(2**40 // 2**p)

São $2^{40}/2^{11} = 2^{29}$: 536.870.912 vezes os 2.048 modelos ajustados acima, o que torna a busca impraticável.

O custo não é só de tempo. Quanto maior o espaço de busca, maior a chance de encontrar, entre tantos modelos, algum que se ajuste bem ao treino por acaso, sem que as colunas escolhidas tenham relação com a resposta fora dele. Uma busca enorme pode sobreajustar a própria escolha de colunas.

### Um preditor por vez

A **seleção passo a passo para frente** (em inglês, *forward stepwise selection*) troca a busca completa por uma sequência de decisões pequenas. Começa no modelo nulo e, a cada passo, acrescenta a coluna que mais reduz o RSS, sem nunca tirar uma coluna que já entrou.

> **🔷 Conceito**
>
> **Seleção passo a passo para frente:**
>
> 1. $\mathcal{M}_0$ é o modelo nulo.
> 2. Para $k = 0, 1, \dots, p - 1$: entre os $p - k$ modelos que acrescentam **uma** coluna às de $\mathcal{M}_k$, chame de $\mathcal{M}_{k+1}$ o de menor RSS.
> 3. Escolha um só entre $\mathcal{M}_0, \dots, \mathcal{M}_p$ por um critério que estime o erro de teste.

O passo 2 ajusta $p$ modelos na primeira rodada, $p - 1$ na segunda, e assim por diante, até 1 na última. Somando o modelo nulo, são $1 + p(p+1)/2$ modelos considerados, contra os $2^p$ da busca completa:

In [ ]:
custo["para frente"] = [milhar(1 + q * (q + 1) // 2) for q in tamanhos]
custo

Com $p = 40$, os mais de um trilhão de modelos da busca completa viram 821. O laço abaixo faz a seleção para frente em `Credit`, reaproveitando a função `rss`, para expor a ordem em que as colunas entram: `passo_a_passo` guarda, para cada $k$, as colunas de $\mathcal{M}_k$ e o RSS dele, e é dessa ordem que sai a comparação com a busca completa. No dia a dia, quem faz a seleção é o `SequentialFeatureSelector` do `scikit-learn`, que aparece adiante.

In [ ]:
escolhidas = []
passo_a_passo = {0: ((), tss)}
for k in range(1, p + 1):
    candidatas = [c for c in colunas if c not in escolhidas]
    rss_melhor, melhor = min((rss(escolhidas + [c]), c) for c in candidatas)
    escolhidas.append(melhor)
    passo_a_passo[k] = (tuple(escolhidas), rss_melhor)

escolhidas

A lista é a ordem de entrada, uma coluna por passo, e o $\mathcal{M}_k$ da seleção para frente é formado pelas $k$ primeiras. A economia tem preço? O melhor modelo de quatro colunas precisa conter o melhor de três? A tabela compara o que o passo a passo escolhe com os $\mathcal{M}_k$ da busca completa:

In [ ]:
comparacao = pd.DataFrame(
    {
        "igual à busca": [
            set(passo_a_passo[k][0]) == set(melhores.loc[k, "preditores"])
            for k in range(1, 5)
        ],
        "para frente": [", ".join(sorted(passo_a_passo[k][0])) for k in range(1, 5)],
    },
    index=pd.Index(range(1, 5), name="k"),
)
comparacao

Os dois métodos concordam nos três primeiros tamanhos: `pontuacao`, depois `renda`, depois `estudante_sim`. No quarto, não: a busca completa (tabela dos $\mathcal{M}_k$, acima) encontra um modelo sem `pontuacao`, com `cartoes`, `estudante_sim`, `limite` e `renda`. A busca completa escolhe o melhor de cada tamanho sem se importar com os tamanhos anteriores; o passo a passo só pode acrescentar, e o que entrou fica. Por isso não tinha como chegar àquele modelo: `pontuacao` entrou no primeiro passo e não sai mais, então o modelo de quatro colunas dele é obrigado a contê-la.

Quanto o passo a passo perde por não poder trocar `pontuacao` por `limite`?

In [ ]:
correlacao = float(X["limite"].corr(X["pontuacao"]))
{
    "correlação entre limite e pontuacao": round(correlacao, 3),
    "R² com 4 colunas, busca completa": round(float(melhores.loc[4, "r2"]), 4),
    "R² com 4 colunas, para frente": round(1 - passo_a_passo[4][1] / tss, 4),
}

> **🔧 Função**
>
> **`serie.corr(outra)`**: a correlação entre duas séries, de −1 a 1.

`limite` e `pontuacao` correlacionam a 0,997, como na seção 8.6: uma quase repete a outra, e as duas podem ocupar o lugar uma da outra num modelo. Por isso a perda é pequena: o $R^2$ de treino do modelo de quatro colunas é 0,9536 na busca completa e 0,9522 no passo a passo. O passo a passo não **garante** o melhor modelo de cada tamanho; ele custa muito menos, e aqui chegou perto.

### Para trás, com o `scikit-learn`

A seleção pode andar no sentido contrário, partindo de todas as colunas.

> **🔷 Conceito**
>
> **Seleção passo a passo para trás** (em inglês, *backward stepwise selection*):
>
> 1. $\mathcal{M}_p$ é o modelo com as $p$ colunas.
> 2. Para $k = p, p - 1, \dots, 1$: entre os $k$ modelos que tiram **uma** coluna de $\mathcal{M}_k$, chame de $\mathcal{M}_{k-1}$ o de menor RSS. O que saiu não volta.
> 3. Escolha um só entre $\mathcal{M}_0, \dots, \mathcal{M}_p$ por um critério que estime o erro de teste.

A conta é a mesma da seleção para frente: $1 + p(p+1)/2$ modelos considerados.

O `scikit-learn` faz as duas direções com `SequentialFeatureSelector`. Por padrão, ele compara as colunas candidatas pelo $R^2$ médio de uma validação cruzada em cinco grupos, uma régua do tipo do passo 3, aplicada à escolha de cada coluna. Aqui o objetivo é outro: reproduzir o passo 2, que compara pelo RSS de treino. Para isso, basta entregar a ele uma divisão só, com as 400 linhas no treino e na validação: o erro que ele mede passa a ser o MSE de treino, $\mathrm{RSS}/n$, e o menor MSE é o menor RSS. Medir o erro nas mesmas linhas do ajuste não é o uso normal da ferramenta; serve aqui para comparar com o laço.

In [ ]:
todas_as_linhas = np.arange(len(y))


def selecao(direcao, k):
    seletor = SequentialFeatureSelector(
        LinearRegression(),
        n_features_to_select=k,
        direction=direcao,
        scoring="neg_mean_squared_error",
        cv=[(todas_as_linhas, todas_as_linhas)],
    ).fit(X, y)
    return tuple(seletor.get_feature_names_out())


para_frente_sklearn = {k: selecao("forward", k) for k in range(1, 5)}
para_tras = {k: selecao("backward", k) for k in range(1, 5)}
all(set(para_frente_sklearn[k]) == set(passo_a_passo[k][0]) for k in range(1, 5))

> **🔧 Função**
>
> **`SequentialFeatureSelector(estimador, n_features_to_select, direction, scoring, cv)`**: seleciona colunas uma a uma, acrescentando (`direction="forward"`) ou tirando (`"backward"`), até chegar a `n_features_to_select` colunas. Em cada passo, acrescenta ou tira a coluna com que o modelo alcança a melhor pontuação `scoring`, medida nas divisões de `cv`. **`seletor.get_feature_names_out()`** devolve os nomes das colunas escolhidas.
>
> - `scoring="neg_mean_squared_error"`: o MSE com o sinal trocado. O `scikit-learn` sempre procura a **maior** pontuação, então um erro, que se quer pequeno, entra negativo.
> - `cv`: além de um número de grupos, aceita uma lista de pares `(índices de treino, índices de validação)`, uma divisão por par. Aqui, um par só, com `np.arange(len(y))`, os índices de 0 a 399, dos dois lados.

O `True` diz que a versão para frente do `SequentialFeatureSelector` escolhe, em cada um dos quatro primeiros tamanhos, as mesmas colunas do laço. A tabela abaixo põe os três métodos lado a lado, com o $R^2$ de treino de cada modelo e se ele coincide com o da busca completa:

In [ ]:
caminhos = {
    "busca completa": {k: melhores.loc[k, "preditores"] for k in range(1, 5)},
    "para frente": {k: passo_a_passo[k][0] for k in range(1, 5)},
    "para trás": para_tras,
}
tres_metodos = pd.DataFrame([
    {
        "k": k,
        "método": metodo,
        "R²": round(1 - rss(caminho[k]) / tss, 4),
        "igual à busca": set(caminho[k]) == set(melhores.loc[k, "preditores"]),
        "preditores": ", ".join(sorted(caminho[k])),
    }
    for k in range(1, 5)
    for metodo, caminho in caminhos.items()
]).set_index(["k", "método"])
tres_metodos

Com uma coluna, a seleção para trás fica com `limite`, e não com `pontuacao`: o $R^2$ desse modelo é 0,7425, contra 0,7458 do melhor modelo de uma coluna. Com duas e com três colunas é a mesma troca: a seleção para trás tem `limite` onde a busca completa tem `pontuacao`. Com quatro, a seleção para trás chega ao mesmo modelo da busca completa, com o mesmo $R^2$ de 0,9536. O passo a passo para frente não chega a esse modelo.

Ao passar de $\mathcal{M}_5$ para $\mathcal{M}_4$, a seleção para trás descarta `pontuacao`. O chunk abaixo mede o que custaria tirar cada uma das duas nesse momento:

In [ ]:
cinco = selecao("backward", 5)
r2_cinco = 1 - rss(cinco) / tss


def perda_sem(coluna):
    return round(r2_cinco - (1 - rss([c for c in cinco if c != coluna]) / tss), 4)


{
    "M_5 para trás": ", ".join(sorted(cinco)),
    "perda de R² sem pontuacao": perda_sem("pontuacao"),
    "perda de R² sem limite": perda_sem("limite"),
    "M_4 é M_5 sem pontuacao": set(para_tras[4]) == set(cinco) - {"pontuacao"},
}

Com as cinco colunas, tirar `pontuacao` baixa o $R^2$ em 0,0006, e tirar `limite`, em 0,0042: as duas quase se repetem. O `True` da última linha confirma que $\mathcal{M}_4$ é $\mathcal{M}_5$ sem `pontuacao`; como o passo 2 tira a coluna cuja saída deixa o menor RSS, entre as cinco é ela a que menos falta faz. Ela sai, e daí em diante `limite` ocupa o lugar dela em todos os modelos menores. Resta comparar os dois caminhos passo a passo, tamanho a tamanho:

In [ ]:
r2 = tres_metodos["R²"]
{k: bool(r2[(k, "para frente")] > r2[(k, "para trás")]) for k in range(1, 5)}

Para frente fica acima nos tamanhos 1 a 3, para trás no tamanho 4, e nenhum dos dois ganha em todos. Os três métodos dão modelos parecidos, mas não iguais.

Sem `cv` nem `scoring`, o seletor volta à régua padrão, a validação cruzada em cinco grupos:

In [ ]:
normal = SequentialFeatureSelector(LinearRegression(), n_features_to_select=4).fit(X, y)
escolhidas_normal = set(normal.get_feature_names_out())
sorted(escolhidas_normal), escolhidas_normal == set(passo_a_passo[4][0])

Com quatro colunas, ele fica com `estudante_sim`, `limite`, `pontuacao` e `renda`, as mesmas da seleção para frente pelo RSS de treino. Com outra régua, as escolhas poderiam divergir. A validação cruzada interna só decide qual coluna entra em cada passo, e o seletor vê as 400 linhas ao escolher as colunas: o erro de validação de um modelo montado com elas ainda pede o cuidado da seção 10.6.

### Quando há mais colunas que observações

Numa base com $n = 30$ clientes e $p = 40$ colunas, quais dos três métodos ainda podem ser usados, e até que tamanho de modelo?

A seleção para trás nem começa: o primeiro modelo dela usa as 40 colunas, e mínimos quadrados só têm solução única quando $n > p$. Uma reta tem dois coeficientes e passa exatamente por quaisquer dois pontos com $x$ diferentes; do mesmo jeito, com tantos coeficientes quanto observações o ajuste acerta todos os pontos, o RSS de treino vai a zero e não diz nada sobre dado novo, e com mais coeficientes que observações há infinitos ajustes perfeitos, sem como escolher entre eles. As outras duas começam do modelo nulo e esbarram no mesmo limite mais adiante: com 30 observações, um modelo com intercepto tem solução única até 29 colunas, $n - 1$, e as duas param em $\mathcal{M}_{29}$. Esse último modelo tem 30 coeficientes, contando o intercepto, para 30 observações: é ele mesmo um ajuste perfeito, com RSS de treino zero. A lista vai até ele só para ficar completa; quem decide onde parar, bem antes, é o passo 3. Até lá, porém, o custo das duas é muito diferente:

In [ ]:
{
    "busca completa até M_29": milhar(sum(math.comb(40, k) for k in range(30))),
    "para frente até M_29": milhar(1 + sum(40 - k for k in range(29))),
}

> **🔧 Função**
>
> **`math.comb(n, k)`**: o número de combinações de `k` elementos entre `n`, $\binom{n}{k}$.

Mesmo parada em 29 colunas, a busca completa teria de ajustar mais de um trilhão de modelos, e a seleção para frente, 755. As duas condições se somam: com mais colunas que observações, a seleção para trás não começa; com $p$ grande, como 40, a busca completa é inviável pelo custo. Por isso, aqui, só a seleção para frente serve, e só até $\mathcal{M}_{29}$. Há ainda versões **híbridas**, que acrescentam colunas como a versão para frente mas, a cada passo, podem tirar uma que deixou de ajudar, na tentativa de chegar mais perto da busca completa sem pagar o custo dela.

Os três métodos terminam no mesmo lugar: uma lista $\mathcal{M}_0, \mathcal{M}_1, \dots, \mathcal{M}_p$, um modelo por tamanho. A mesma construção vale para a regressão logística do capítulo 9, trocando o RSS por outra medida de ajuste. O que muda entre eles é o número de modelos considerados, a exigência sobre $n$ e a garantia de achar o melhor modelo de cada tamanho:

In [ ]:
pd.DataFrame(
    {
        "modelos": ["2^p", "1 + p(p+1)/2", "1 + p(p+1)/2"],
        f"com p = {p}": [milhar(2**p), milhar(1 + p * (p + 1) // 2),
                         milhar(1 + p * (p + 1) // 2)],
        "exige n > p": ["não", "não", "sim"],
        "garante o melhor": ["sim", "não", "não"],
    },
    index=pd.Index(["busca completa", "para frente", "para trás"], name="método"),
)

Com poucas colunas, como as 11 de `Credit`, a busca completa cabe no orçamento e é a única com garantia; com muitas, fica o passo a passo, para trás quando $n > p$ e para frente em qualquer caso. Em `Credit`, a lista já indica onde olhar: quase todo o ganho de $R^2$ de treino chega até três preditores. Verificar se o erro de teste confirma isso é o passo 3, assunto da seção 11.2.

## Escolhendo o Modelo: Cp, AIC, BIC e Validação

> **📌 Nota**
>
> Esta seção corresponde à seção 6.1.3 de James et al. (2023).

A busca completa em `Credit` deixa doze candidatos: para cada tamanho $k$, o $\mathcal{M}_k$, o melhor modelo com $k$ preditores entre todos os daquele tamanho, de $\mathcal{M}_0$, sem preditor nenhum, a $\mathcal{M}_{11}$, com as 11 colunas. Qual deles usar? O RSS de treino não decide: ele cai quando o modelo ganha colunas, e acabaria escolhendo o modelo com todas. Há duas saídas. A primeira é **corrigir** o erro de treino, somando a ele uma penalidade que cresce com o número de preditores. A segunda é **medir** o erro de teste diretamente, em observações que o ajuste não viu.

In [ ]:
import itertools

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold, train_test_split

plt.style.use("estilo-figuras.mplstyle")

credit = pd.read_csv("dados/Credit.csv")
X = pd.get_dummies(credit.drop(columns="saldo"), drop_first=True, dtype=float)
y = credit["saldo"]

### Corrigir o erro de treino

Os coeficientes de mínimos quadrados são escolhidos para deixar o RSS **de treino** o menor possível. É por isso que o MSE de treino subestima o de teste, e subestima mais quanto mais flexível é o modelo: na seção 7.6, o erro de treino só cai com a flexibilidade, enquanto o de teste desenha um U. Um modelo com mais colunas tem mais liberdade para se ajustar ao ruído do treino, e a correção precisa crescer com o número de preditores.

A função abaixo é o laço da busca completa da seção 11.1: devolve, para cada tamanho $k$, as colunas de $\mathcal{M}_k$ e o RSS de treino dele.

In [ ]:
def melhores_por_tamanho(X, y):
    """Busca completa: para cada tamanho k, as colunas de M_k e o RSS de treino."""
    colunas = X.columns.tolist()
    melhores = {0: ((), float(((y - y.mean()) ** 2).sum()))}
    for k in range(1, len(colunas) + 1):
        for preditores in itertools.combinations(colunas, k):
            Xk = X[list(preditores)]
            previsao = LinearRegression().fit(Xk, y).predict(Xk)
            rss = float(((y - previsao) ** 2).sum())
            if k not in melhores or rss < melhores[k][1]:
                melhores[k] = (preditores, rss)
    linhas = [{"k": k, "preditores": t, "rss": r} for k, (t, r) in melhores.items()]
    return pd.DataFrame(linhas).set_index("k")


n, p = X.shape
melhores = melhores_por_tamanho(X, y)
print(f"{n} clientes, {p} colunas; o RSS nunca sobe: "
      f"{melhores['rss'].is_monotonic_decreasing}")
melhores["rss"].map(lambda r: f"{r:,.0f}".replace(",", ".")).to_frame("RSS")

> **🔧 Função**
>
> **`serie.is_monotonic_decreasing`**: `True` se cada valor da série é menor ou igual ao anterior. Na última linha, o `map` só escreve cada RSS arredondado, com ponto de milhar. **`serie.to_frame(nome)`**: a série como uma tabela de uma coluna, com o título `nome`.

A linha $k = 0$ é o TSS, o RSS do modelo nulo, e o `True` confirma que o RSS nunca sobe de um tamanho para o seguinte: vai de 84.339.912 no modelo nulo a 3.786.730 com as 11 colunas.

O $C_p$ e o BIC, a seguir, precisam de uma estimativa de $\sigma^2$, a variância do erro $\varepsilon$: quanto o `saldo` varia em torno do que as colunas conseguem prever, a parte que nenhum modelo explica. A estimativa usual sai do modelo com todas as colunas: é o quadrado do RSE da seção 8.3, o RSS de $\mathcal{M}_p$ dividido por $n - p - 1$.

In [ ]:
sigma2 = melhores.loc[p, "rss"] / (n - p - 1)
round(float(sigma2), 1), round(float(np.sqrt(sigma2)), 2)

> **🔧 Função**
>
> **`np.sqrt(x)`**: a raiz quadrada de `x`, seja um número, um array ou uma série; numa série, a raiz de cada valor, com o mesmo índice.

$\hat\sigma^2$ vale 9.759,6, e a raiz dele, 98,79 dólares, é o afastamento típico entre o `saldo` de um cliente e a previsão do modelo completo. Com $\hat\sigma^2$, e com $d$ o número de preditores de um modelo, o primeiro critério é o $C_p$.

> **🔷 Conceito**
>
> Para um modelo de mínimos quadrados com $d$ preditores, ajustado em $n$ observações,
>
> $$
> C_p = \frac{1}{n}\left(\mathrm{RSS} + 2d\hat\sigma^2\right).
> $$
>
> O $C_p$ é uma estimativa do MSE de teste, e escolhe-se o modelo de **menor** $C_p$. O **AIC** (critério de informação de Akaike) vale para muitos tipos de modelo. Na regressão linear, supondo que o erro $\varepsilon$ segue uma distribuição normal, a curva em sino, ele tem a mesma fórmula que o $C_p$, a menos de constantes que não mudam a escolha.

O RSS sozinho é otimista, e o $C_p$ soma a ele $2d\hat\sigma^2$: cada coluna a mais custa $2\hat\sigma^2$, e só compensa se reduzir o RSS mais do que isso. O segundo critério tem a mesma forma, com outra penalidade.

> **🔷 Conceito**
>
> $$
> \mathrm{BIC} = \frac{1}{n}\left(\mathrm{RSS} + \log(n)\,d\hat\sigma^2\right).
> $$
>
> O **BIC** (critério de informação bayesiano) não vem de uma estimativa do MSE de teste, e sim de uma aproximação de quão provável cada modelo é, dado o que se observou. Para usá-lo, basta a fórmula: também é o RSS mais uma penalidade por coluna, e também se escolhe o modelo de **menor** valor.

O BIC troca o 2 por $\log n$, o logaritmo natural de $n$. Quando $\log n$ passa de 2, o BIC cobra mais caro por coluna que o $C_p$ e tende a escolher modelos menores. O terceiro critério não soma penalidade nenhuma.

> **🔷 Conceito**
>
> $$
> R^2\ \text{ajustado} = 1 - \frac{\mathrm{RSS}/(n - d - 1)}{\mathrm{TSS}/(n - 1)}.
> $$
>
> No $R^2$ ajustado, escolhe-se o modelo de **maior** valor.

O $R^2$ ajustado divide o RSS por $n - d - 1$, que diminui quando $d$ cresce. Uma coluna que quase não reduz o RSS faz $\mathrm{RSS}/(n - d - 1)$ subir, e o $R^2$ ajustado cair. Os três critérios são uma expressão cada sobre a coluna `rss`:

In [ ]:
d = melhores.index.to_numpy()
rss = melhores["rss"]
tss = rss[0]
criterios = pd.DataFrame({
    "Cp": (rss + 2 * d * sigma2) / n,
    "BIC": (rss + np.log(n) * d * sigma2) / n,
    "R² ajustado": 1 - (rss / (n - d - 1)) / (tss / (n - 1)),
})
escolhas = {
    "Cp": int(criterios["Cp"].idxmin()),
    "BIC": int(criterios["BIC"].idxmin()),
    "R² ajustado": int(criterios["R² ajustado"].idxmax()),
}
round(float(np.log(n)), 2), escolhas

> **🔧 Função**
>
> **`melhores.index.to_numpy()`**: o índice da tabela, os tamanhos de 0 a 11, como array do `numpy`, para entrar na conta como o $d$ de cada linha. **`np.log(x)`**: o logaritmo natural.
>
> **`serie.idxmin()`** e **`serie.idxmax()`**: o rótulo do índice onde a série tem o menor valor e onde tem o maior, respectivamente. Aqui o índice é $k$, e o rótulo é o tamanho do modelo escolhido.

Com $n = 400$, $\log n$ vale 5,99, quase três vezes o 2 do $C_p$. O $C_p$ escolhe o modelo de 6 preditores, o BIC o de 4 e o $R^2$ ajustado o de 7. As colunas de cada um, e se um modelo está contido no outro:

In [ ]:
colunas = {
    criterio: set(melhores.loc[k, "preditores"]) for criterio, k in escolhas.items()
}
encaixados = (colunas["BIC"].issubset(colunas["Cp"])
              and colunas["Cp"].issubset(colunas["R² ajustado"]))
for criterio, c in colunas.items():
    print(f"{criterio}: {', '.join(sorted(c))}")
print("encaixados:", encaixados)

`a.issubset(b)` pergunta se todo elemento do conjunto `a` também está em `b`. O `True` diz que os três modelos são encaixados: o do BIC está contido no do $C_p$, que está contido no do $R^2$ ajustado. O do BIC tem `cartoes`, `estudante_sim`, `limite` e `renda`; o do $C_p$ acrescenta `idade` e `pontuacao`; o do $R^2$ ajustado acrescenta ainda `imovel_proprio_sim`. A figura mostra as três curvas, de 1 a 11 preditores.

In [ ]:
# Figura: '$C_p$, BIC e $R^2$ ajustado do melhor modelo de cada tamanho em `Credit`, de 1 a 11 preditores. O "x" laranja marca o modelo escolhido por cada critério: o menor $C_p$, o menor BIC e o maior $R^2$ ajustado. $C_p$ e BIC estão em milhares de dólares², como um MSE.'
rotulos = {
    "Cp": "$C_p$ (mil dólares²)",
    "BIC": "BIC (mil dólares²)",
    "R² ajustado": "$R^2$ ajustado",
}
fig, eixos = plt.subplots(3, 1, figsize=(6.4, 5.6), sharex=True)
tamanhos = criterios.index[1:]
for ax, nome in zip(eixos, criterios.columns):
    escala = 1 if nome == "R² ajustado" else 1000
    curva = criterios.loc[tamanhos, nome] / escala
    ax.plot(tamanhos, curva, color="C0", marker="o", markersize=4)
    k = escolhas[nome]
    ax.plot(k, curva[k], "x", color="C1", markersize=11, markeredgewidth=2.5,
            label="_nolegend_")
    ax.set_ylabel(rotulos[nome])
eixos[2].yaxis.set_major_formatter(lambda valor, _: f"{valor:.2f}".replace(".", ","))
eixos[2].set_xticks(range(1, 12))
eixos[2].set_xlabel("número de preditores")
fig.tight_layout()
plt.show()

> **🔧 Função**
>
> **`ax.yaxis.set_major_formatter(funcao)`**: escreve cada marca do eixo vertical com a função dada, que recebe o valor e a posição; aqui, com duas casas e vírgula decimal.

As três curvas mudam depressa no começo e quase nada a partir de 4 preditores. Quanto, em cada trecho?

In [ ]:
def mudancas(tabela):
    """Quanto cada coluna muda de 1 a 3, de 3 a 4, e a amplitude de 4 a 11."""
    return pd.DataFrame({
        "de 1 a 3": (tabela.loc[1] - tabela.loc[3]).abs(),
        "de 3 a 4": (tabela.loc[3] - tabela.loc[4]).abs(),
        "de 4 a 11, maior − menor": tabela.loc[4:].max() - tabela.loc[4:].min(),
    }).T


mudancas(criterios).round({"Cp": 1, "BIC": 1, "R² ajustado": 4})

> **🔧 Função**
>
> **`tabela.loc[4:]`**: as linhas de rótulo 4 em diante; **`.max()`** e **`.min()`**, numa tabela, o maior e o menor valor de cada coluna. **`.abs()`**: o valor absoluto, para a queda do $C_p$ e a subida do $R^2$ ajustado saírem positivas. **`.T`**: a tabela transposta, com as linhas no lugar das colunas. **`df.round({coluna: casas})`**: arredonda cada coluna com o seu número de casas.

De 1 a 3 preditores, o $C_p$ cai 42.922,2 dólares²; de 4 a 11, a distância entre o maior e o menor valor é de só 156,8. O BIC e o $R^2$ ajustado têm o mesmo desenho na tabela: uma mudança grande de 1 a 3 e uma variação pequena de 4 a 11. Todo modelo de 4 a 11 preditores tem um $C_p$ parecido, e a escolha exata entre eles pesa pouco. O BIC, que cobra mais por coluna, é o que fica com o modelo menor.

Os três critérios dependem da contagem $d$ de preditores, e o $C_p$ e o BIC dependem também de $\hat\sigma^2$. Num método sem um número claro de parâmetros, como o *k*-NN da seção 7.3, falta o $d$ para pôr na fórmula.

### Medir o erro de teste

A alternativa é a do capítulo 10: separar observações que o ajuste não vê e medir nelas o erro de cada tamanho, por conjunto de validação ou por validação cruzada. Não depende de $\hat\sigma^2$ nem de contar parâmetros, e estima o erro de teste diretamente.

Há um cuidado, o mesmo da seção 10.6. Os $\mathcal{M}_k$ da tabela `melhores` foram escolhidos olhando os 400 clientes. Validar esses modelos em parte desses clientes deixaria as linhas de validação influenciarem a escolha das colunas: é vazamento. A busca completa tem de ser **refeita só com as linhas de treino**, e o $\mathcal{M}_k$ de um treino pode ser diferente do de outro. A função abaixo faz isso: recebe um treino e uma validação, refaz a busca no treino e mede, na validação, o MSE do melhor modelo de cada tamanho.

In [ ]:
def erros_de_validacao(X_treino, y_treino, X_val, y_val):
    """MSE de validação do M_k de cada tamanho, escolhido só com o treino."""
    erros = []
    busca = melhores_por_tamanho(X_treino, y_treino)
    for k, preditores in busca["preditores"].items():
        if k == 0:
            previsao = np.full(len(y_val), y_treino.mean())
        else:
            colunas = list(preditores)
            modelo = LinearRegression().fit(X_treino[colunas], y_treino)
            previsao = modelo.predict(X_val[colunas])
        erros.append(float(((y_val - previsao) ** 2).mean()))
    return pd.Series(erros, index=pd.Index(range(p + 1), name="k"))


X_treino, X_val, y_treino, y_val = train_test_split(
    X, y, test_size=0.25, random_state=11
)
validacao = erros_de_validacao(X_treino, y_treino, X_val, y_val)
len(y_treino), len(y_val), int(validacao.idxmin())

> **🔧 Função**
>
> **`train_test_split(X, y, test_size, random_state)`**: sorteia quais linhas vão para cada lado e devolve, nesta ordem, `X` de treino, `X` de validação, `y` de treino e `y` de validação. `test_size=0.25` manda um quarto das linhas para a validação, e `random_state=11` é a semente do sorteio.
>
> **`np.full(tamanho, valor)`**: um array de `tamanho` posições, todas iguais a `valor`. É a previsão do modelo nulo: a média de `saldo` no treino, para todos os clientes da validação.
>
> **`pd.Series(valores, index)`**: uma série com os valores e o índice dados; aqui, um MSE por tamanho, com o índice $k$ de 0 a 11.

Com 300 clientes no treino e 100 na validação, o menor erro de validação é o do modelo de 6 preditores. Uma única partição depende muito do sorteio; a validação cruzada em 10 grupos repete a conta 10 vezes, cada grupo servindo uma vez de validação, e tira a média. Cada grupo refaz a busca completa no seu treino. A função abaixo recebe a semente, para a mesma conta servir a outros sorteios adiante. A célula demora a terminar, e o tempo depende da máquina: espere a saída aparecer.

In [ ]:
def erros_por_grupo(semente):
    """MSE de validação de cada tamanho em cada um dos 10 grupos, um por linha."""
    grupos = KFold(10, shuffle=True, random_state=semente)
    return pd.DataFrame([
        erros_de_validacao(X.iloc[treino], y.iloc[treino], X.iloc[val], y.iloc[val])
        for treino, val in grupos.split(X)
    ])


erros = erros_por_grupo(11)
cv = erros.mean()
{
    "ajustes nas buscas": 10 * (2**p - 1),
    "tamanho escolhido": int(cv.idxmin()),
}

> **🔧 Função**
>
> **`KFold(n_splits, shuffle, random_state)`**: o esquema de divisão do *k*-fold, aqui com 10 grupos, linhas embaralhadas antes da divisão e a semente recebida pela função. **`grupos.split(X)`** devolve, uma divisão por vez, o par (posições de treino, posições de validação), que **`X.iloc[posicoes]`** usa para recortar as linhas pela posição.
>
> **`erros.mean()`**: numa tabela, a média de cada coluna. `erros` tem uma linha por grupo e uma coluna por tamanho, e `cv` fica com o erro de validação cruzada de cada tamanho.

São 10 buscas completas, uma por grupo, com 20.470 ajustes só nelas. A validação cruzada também escolhe o modelo de 6 preditores. Para pôr lado a lado o BIC e os dois erros medidos fora do treino, a figura usa a raiz de cada um, que fica na unidade de `saldo`, em dólares.

In [ ]:
# Figura: 'O BIC e dois erros medidos fora do treino, para o melhor modelo de cada tamanho em `Credit`, de 1 a 11 preditores, todos como raiz, em dólares. De cima para baixo: a raiz do BIC; a raiz do MSE de validação, com 300 clientes no treino e 100 na validação; e a raiz do erro de validação cruzada em 10 grupos. O "x" laranja marca o mínimo de cada curva. Nos dois de baixo, os modelos de cada tamanho foram escolhidos de novo em cada treino.'
estimativas = {
    "raiz do BIC": np.sqrt(criterios["BIC"]),
    "validação": np.sqrt(validacao),
    "validação cruzada": np.sqrt(cv),
}
fig, eixos = plt.subplots(3, 1, figsize=(6.4, 5.6), sharex=True, sharey=True)
for ax, (nome, curva) in zip(eixos, estimativas.items()):
    curva = curva[1:]
    ax.plot(curva.index, curva, color="C0", marker="o", markersize=4)
    k = curva.idxmin()
    ax.plot(k, curva[k], "x", color="C1", markersize=11, markeredgewidth=2.5,
            label="_nolegend_")
    ax.set_title(nome)
eixos[1].set_ylabel("raiz, em dólares")
eixos[2].set_xticks(range(1, 12))
eixos[2].set_xlabel("número de preditores")
fig.tight_layout()
plt.show()

> **🔧 Função**
>
> **`sharey=True`**, em `plt.subplots`: os painéis dividem também o eixo vertical, e a mesma altura quer dizer o mesmo erro nos três. **`ax.set_title(texto)`**: o título escrito acima de cada painel.

As três curvas têm a mesma forma. Nos mesmos trechos da tabela dos critérios:

In [ ]:
print("menor raiz do erro de validação cruzada:", round(float(np.sqrt(cv).min()), 1))
mudancas(pd.DataFrame(estimativas)).round(1)

De 1 a 3 preditores, a raiz cai 126,9 dólares no BIC, 120,2 na validação e 126,6 na validação cruzada. De 4 a 11, a distância entre o maior e o menor valor é de 3,4, 1,7 e 1,2 dólares, na mesma ordem: pouco, perto de um erro típico de previsão de 100,0 dólares, a menor raiz do erro de validação cruzada. Se o sorteio da validação fosse outro, ou os grupos da validação cruzada fossem outros, o mínimo cairia no mesmo tamanho?

### A regra do um desvio

A resposta se mede repetindo as duas contas com as sementes 12 e 13. Na validação cruzada, são mais 20 buscas completas, e esta célula também demora.

In [ ]:
k_minimo = int(cv.idxmin())
escolhidos = {
    "validação": {11: int(validacao.idxmin())},
    "validação cruzada": {11: k_minimo},
}
for semente in [12, 13]:
    Xt, Xv, yt, yv = train_test_split(X, y, test_size=0.25, random_state=semente)
    menor = erros_de_validacao(Xt, yt, Xv, yv).idxmin()
    escolhidos["validação"][semente] = int(menor)
    cv_semente = erros_por_grupo(semente).mean()
    escolhidos["validação cruzada"][semente] = int(cv_semente.idxmin())
escolhidos

Um conjunto de validação muda de ideia conforme o sorteio: com as sementes 11, 12 e 13, o menor erro fica em 6, 11 e 5 preditores. A validação cruzada, que tira a média de 10 grupos, escolhe 6 nas três.

A escolha se repete, mas isso não faz do modelo de 6 um modelo melhor que o de 4: a curva é quase plana de 4 a 11, e os 400 clientes são uma amostra. Com outros clientes, a curva inteira seria outra, e uma diferença pequena entre dois tamanhos pode não se repetir. A **regra do um desvio** transforma essa cautela numa escolha.

> **🔷 Conceito**
>
> **Regra do um desvio.** Para cada tamanho $k$, seja $s_k$ o desvio padrão dos erros dos 10 grupos.
>
> 1. Ache o tamanho $k^*$ de menor erro de validação cruzada.
> 2. Some a esse erro mínimo a margem $s_{k^*}/\sqrt{10}$. O resultado é o **teto**.
> 3. Escolha o **menor** tamanho cujo erro de validação cruzada não passe do teto.

A margem é uma medida grosseira de quanto a média dos 10 erros de grupo mudaria com outra amostra de clientes. Se os 10 erros de grupo fossem medidas independentes, $s_k/\sqrt{10}$ estimaria o desvio padrão da média deles; não são, porque os treinos de dois grupos quaisquer têm quase todas as linhas em comum, e a margem serve só de régua. Dentro dela, os modelos não se distinguem com segurança, e a regra fica com o mais simples: menos colunas para coletar e para manter.

In [ ]:
desvio = erros.std(ddof=1) / np.sqrt(10)
teto = cv[k_minimo] + desvio[k_minimo]
k_regra = int(cv[cv <= teto].index.min())
modelo_da_regra = set(melhores.loc[k_regra, "preditores"])
{
    "raiz do teto": round(float(np.sqrt(teto)), 2),
    "tamanho pela regra": k_regra,
    "colunas": ", ".join(sorted(modelo_da_regra)),
    "igual ao modelo do BIC": modelo_da_regra == colunas["BIC"],
    "raiz do erro, 1 a 3 preditores": np.sqrt(cv[1:4]).round(1).tolist(),
    "tamanhos abaixo do teto": cv[cv <= teto].index.tolist(),
}

> **🔧 Função**
>
> **`erros.std(ddof=1)`**: o desvio padrão de cada coluna, com $n - 1$ no denominador (`ddof=1`). **`cv[cv <= teto]`** fica só com os tamanhos cujo erro não passa do teto, e **`.index.min()`** pega o menor deles. **`serie.round(casas)`**: arredonda cada valor com o número de casas dado.

A raiz do teto é 103,19 dólares, e todos os tamanhos de 4 a 11 ficam no teto ou abaixo dele; o de 3, com 105,5, fica acima. A regra escolhe então o modelo de 4 preditores. As colunas vêm da tabela `melhores`, a busca completa com os 400 clientes: a validação cruzada decide só o tamanho, e o modelo final é o melhor daquele tamanho no conjunto todo. O `True` diz que é o mesmo modelo que o BIC escolheu, com `cartoes`, `estudante_sim`, `limite` e `renda`.

A figura começa em 3 preditores: com 1 e 2, a raiz do erro é de 232,2 e 163,3 dólares, e as duas alturas achatariam o resto da curva.

In [ ]:
# Figura: 'Raiz do erro de validação cruzada em 10 grupos contra o número de preditores, em `Credit`, de 3 a 11. A faixa azul vai da raiz de (erro − $s_k/\sqrt{10}$) à raiz de (erro + $s_k/\sqrt{10}$) em cada tamanho. A linha tracejada é o teto da regra: o mínimo mais a margem do tamanho do mínimo. O círculo cinza marca o mínimo, e o "x" laranja, o tamanho que a regra escolhe, o menor que não passa do teto.'
tamanhos = cv.index[3:]
abaixo = np.sqrt(cv[tamanhos] - desvio[tamanhos])
acima = np.sqrt(cv[tamanhos] + desvio[tamanhos])
fig, ax = plt.subplots(figsize=(6.4, 3.8))
ax.fill_between(tamanhos, abaixo, acima, color="C0", alpha=0.3, linewidth=0)
ax.plot(tamanhos, np.sqrt(cv[tamanhos]), color="C0", marker="o", markersize=4)
ax.axhline(np.sqrt(teto), color="0.5", linestyle="--", linewidth=1.6)
ax.plot(k_minimo, np.sqrt(cv[k_minimo]), "o", color="0.5", markersize=11,
        markerfacecolor="none", markeredgewidth=1.8, label="_nolegend_")
ax.plot(k_regra, np.sqrt(cv[k_regra]), "x", color="C1", markersize=11,
        markeredgewidth=2.5, label="_nolegend_")
ax.annotate("teto", xy=(11.3, np.sqrt(teto)), xytext=(0, 3),
            textcoords="offset points", ha="left", va="bottom")
fio = {"arrowstyle": "-", "color": "0.5", "linewidth": 1.0, "shrinkB": 7}
for texto, k in [("mínimo", k_minimo), ("regra", k_regra)]:
    ax.annotate(texto, xy=(k, np.sqrt(cv[k])), xytext=(k, abaixo.min() - 1.5),
                ha="center", va="top", arrowprops=fio)
ax.set_ylim(abaixo.min() - 5, None)
ax.yaxis.set_major_formatter(lambda valor, _: f"{valor:.1f}".replace(".", ","))
ax.set_xticks(tamanhos)
ax.set_xlim(2.5, 12.2)
ax.set_xlabel("número de preditores")
ax.set_ylabel("raiz do erro de validação\ncruzada (dólares)")
fig.tight_layout()
plt.show()

> **🔧 Função**
>
> **`ax.fill_between(x, abaixo, acima, alpha)`**: pinta a faixa entre as curvas `abaixo` e `acima`; `alpha` é a opacidade.
>
> **`ax.axhline(y, linestyle)`**: uma linha horizontal na altura `y`, de uma ponta à outra do eixo.
>
> **`ax.annotate(texto, xy, xytext, textcoords, arrowprops)`**: escreve `texto` no ponto `xy` ou, se houver `xytext`, nessa outra posição, ligada a `xy` pelo fio descrito em `arrowprops`. Com `textcoords="offset points"`, `xytext` é um deslocamento em pontos tipográficos a partir de `xy`.
>
> **`ax.set_ylim(baixo, None)`**: fixa o limite de baixo do eixo vertical e deixa o de cima automático, para os rótulos caberem abaixo da faixa.

O "x" está no primeiro ponto que não passa da linha tracejada, 2 tamanhos à esquerda do mínimo. Todas as escolhas da seção, juntas num dicionário só (`a | b` junta os dicionários `a` e `b`):

In [ ]:
todas = escolhas | {
    "validação": int(validacao.idxmin()),
    "validação cruzada": k_minimo,
    "regra do um desvio": k_regra,
}
todas, min(todas.values()), max(todas.values())

As seis escolhas vão de 4 a 7 preditores, e a diferença entre elas pesa pouco: nesses tamanhos, a raiz do erro de validação cruzada fica entre a menor raiz, 100,0 dólares, e a raiz do teto, 103,19: uma faixa de pouco mais de 3 dólares.

Depois de escolhido o $\mathcal{M}_k$ de cada tamanho, o $C_p$ e o BIC custam só o RSS dele, que a busca já calculou, mas exigem o $d$ e o $\hat\sigma^2$. O $\hat\sigma^2$ sai do modelo com todas as colunas, e o denominador $n - p - 1$ só é positivo quando $n > p + 1$, isto é, quando há mais observações do que coeficientes a estimar, contando o intercepto. A validação cruzada não pede nenhum dos dois, mas refaz a busca completa em cada grupo.

Os critérios podem partir dos $\mathcal{M}_k$ escolhidos nos 400 clientes porque corrigem o erro de treino em vez de medi-lo fora. A correção, porém, conta só os $d$ preditores do modelo, e não a busca entre as combinações que o escolheu: mais um motivo para não levar a sério diferenças pequenas entre os tamanhos. A validação cruzada mede em linhas que o ajuste não viu, e elas só continuam de fora se não ajudaram a escolher as colunas.

Qual usar, então? A tabela junta o que cada caminho faz com o erro de treino, o tamanho que escolheu em `Credit` e o que precisa para ser calculado.

| Caminho | O que faz com o erro de treino | Escolheu em `Credit` | Precisa de $d$ e $\hat\sigma^2$? |
|---|---|---|---|
| $C_p$ | soma $2d\hat\sigma^2$ ao RSS | 6 | os dois |
| BIC | soma $\log(n)\,d\hat\sigma^2$ ao RSS | 4 | os dois |
| $R^2$ ajustado | divide o RSS por $n - d - 1$ | 7 | só o $d$ |
| validação cruzada | mede o erro fora do treino | 6; com a regra do um desvio, 4 | nenhum |

Quando dá para contar $d$ e estimar $\hat\sigma^2$, o $C_p$ mira o erro de teste, e o BIC, com a penalidade maior, prefere o modelo enxuto; o $R^2$ ajustado, o de justificativa teórica mais fraca dos três (James et al., 2023), ficou aqui com o maior modelo. Quando falta o $d$ ou o $\hat\sigma^2$, ou quando se quer medir em vez de corrigir, fica a validação cruzada, e entre tamanhos quase empatados a regra do um desvio fica com o menor.

## Regressão Ridge

> **📌 Nota**
>
> Esta seção corresponde à seção 6.2.1 de James et al. (2023).

A seleção de subconjuntos decide coluna a coluna: cada uma entra no modelo inteira ou fica de fora. Existe um meio-termo. O modelo fica com as 11 colunas de `Credit`, mas o ajuste paga um preço por coeficientes grandes, e isso puxa todos na direção de zero. Um modelo assim ajusta o treino pior que os mínimos quadrados, de propósito. O que se ganha com isso?

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.lines import Line2D
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

plt.style.use("estilo-figuras.mplstyle")

credit = pd.read_csv("dados/Credit.csv")
X = pd.get_dummies(credit.drop(columns="saldo"), drop_first=True, dtype=float)
y = credit["saldo"]

### Mínimos quadrados com uma penalidade

Os mínimos quadrados escolhem $\beta_0, \beta_1, \dots, \beta_p$ para deixar o RSS o menor possível:

$$
\mathrm{RSS} = \sum_{i=1}^{n}\Big(y_i - \beta_0 - \sum_{j=1}^{p}\beta_j x_{ij}\Big)^2.
$$

A regressão ridge soma ao RSS um termo que cresce com o tamanho dos coeficientes.

> **🔷 Conceito**
>
> Os coeficientes da **regressão ridge**, $\hat\beta^R$, são os valores que minimizam
>
> $$
> \mathrm{RSS} + \lambda\sum_{j=1}^{p}\beta_j^2,
> $$
>
> em que $\lambda \ge 0$ é um **parâmetro de ajuste**, escolhido à parte. O segundo termo, $\lambda\sum_j\beta_j^2$, é a **penalidade de encolhimento**: ele é pequeno quando $\beta_1, \dots, \beta_p$ estão perto de zero, e por isso puxa as estimativas para zero. O intercepto $\beta_0$ fica fora da penalidade.

Os dois termos brigam. O RSS quer os coeficientes que melhor ajustam o treino; a penalidade quer coeficientes pequenos, e $\lambda$ decide quanto pesa cada lado. Com $\lambda = 0$, a penalidade some e a ridge devolve os mínimos quadrados. Quando $\lambda$ cresce sem limite, só coeficientes cada vez mais perto de zero compensam, e o modelo tende ao modelo nulo, que prevê a média de $y$ para todos os clientes. No meio, cada valor de $\lambda$ dá um conjunto de coeficientes diferente, $\hat\beta^R_\lambda$.

O intercepto fica de fora porque ele não mede a relação de coluna nenhuma com a resposta: é só o valor previsto quando todas as colunas valem zero.

A tabela abaixo traz os coeficientes da ridge em `Credit` com três valores de $\lambda$; a coluna $\lambda = 0$ traz os mínimos quadrados:

In [ ]:
modelos = {"0": LinearRegression().fit(X, y)}
for lam in [1_000, 100_000]:
    modelos[f"{lam:,}".replace(",", ".")] = Ridge(alpha=lam).fit(X, y)
coeficientes = pd.DataFrame(
    {nome: modelo.coef_ for nome, modelo in modelos.items()}, index=X.columns
)
coeficientes.columns.name = "λ"
coeficientes.style.set_uuid("coef").format(decimal=",", thousands=".", precision=3)

> **🔧 Função**
>
> - **`Ridge(alpha)`**: a regressão ridge. O `alpha` do `scikit-learn` é o $\lambda$ da fórmula: `Ridge(alpha=lam)` minimiza $\mathrm{RSS} + \lambda\sum_j\beta_j^2$, sem penalizar o intercepto.
> - **`modelo.coef_`**: depois do `fit`, os coeficientes estimados $\hat\beta_1, \dots, \hat\beta_p$, na ordem das colunas de `X`.
> - **`df.columns.name`**: o nome do conjunto de colunas, que a tabela mostra no canto do cabeçalho; **`serie.index.name`** faz o mesmo com as linhas.
> - **`df.style.format(decimal, thousands, precision)`**: mostra a tabela com o separador decimal, o de milhar e o número de casas dados, sem mudar os números guardados. O **`.set_uuid(nome)`** antes dele fixa o identificador que a tabela recebe no HTML, que de outro modo é sorteado a cada execução.

Com $\lambda$ crescendo, os coeficientes mudam em ritmos muito diferentes. O de `estudante_sim` cai de 425,747 nos mínimos quadrados para 14,538 com $\lambda = 1.000$. O de `renda` quase não se mexe nesse trecho: vai de −7,803 a −7,558. Ser estudante pesa menos no `saldo` que a renda, ou há outro motivo?

O preço aparece no treino. O chunk abaixo calcula o RSS de treino dos mesmos três ajustes e imprime também o formato de `X`, $n \times p$:

In [ ]:
rss = pd.Series(
    {nome: ((y - modelo.predict(X)) ** 2).sum() for nome, modelo in modelos.items()}
)
print("linhas e colunas de X:", X.shape)
print("o RSS de treino só sobe com λ:", bool(np.all(np.diff(rss) > 0)))
rss.index.name = "λ"
pd.DataFrame({"RSS de treino": rss}).style.set_uuid("rss").set_table_attributes(
    'style="width: auto"'
).format(thousands=".", precision=0)

> **🔧 Função**
>
> - **`np.diff(arr)`**: a diferença entre cada elemento e o anterior; **`np.all(mascara)`**: `True` se todos forem `True`.
> - **`.set_table_attributes(atributos)`**: acrescenta atributos HTML à tabela; aqui, `width: auto` a deixa só da largura do conteúdo.

São 400 clientes e 11 colunas. O RSS de treino sobe a cada $\lambda$ da tabela, de 3.786.730 nos mínimos quadrados para 11.847.402 com $\lambda = 100.000$: nenhum outro conjunto de coeficientes tem RSS de treino menor que o dos mínimos quadrados, e a ridge abre mão disso.

### A escala importa

Os coeficientes da primeira tabela estão em unidades diferentes. O de `renda` é a variação de `saldo` para mil dólares a mais de renda, com as outras colunas fixas; o de `limite`, para um dólar a mais de limite; o de `estudante_sim`, para a diferença entre ser e não ser estudante, uma coluna que só vale 0 ou 1. A penalidade soma os quadrados desses números como se fossem comparáveis. Uma coluna que varia pouco na sua unidade precisa de um coeficiente grande para fazer efeito, e esse coeficiente grande é o que a penalidade mais cobra.

Nos mínimos quadrados, a unidade não muda o ajuste: medir $X_j$ numa unidade $c$ vezes maior divide os valores da coluna por $c$ e multiplica $\hat\beta_j$ por $c$, e o produto $X_j\hat\beta_j$ e, com ele, as previsões ficam iguais. Na ridge, não.

A saída é **padronizar** as colunas antes da ridge. Cada coluna é centrada e dividida pelo seu desvio padrão:

$$
\tilde x_{ij} = \frac{x_{ij} - \bar x_j}{\sqrt{\frac{1}{n}\sum_{i=1}^{n}(x_{ij} - \bar x_j)^2}},
$$

e todas passam a ter desvio padrão 1. Numa coluna padronizada, o coeficiente é o **efeito de um desvio padrão**: a variação de `saldo` quando a coluna sobe um desvio padrão. Numa coluna crua, o mesmo efeito é o coeficiente vezes o desvio padrão da coluna.

O teste é medir `renda` em milhões de dólares em vez de milhares, dividindo a coluna por 1.000, e comparar três ajustes, cada um feito nas duas unidades:

1. os mínimos quadrados;
2. a ridge com $\lambda = 100$ nas colunas cruas;
3. a ridge com $\lambda = 100$ nas colunas padronizadas.

In [ ]:
X_milhoes = X.assign(renda=X["renda"] / 1000)
j = X.columns.get_loc("renda")  # a posição de renda nas colunas de X
dp_milhares = X["renda"].std(ddof=0)
dp_milhoes = X_milhoes["renda"].std(ddof=0)

mq_milhares = LinearRegression().fit(X, y)
mq_milhoes = LinearRegression().fit(X_milhoes, y)
crua_milhares = Ridge(alpha=100).fit(X, y)
crua_milhoes = Ridge(alpha=100).fit(X_milhoes, y)
padr_milhares = make_pipeline(StandardScaler(), Ridge(alpha=100)).fit(X, y)
padr_milhoes = make_pipeline(StandardScaler(), Ridge(alpha=100)).fit(X_milhoes, y)

> **🔧 Função**
>
> - **`df.assign(coluna=valores)`**: uma cópia de `df` com a coluna trocada (ou acrescentada); `X` fica como estava.
> - **`X.columns.get_loc(nome)`**: a posição da coluna `nome` em `X`, a mesma do coeficiente dela em `coef_`.
> - **`serie.std(ddof=0)`**: o desvio padrão com $n$ no denominador, o da fórmula acima.

> **🔧 Função**
>
> - **`StandardScaler()`**: subtrai de cada coluna a sua média e divide pelo seu desvio padrão (com $n$ no denominador), deixando-a com média 0 e desvio padrão 1.
> - **`make_pipeline(StandardScaler(), Ridge(alpha=100))`**: encadeia a padronização e a ridge num objeto só. `fit` padroniza cada coluna com as médias e os desvios das linhas que recebe e ajusta a ridge sobre o resultado; `predict` repete a mesma padronização antes de prever.
> - **`modelo[-1]`**: o último passo do `Pipeline`, a ridge, de onde sai o `coef_` na escala padronizada.

A tabela tem uma linha por ajuste. As duas primeiras colunas são o efeito de um desvio padrão de `renda`, com a coluna em milhares e em milhões; a terceira é a maior diferença, sobre os 400 clientes, entre a previsão feita com a coluna em milhares e a feita com ela em milhões.

In [ ]:
escala = pd.DataFrame(
    {
        "em milhares": [
            mq_milhares.coef_[j] * dp_milhares,
            crua_milhares.coef_[j] * dp_milhares,
            padr_milhares[-1].coef_[j],
        ],
        "em milhões": [
            mq_milhoes.coef_[j] * dp_milhoes,
            crua_milhoes.coef_[j] * dp_milhoes,
            padr_milhoes[-1].coef_[j],
        ],
        "diferença máxima": [
            np.abs(mq_milhares.predict(X) - mq_milhoes.predict(X_milhoes)).max(),
            np.abs(crua_milhares.predict(X) - crua_milhoes.predict(X_milhoes)).max(),
            np.abs(padr_milhares.predict(X) - padr_milhoes.predict(X_milhoes)).max(),
        ],
    },
    index=["mínimos quadrados", "ridge crua", "ridge padronizada"],
)
dp_estudante = X["estudante_sim"].std(ddof=0)
print("desvio padrão de estudante_sim:", f"{dp_estudante:.3f}".replace(".", ","))
escala.style.set_uuid("escala").format(decimal=",", thousands=".", precision=2)

> **🔧 Função**
>
> **`np.abs(arr)`**: o valor absoluto de cada elemento; **`arr.max()`**: o maior elemento.

Com as colunas cruas, a ridge com $\lambda = 100$ dá a `renda` um efeito de −268,38 dólares por desvio padrão quando ela está em milhares, e de −0,48 quando está em milhões: a penalidade praticamente zerou a coluna só porque a unidade mudou. As previsões dos dois ajustes chegam a diferir em 578,45 dólares para o mesmo cliente.

Nos mínimos quadrados e na ridge padronizada, a unidade não muda nada: o efeito é o mesmo nas duas colunas, e a maior diferença nas previsões aparece como 0,00, menos de meio centavo. O efeito na ridge padronizada é outro número, −94,60 contra os −274,67 dos mínimos quadrados, cerca de um terço (94,60 ÷ 274,67 ≈ 0,34): padronizadas, todas as colunas pagam a penalidade na mesma moeda, e $\lambda = 100$ já puxa bastante o coeficiente de `renda` para zero.

O coeficiente de `estudante_sim` encolheu depressa porque era um número grande, e era grande em boa parte por causa da unidade: numa coluna que só vale 0 ou 1, um desvio padrão é menos de uma unidade, e o coeficiente por unidade sai grande. A velocidade do encolhimento não diz que ser estudante pesa menos no `saldo` que a renda.

> **⚠️ Atenção — Ridge só depois de padronizar**
>
> Nas colunas cruas, o quanto a ridge encolhe cada coeficiente depende da unidade em que a coluna foi medida. A ridge se aplica **depois** de padronizar as colunas, inclusive as indicadoras, e toda ridge daqui em diante é um `make_pipeline(StandardScaler(), Ridge(alpha=lam))`.

Padronizar uma indicadora tem um custo de leitura. O coeficiente de `estudante_sim` deixa de ser a diferença entre estudante e não estudante e passa a ser o efeito de um desvio padrão da coluna; para voltar à diferença entre os dois grupos, divide-se o coeficiente pelo desvio padrão da indicadora.

Nas colunas padronizadas, que têm média zero, o intercepto da ridge é a média de `saldo`:

In [ ]:
ridge_padronizada = make_pipeline(StandardScaler(), Ridge(alpha=100)).fit(X, y)
intercepto = ridge_padronizada[-1].intercept_
print("intercepto com λ = 100:", f"{intercepto:.3f}".replace(".", ","))
print("média de saldo:", f"{y.mean():.3f}".replace(".", ","))

iguais = []
for lam in np.logspace(-2, 5, 8):  # 0,01, 0,1, 1, ..., 100.000
    modelo = make_pipeline(StandardScaler(), Ridge(alpha=lam)).fit(X, y)
    iguais.append(np.isclose(modelo[-1].intercept_, y.mean()))
print("intercepto = média nos 8 valores de λ:", bool(np.all(iguais)))

> **🔧 Função**
>
> - **`modelo.intercept_`**: o intercepto $\hat\beta_0$ estimado.
> - **`np.logspace(a, b, n)`**: `n` números igualmente espaçados em escala logarítmica, de $10^a$ a $10^b$; aqui, 8 valores de $\lambda$, de 0,01 a 100.000, cada um dez vezes o anterior.
> - **`np.isclose(a, b)`**: `True` se os dois valores coincidem a menos de uma tolerância pequena, que absorve o arredondamento de ponto flutuante.

Os dois valores são 520,015, e o intercepto repete a média de `saldo` nos oito valores de $\lambda$ que o chunk testa, de 0,01 a 100.000. Com as colunas centradas, a previsão de um cliente com todas as colunas na média é o `saldo` médio, e $\lambda$ não entra nessa conta, porque o intercepto fica fora da penalidade.

### O caminho dos coeficientes em `Credit`

A ridge dá um conjunto de coeficientes para cada $\lambda$. Ajustar uma grade de valores de $\lambda$ mostra o caminho que cada coeficiente percorre, dos mínimos quadrados até perto de zero. Para resumir o quanto o vetor inteiro encolheu, usa-se a **norma $\ell_2$** (lê-se "ele dois"), a distância do vetor de coeficientes até o zero:

$$
\lVert\beta\rVert_2 = \sqrt{\sum_{j=1}^{p}\beta_j^2}.
$$

A razão $\lVert\hat\beta^R_\lambda\rVert_2/\lVert\hat\beta\rVert_2$, com $\hat\beta$ os coeficientes dos mínimos quadrados nas colunas padronizadas, vale 1 quando a ridge devolve os mínimos quadrados e 0 quando todos os coeficientes são zero.

In [ ]:
lams = np.logspace(-2, 5, 100)
caminho = pd.DataFrame(
    [make_pipeline(StandardScaler(), Ridge(alpha=lam)).fit(X, y)[-1].coef_
     for lam in lams],
    index=lams, columns=X.columns,
)
mq_padr = make_pipeline(StandardScaler(), LinearRegression()).fit(X, y)
coef_mq = mq_padr[-1].coef_
razao = np.linalg.norm(caminho, axis=1) / np.linalg.norm(coef_mq)
{
    "razão só cai": bool(np.all(np.diff(razao) < 0)),
    "razão no menor e no maior λ": (round(float(razao[0]), 3),
                                    round(float(razao[-1]), 3)),
    "nenhum coeficiente é zero no maior λ": bool((caminho.iloc[-1] != 0).all()),
}

> **🔧 Função**
>
> **`np.linalg.norm(matriz, axis=1)`**: a norma $\ell_2$ de cada linha da matriz, um número por $\lambda$; sem `axis`, a norma do vetor inteiro.

A razão cai a cada passo da grade, de 0,999 no menor $\lambda$ a 0,004 no maior, e mesmo aí nenhum dos 11 coeficientes é exatamente zero. Nas colunas padronizadas, os coeficientes dos mínimos quadrados são comparáveis entre si:

In [ ]:
mq_padronizado = pd.Series(coef_mq, index=X.columns)
tamanho = mq_padronizado.abs()
print("as quatro maiores em valor absoluto:")
print(", ".join(sorted(tamanho.nlargest(4).index)))
print("a maior:", tamanho.idxmax())
mq_padronizado.round(1).sort_values()

> **🔧 Função**
>
> - **`serie.nlargest(k)`**: os `k` maiores valores da série, com os rótulos.
> - **`serie.sort_values()`**: a série em ordem crescente de valor.

Em valor absoluto, os quatro maiores coeficientes são os de `estudante_sim` (127,7), `pontuacao` (175,6), `renda` (−274,7) e `limite` (440,1), o maior de todos; os das outras sete colunas ficam entre −10,6 e 24,3. Na figura, as quatro ganham cor, e as sete vão em cinza.

Isso fecha a pergunta sobre estudante e renda. Dividido pelo desvio padrão de `estudante_sim`, 0,300, o 127,7 volta à escala da indicadora: 127,7 ÷ 0,300 ≈ 425,7, o 425,747 dos mínimos quadrados da primeira tabela, a diferença de `saldo` entre estudante e não estudante. Medidos na mesma moeda, um desvio padrão de cada coluna, o efeito de `renda` (−274,7) é maior em valor absoluto que o de `estudante_sim` (127,7). É nessa moeda que a comparação se faz; a velocidade com que cada coeficiente cru encolhe não responde à pergunta.

Antes de olhar: se o vetor inteiro encolhe a cada passo, cada coeficiente, sozinho, também só anda na direção de zero?

In [ ]:
# Figura: 'Coeficientes da ridge em `Credit`, com as 11 colunas padronizadas. Em cima, contra $\lambda$ em escala logarítmica; embaixo, contra $\lVert\hat\beta^R_\lambda\rVert_2/\lVert\hat\beta\rVert_2$, o tamanho do vetor de coeficientes como fração do dos mínimos quadrados. Os dois painéis contam o mesmo caminho em sentidos opostos: $\lambda$ pequeno, à esquerda em cima, corresponde à razão perto de 1, à direita embaixo.'
destaque = {"renda": "C0", "limite": "C1", "pontuacao": "C2", "estudante_sim": "C3"}
fig, (ax_lam, ax_razao) = plt.subplots(2, 1, figsize=(6.4, 5.4))
for coluna in X.columns:
    em_destaque = coluna in destaque
    estilo = {
        "color": destaque.get(coluna, "0.45"),
        "linewidth": 2 if em_destaque else 1,
        "zorder": 3 if em_destaque else 1,
    }
    ax_lam.plot(lams, caminho[coluna], **estilo)
    ax_razao.plot(razao, caminho[coluna], **estilo)
ax_lam.set_xscale("log")
ax_lam.set_xlim(lams.min(), lams.max())
ax_lam.set_xlabel("λ")
ax_razao.set_xlim(0, 1)
ax_razao.xaxis.set_major_formatter(lambda valor, _: f"{valor:.1f}".replace(".", ","))
ax_razao.set_xlabel(
    r"$\|\hat{\beta}^{R}_{\lambda}\|_2 \;/\; \|\hat{\beta}\|_2$"
)
for ax in (ax_lam, ax_razao):
    ax.axhline(0, color="0.45", linewidth=0.8)
    ax.set_ylim(-320, 480)
    ax.set_ylabel("coeficiente padronizado")
entradas = [Line2D([], [], color=cor, linewidth=2) for cor in destaque.values()]
entradas.append(Line2D([], [], color="0.45", linewidth=1))
fig.legend(entradas, [*destaque, "outras sete"], loc="lower center", ncols=3,
           frameon=False)
fig.tight_layout(rect=(0, 0.1, 1, 1))
plt.show()

> **🔧 Função**
>
> - **`ax.set_xscale("log")`**: põe o eixo horizontal em escala logarítmica, em que cada marca vale dez vezes a anterior.
> - **`Line2D([], [], color, linewidth)`**: um traço sem dados, que serve só de amostra na legenda.
> - **`fig.legend(amostras, rotulos, loc, ncols, frameon)`**: uma legenda só para a figura inteira, fora dos painéis, com `ncols` colunas e sem moldura.
> - **`fig.tight_layout(rect)`**: arruma os painéis dentro do retângulo `rect`, em frações da figura; a faixa de baixo fica livre para a legenda.

No agregado, tudo encolhe: a razão das normas só cai. Coluna a coluna, não. O chunk abaixo lista quais das quatro curvas coloridas sobem em algum trecho e onde `pontuacao` e `renda` chegam ao máximo:

In [ ]:
pontuacao, renda = caminho["pontuacao"], caminho["renda"]
i_pont = X.columns.get_loc("pontuacao")
lam_positivo = renda[renda > 0].index.min()
depois_de_100 = caminho.loc[caminho.index >= 100]
distancia = (depois_de_100["limite"] - depois_de_100["pontuacao"]).abs()
{
    "sobem em algum trecho": [
        c for c in destaque if bool((np.diff(caminho[c]) > 0).any())
    ],
    "pontuacao, mín. quadrados": round(float(coef_mq[i_pont]), 1),
    "pontuacao, máximo": round(float(pontuacao.max()), 1),
    "pontuacao, λ do máximo": round(float(pontuacao.idxmax()), 1),
    "renda > 0 a partir de λ": round(float(lam_positivo), 1),
    "renda > 0 daí em diante": bool(
        (renda[renda.index >= lam_positivo] > 0).all()
    ),
    "renda, máximo": round(float(renda.max()), 1),
    "|limite − pontuacao| < 2, λ ≥ 100": bool(distancia.max() < 2),
}

Só `renda` e `pontuacao` sobem em algum trecho; `limite` e `estudante_sim` só descem. O coeficiente de `pontuacao` **sobe** de 175,6 nos mínimos quadrados até 281,8, com $\lambda$ perto de 7,9, antes de começar a cair. O de `renda` troca de sinal: começa em −274,7, é positivo em toda a grade a partir de $\lambda = 394{,}4$ e passa por um máximo de 23,4.

In [ ]:
coef_100 = ridge_padronizada[-1].coef_
{
    "corr(limite, pontuacao)": round(float(X["limite"].corr(X["pontuacao"])), 3),
    "corr(renda, limite)": round(float(X["renda"].corr(X["limite"])), 3),
    "corr(renda, saldo)": round(float(X["renda"].corr(y)), 3),
    "λ = 100: limite, pontuacao": (
        round(float(coef_100[X.columns.get_loc("limite")]), 1),
        round(float(coef_100[i_pont]), 1),
    ),
}

As duas curvas dependem de `limite`, o maior coeficiente e o que a penalidade mais cobra. `pontuacao` quase repete `limite`, com correlação de 0,997: quando a penalidade encolhe `limite`, parte do papel dele passa para `pontuacao`. Com $\lambda = 100$, os dois coeficientes já são quase iguais, 211,4 e 209,6, e em todos os $\lambda$ da grade acima de 100 ficam a menos de 2 um do outro: a ridge divide o peso entre as duas colunas, e na figura dos caminhos a curva verde cobre a laranja.

O −274,7 de `renda` compara clientes com os mesmos valores nas outras colunas, `limite` inclusive, e `renda` anda junto com `limite`, com correlação de 0,792. Sozinha, `renda` anda junto com `saldo`, com correlação de 0,464. Com `limite` encolhido, o coeficiente de `renda` passa a carregar parte do que `limite` deixou de explicar, e a troca de sinal é coerente com essa relação positiva.

### Por que encolher ajuda

Se a ridge ajusta o treino pior que os mínimos quadrados, por que ela preveria melhor? O erro de previsão esperado num ponto $x_0$, a média sobre os conjuntos de treino possíveis, se decompõe em três parcelas (seção 7.6):

$$
\begin{aligned}
\mathrm{E}\left[\left(y_0 - \hat f(x_0)\right)^2\right] &= \mathrm{Var}\left(\hat f(x_0)\right) \\
&\quad + \left[\mathrm{Bias}\left(\hat f(x_0)\right)\right]^2 \\
&\quad + \mathrm{Var}(\varepsilon).
\end{aligned}
$$

A variância é o quanto $\hat f(x_0)$ muda de um conjunto de treino para outro; o viés é a distância entre a média dessas previsões e $f(x_0)$; $\mathrm{Var}(\varepsilon)$ é o ruído, que nenhum modelo remove. A média dessa soma sobre os pontos de teste é o **MSE de teste esperado**. A ridge troca uma parcela pela outra. Quanto maior $\lambda$, menos o ajuste se mexe de uma amostra de treino para outra, o que reduz a variância, e mais os coeficientes, em conjunto, ficam perto de zero em vez de perto dos verdadeiros, o que aumenta o viés.

Em `Credit`, onde a $f$ verdadeira é desconhecida, é a validação cruzada, na seção 11.5, que estima se a ridge baixa o erro de teste. Para medir as duas parcelas, é preciso conhecer a $f$ verdadeira, e isso só uma simulação dá. O simulador tem $n = 50$ observações e $p = 45$ preditores, sorteados da normal com média 0 e desvio 1. A resposta é $f(x) = \sum_j \beta_j x_j$ mais ruído, com os 45 coeficientes também sorteados da normal: todos os 45 preditores importam. O ruído tem desvio $\sigma = 2$, e $\mathrm{Var}(\varepsilon) = \sigma^2 = 4$.

A matriz de treino é sorteada uma vez só e fica fixa; o que muda de uma réplica para outra é o ruído somado a $f$. Cada uma das 200 réplicas é um conjunto de treino possível, e o modelo é ajustado em cada uma e prevê 2.000 pontos de teste.

In [ ]:
rng = np.random.default_rng(11)
n_sim, p_sim, sigma = 50, 45, 2.0
X_sim = rng.normal(size=(n_sim, p_sim))            # matriz de treino, fixa
X_teste = rng.normal(size=(2000, p_sim))           # pontos de teste
beta_denso = rng.normal(0, 1, p_sim)               # todos os 45 preditores importam
ruidos = rng.normal(0, sigma, size=(200, n_sim))   # 200 réplicas do ruído

f_treino = X_sim @ beta_denso
f_teste = X_teste @ beta_denso
X_sim.shape, ruidos.shape

> **🔧 Função**
>
> - **`np.random.default_rng(semente)`**: um gerador de números aleatórios do `numpy`, com a semente dada.
> - **`rng.normal(media, desvio, size)`**: sorteios da normal com a média e o desvio dados (0 e 1, se omitidos), no formato `size`.
> - **`A @ b`**: o produto de matriz por vetor; cada linha de `X_sim` vezes `beta_denso`, somado, dá $f$ naquela observação.

Em cada ponto de teste, o **viés²** é o quadrado da distância entre a média das 200 previsões e $f$; a **variância** é o quanto as 200 previsões se espalham em torno da média delas. Tira-se a média de cada uma sobre os 2.000 pontos, e o MSE de teste esperado é a soma das duas mais $\sigma^2$. Ele não é medido em respostas de teste sorteadas: sai da conta, com a $f$ verdadeira.

A função `decomposicao` faz essa conta para qualquer modelo. Ela recebe uma **receita de ajuste**: uma função que, dada uma resposta de treino, ajusta o modelo em `X_sim` e devolve as previsões nos pontos de teste. `ridge_sim(lam)` monta a receita da ridge para um $\lambda$. São 31 valores de $\lambda$ mais os mínimos quadrados, cada um ajustado nas 200 réplicas: $32 \times 200 = 6.400$ ajustes, fora o modelo nulo, que em cada réplica só tira a média da resposta.

In [ ]:
def decomposicao(ajusta_e_preve):
    """Viés², variância e MSE de teste esperado sobre as 200 réplicas do ruído."""
    previsoes = np.array([ajusta_e_preve(f_treino + ruido) for ruido in ruidos])
    vies2 = float(((previsoes.mean(axis=0) - f_teste) ** 2).mean())
    variancia = float(previsoes.var(axis=0).mean())
    return {"viés²": vies2, "variância": variancia,
            "MSE esperado": vies2 + variancia + sigma**2}


def ridge_sim(lam):
    """A receita de ajuste da ridge com esse λ."""
    modelo = make_pipeline(StandardScaler(), Ridge(alpha=lam))
    return lambda y_sim: modelo.fit(X_sim, y_sim).predict(X_teste)


def mq_sim(y_sim):
    return LinearRegression().fit(X_sim, y_sim).predict(X_teste)


def nulo_sim(y_sim):
    return np.full(len(X_teste), y_sim.mean())


lams_sim = np.logspace(-2, 4, 31)
curvas = pd.DataFrame([decomposicao(ridge_sim(lam)) for lam in lams_sim],
                      index=lams_sim)
mq, nulo = decomposicao(mq_sim), decomposicao(nulo_sim)
lam_min = float(curvas["MSE esperado"].idxmin())
resumo = pd.DataFrame(
    {"mínimos quadrados": mq, "ridge no melhor λ": curvas.loc[lam_min],
     "modelo nulo": nulo}
).T
print("melhor λ da grade:", f"{lam_min:.2f}".replace(".", ","))
resumo.style.set_uuid("resumo").format(decimal=",", thousands=".", precision=2)

> **🔧 Função**
>
> - **`previsoes.mean(axis=0)`** e **`previsoes.var(axis=0)`**: a média e a variância de cada coluna do array, isto é, de cada ponto de teste sobre as 200 réplicas (uma réplica por linha).
> - **`np.array(lista)`**: transforma a lista das 200 previsões num array de 200 linhas, uma por réplica.
> - **`lambda y_sim: ...`**: uma função sem nome, de um argumento, escrita numa linha só. A palavra `lambda` do Python não tem relação com o $\lambda$ da ridge.

Os mínimos quadrados não têm viés aqui, porque a $f$ verdadeira é linear; o viés² de 0,12 da tabela é resto de uma simulação com 200 réplicas. A variância, sim, é de 49,38: com 45 coeficientes para 50 observações, o ajuste acompanha o ruído de cada réplica. O MSE de teste esperado deles, 53,50, fica acima até do modelo nulo, 48,67, que ignora os 45 preditores.

A ridge com $\lambda = 3{,}98$, o melhor da grade, aceita um viés² de 5,34 para baixar a variância a 5,18, e o MSE de teste esperado cai a 14,52. A figura mostra a troca ao longo de toda a grade.

In [ ]:
# Figura: 'Viés², variância e MSE de teste esperado das previsões da ridge no simulador ($n = 50$, $p = 45$, todos os preditores importam), contra $\lambda$ em escala logarítmica, com médias sobre 200 réplicas do ruído. A linha tracejada cinza é $\sigma^2 = 4$, o piso irredutível do MSE de teste esperado. O "x" marca o $\lambda$ de menor MSE de teste esperado.'
fig, ax = plt.subplots(figsize=(6.4, 4.0))
ax.plot(lams_sim, curvas["viés²"], color="C0", linewidth=2, label="viés²")
ax.plot(lams_sim, curvas["variância"], color="C2", linewidth=2, label="variância")
ax.plot(lams_sim, curvas["MSE esperado"], color="C3", linewidth=2,
        label="MSE de teste esperado")
ax.axhline(sigma**2, color="0.45", linestyle="--", linewidth=1.5,
           label="piso, σ² = 4")
ax.plot(lam_min, curvas.loc[lam_min, "MSE esperado"], "x", color="C3",
        markersize=11, markeredgewidth=2.5, label="_nolegend_")
ax.set_xscale("log")
ax.set_xlim(lams_sim.min(), lams_sim.max())
ax.set_ylim(0, 60)
ax.set_xlabel("λ")
ax.set_ylabel("erro quadrático médio")
ax.legend(loc="upper center", ncols=2)
fig.tight_layout()
plt.show()

Da esquerda para a direita, a variância cai, e o viés², depois de um trecho perto de zero com o mesmo resto de simulação dos mínimos quadrados, sobe. O chunk abaixo confere a forma das três curvas e mede as duas pontas:

In [ ]:
mse = curvas["MSE esperado"]
vies2 = curvas["viés²"]
{
    "variância só cai": bool(np.all(np.diff(curvas["variância"]) < 0)),
    "maior viés² com λ ≤ 0,1": round(float(vies2[vies2.index < 0.11].max()), 2),
    "viés² só sobe de λ = 0,1 em diante": bool(
        np.all(np.diff(vies2[vies2.index > 0.09]) > 0)
    ),
    "MSE desce até o mínimo e sobe depois": bool(
        np.all(np.diff(mse.loc[:lam_min]) < 0)
        and np.all(np.diff(mse.loc[lam_min:]) > 0)
    ),
    "mínimos quadrados acima do nulo": mq["MSE esperado"] > nulo["MSE esperado"],
    "MSE no menor λ da grade": round(float(mse.iloc[0]), 2),
    "MSE no maior λ da grade": round(float(mse.iloc[-1]), 2),
    "variância no maior λ da grade": round(float(curvas["variância"].iloc[-1]), 2),
}

A variância cai a cada passo da grade. O viés² não passa de 0,11 enquanto $\lambda \le 0{,}1$, oscilação do tamanho do resto de simulação, e a partir de $\lambda = 0{,}1$ só sobe. O MSE de teste esperado desce a cada passo até o mínimo e sobe a cada passo depois dele. Na ponta esquerda, com $\lambda = 0{,}01$, o MSE de teste esperado é de 51,44, ainda longe do mínimo e pouco abaixo dos 53,50 dos mínimos quadrados; na direita, com $\lambda = 10.000$, a variância é de 0,08 e o MSE, 48,15, está perto do modelo nulo. O mínimo de 14,52 fica num $\lambda$ intermediário, longe das duas pontas.

O `True` de "mínimos quadrados acima do nulo" confirma que os mínimos quadrados, nesse cenário, preveem pior que o modelo que ignora os preditores. É um caso em que a ridge ajuda muito: com $p$ perto de $n$, os mínimos quadrados são muito sensíveis à amostra. Com $p > n$, os mínimos quadrados nem têm uma solução única, porque há mais incógnitas, os coeficientes, que equações, as observações; a ridge continua a ter uma para cada $\lambda > 0$.

Puxar os coeficientes para zero troca um pouco de viés por muito menos variância, e, quando os mínimos quadrados variam muito de uma amostra para outra, a troca baixa o erro de teste. A ridge não exclui nenhuma coluna: mesmo no maior $\lambda$ do caminho de `Credit`, nenhum dos 11 coeficientes é zero, e quando o objetivo é escolher quais colunas ficam, ela não faz esse trabalho. Em compensação, sai barata. A busca completa ajusta $2^p$ modelos; a ridge ajusta um modelo por valor de $\lambda$, e o caminho de `Credit`, com 100 valores de $\lambda$, custou 100 ajustes. O que falta é escolher o $\lambda$ sem conhecer a $f$ verdadeira, com validação cruzada, na seção 11.5.

## O Lasso

> **📌 Nota**
>
> Esta seção corresponde à seção 6.2.2 de James et al. (2023).

A ridge aproxima os coeficientes de zero, mas não põe nenhum deles em zero: para qualquer $\lambda$ finito, todas as colunas continuam no modelo, cada uma com o seu peso. Com as 11 colunas de `Credit`, dá para ler a lista inteira. Com centenas de colunas, uma lista de centenas de coeficientes pequenos, nenhum igual a zero, não diz quais colunas importam. A seleção de subconjuntos diz, mas compara $2^p$ modelos, e esse número cresce exponencialmente com $p$. Existe uma penalidade que encolhe os coeficientes, como a ridge, e também escolhe colunas, zerando as outras?

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.lines import Line2D
from sklearn.linear_model import Lasso, LinearRegression, Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

plt.style.use("estilo-figuras.mplstyle")

credit = pd.read_csv("dados/Credit.csv")
X = pd.get_dummies(credit.drop(columns="saldo"), drop_first=True, dtype=float)
y = credit["saldo"]
n = len(X)

### Trocar o quadrado pelo módulo

A penalidade da ridge soma os quadrados dos coeficientes, $\lambda\sum_j\beta_j^2$. O lasso troca o quadrado pelo módulo.

> **🔷 Conceito**
>
> Os coeficientes do **lasso**, $\hat\beta^L_\lambda$, são os valores que minimizam
>
> $$
> \mathrm{RSS} + \lambda\sum_{j=1}^{p}|\beta_j|,
> $$
>
> com $\lambda \ge 0$ o parâmetro de ajuste. A soma $\sum_j|\beta_j|$ é a **norma $\ell_1$** (lê-se "ele um") do vetor de coeficientes, $\lVert\beta\rVert_1$. Como na ridge, o intercepto fica fora da penalidade, e as colunas são padronizadas antes do ajuste.

Com $\lambda = 0$, o lasso devolve os mínimos quadrados, como a ridge. A diferença aparece com $\lambda > 0$: com o módulo, um $\lambda$ grande o bastante põe alguns coeficientes **exatamente** em zero, e a coluna sai do modelo. O lasso faz seleção de variáveis, e os modelos que ele produz são **esparsos**: usam só parte das colunas. Na outra ponta, um $\lambda$ finito já zera todos os coeficientes e dá o modelo nulo, de que a ridge só se aproxima quando $\lambda$ cresce sem limite.

A função `lasso` abaixo monta o lasso, nas colunas padronizadas, a partir do $\lambda$ da fórmula. O chunk ajusta três valores de $\lambda$ em `Credit` e conta, em cada $\lambda$, quantos coeficientes saíram iguais a zero; a coluna $\lambda = 0$ traz os mínimos quadrados, também nas colunas padronizadas.

In [ ]:
def lasso(lam, n_linhas):
    """O lasso com o λ da fórmula, nas colunas padronizadas."""
    return make_pipeline(
        StandardScaler(),
        Lasso(alpha=lam / (2 * n_linhas), max_iter=100_000, tol=1e-8),
    )


mq_padr = make_pipeline(StandardScaler(), LinearRegression()).fit(X, y)
coeficientes = pd.DataFrame({"0": mq_padr[-1].coef_}, index=X.columns)
for lam in [1_000, 10_000, 100_000]:
    nome = f"{lam:,}".replace(",", ".")
    coeficientes[nome] = lasso(lam, n).fit(X, y)[-1].coef_
coeficientes = coeficientes + 0.0  # troca −0,0 por 0,0
coeficientes.columns.name = "λ"
print("zeros por λ:", (coeficientes == 0).sum().tolist())
maior = coeficientes["0"].abs().idxmax()
print("maior |coeficiente| com λ = 0:")
print(" ", maior, f"{coeficientes.loc[maior, '0']:.1f}".replace(".", ","))
numero_br = lambda valor: f"{valor:.1f}".replace(".", ",").replace("-", "−")
coeficientes.style.set_uuid("coef").set_table_styles(
    [{"selector": "th, td", "props": "padding: 0.2em 0.15em; font-size: 0.85em"}]
).format(numero_br)

> **🔧 Função**
>
> - **`Lasso(alpha, max_iter, tol)`**: o lasso. O `alpha` do `scikit-learn` **não** é o $\lambda$ da fórmula: o `Lasso` minimiza $\frac{1}{2n}\mathrm{RSS} + \alpha\lVert\beta\rVert_1$, que é a fórmula acima dividida por $2n$, e por isso `alpha=lam / (2 * n)`. `max_iter` e `tol` dizem quando o algoritmo para de refinar os coeficientes; com os padrões (1.000 e $10^{-4}$), ele pode parar antes de chegar à solução quando duas colunas quase se repetem, como `limite` e `pontuacao` (seção 8.6).
> - **`(df == 0).sum()`**: quantos valores de cada coluna são iguais a zero; cada `True` conta 1.
> - **`df + 0.0`**: o lasso às vezes devolve `-0.0`, que é igual a zero mas apareceria na tabela com sinal; somar `0.0` o troca por `0.0`.
> - **`.set_table_styles(regras)`**: acrescenta regras de CSS à tabela; aqui, um espaçamento e uma letra menores, para a tabela caber numa tela estreita. **`.format(funcao)`** aplica `funcao` a cada célula; `numero_br` escreve o número com vírgula e com o sinal de menos tipográfico.

Com $\lambda = 1.000$, nenhum coeficiente é zero. Com $\lambda = 10.000$, cinco são exatamente zero: `escolaridade`, `imovel_proprio_sim`, `casado_sim` e as duas indicadoras de região; o modelo fica com as outras seis colunas. Com $\lambda = 100.000$, dez são zero, e sobra só `pontuacao`, com 271,6.

`limite` tinha o maior coeficiente dos mínimos quadrados em módulo, 440,1, e com $\lambda = 100.000$ já é zero, enquanto `pontuacao`, que tinha 175,6, continua no modelo. Por que é `pontuacao` que sobra?

### O caminho do lasso em `Credit`

Como na ridge, ajustar uma grade de valores de $\lambda$ mostra o caminho de cada coeficiente. A grade tem 100 valores, de 10 a $10^{5{,}6}$. O tamanho do vetor de coeficientes agora se resume pela norma $\ell_1$: a razão $\lVert\hat\beta^L_\lambda\rVert_1/\lVert\hat\beta\rVert_1$, com $\hat\beta$ os mínimos quadrados nas colunas padronizadas, vale 1 nos mínimos quadrados e 0 no modelo nulo. O chunk conta também, em cada $\lambda$, quantos coeficientes não são zero.

In [ ]:
lams = np.logspace(1, 5.6, 100)
caminho = pd.DataFrame(
    [lasso(lam, n).fit(X, y)[-1].coef_ for lam in lams],
    index=lams, columns=X.columns,
)
coef_mq = mq_padr[-1].coef_
razao = caminho.abs().sum(axis=1) / np.abs(coef_mq).sum()
nao_nulos = (caminho != 0).sum(axis=1)
print("não nulos não caem com λ caindo:", bool(np.all(np.diff(nao_nulos) <= 0)))
print("razão no menor λ:", f"{razao.iloc[0]:.3f}".replace(".", ","))
print("razão no maior λ:", f"{razao.iloc[-1]:.3f}".replace(".", ","))
print("tamanhos na grade:")
print(" ", sorted(set(nao_nulos.tolist())))

> **🔧 Função**
>
> - **`df.sum(axis=1)`**: soma cada linha, em vez de cada coluna; aqui, uma soma por $\lambda$. Com `caminho.abs()`, é a norma $\ell_1$ dos coeficientes em cada $\lambda$; com `caminho != 0`, o número de coeficientes não nulos.

Da direita para a esquerda da grade, isto é, com $\lambda$ caindo, o número de coeficientes não nulos nunca diminui, e a razão vai de 0,000 a 1,000. Os tamanhos de modelo que aparecem na grade vão de 0 a 11 colunas, com uma exceção: não há modelo de duas colunas. O ponto da grade em que cada coluna entra, o maior $\lambda$ em que o coeficiente dela não é zero, explica a falta:

In [ ]:
entrada = pd.Series(
    {coluna: caminho.index[caminho[coluna] != 0].max() for coluna in X.columns}
).sort_values(ascending=False)
fica = [bool((caminho.loc[:entrada[c], c] != 0).all()) for c in X.columns]
print("nenhuma volta a zero:", all(fica))
entrada.index.name = "coluna"
anterior = [lams[lams > lam].min() for lam in entrada.values]  # ponto anterior
tabela_entrada = pd.DataFrame({
    "λ de entrada": entrada,
    "razão nesse λ": razao.loc[entrada.values].values,
    "razão no ponto anterior": razao.loc[anterior].values,
})
tabela_entrada.style.set_uuid("entrada").set_table_attributes(
    'style="width: auto"'
).set_table_styles(
    [{"selector": "th, td", "props": "padding: 0.2em 0.3em; font-size: 0.85em"}]
).format({"λ de entrada": "{:,.0f}", "razão nesse λ": "{:.2f}",
          "razão no ponto anterior": "{:.2f}"}, decimal=",", thousands=".")

`pontuacao` entra sozinha, em $\lambda = 288.806$. `limite` e `estudante_sim` entram no mesmo ponto da grade, $\lambda = 89.022$, com a razão das normas em 0,27, e por isso o modelo passa de uma coluna para três; uma grade mais fina poderia separar as duas. Depois vêm `renda`, em 42.096, `cartoes` e `idade`, e por último as cinco que já eram zero com $\lambda = 10.000$. Nesta grade, nenhuma coluna volta a zero depois de entrar.

Cada coluna entra entre o ponto anterior da grade, em que o coeficiente dela ainda é zero, e o $\lambda$ de entrada. Por isso, no gráfico contra a razão das normas, a curva de uma coluna começa a sair do zero na razão da última coluna da tabela: a de `renda`, por exemplo, sai do zero em 0,37 e já está fora dele em 0,42.

Uma pergunta para a figura abaixo: a curva de `limite`, que entra depois de `pontuacao`, chega a passar por cima dela?

In [ ]:
# Figura: 'Coeficientes do lasso em `Credit`, com as 11 colunas padronizadas. Em cima, contra $\lambda$ em escala logarítmica; embaixo, contra $\lVert\hat\beta^L_\lambda\rVert_1/\lVert\hat\beta\rVert_1$, o tamanho do vetor de coeficientes, medido pela norma $\ell_1$, como fração do dos mínimos quadrados. As cores e a escala vertical são as da figura dos caminhos da ridge, na seção 11.3.'
destaque = {"renda": "C0", "limite": "C1", "pontuacao": "C2", "estudante_sim": "C3"}
fig, (ax_lam, ax_razao) = plt.subplots(2, 1, figsize=(4.8, 5.8))
for coluna in X.columns:
    em_destaque = coluna in destaque
    estilo = {
        "color": destaque.get(coluna, "0.45"),
        "linewidth": 2 if em_destaque else 1,
        "zorder": 3 if em_destaque else 1,
    }
    ax_lam.plot(lams, caminho[coluna], **estilo)
    ax_razao.plot(razao, caminho[coluna], **estilo)
ax_lam.set_xscale("log")
ax_lam.set_xlim(lams.min(), lams.max())
ax_lam.set_xlabel("λ")
ax_razao.set_xlim(0, 1)
ax_razao.xaxis.set_major_formatter(lambda valor, _: f"{valor:.1f}".replace(".", ","))
ax_razao.set_xlabel(
    r"$\|\hat{\beta}^{L}_{\lambda}\|_1 \;/\; \|\hat{\beta}\|_1$"
)
for ax in (ax_lam, ax_razao):
    ax.axhline(0, color="0.45", linewidth=0.8)
    ax.set_ylim(-320, 480)
    ax.set_ylabel("coeficiente padronizado")
entradas = [Line2D([], [], color=cor, linewidth=2) for cor in destaque.values()]
entradas.append(Line2D([], [], color="0.45", linewidth=1))
fig.legend(entradas, [*destaque, "outras sete"], loc="lower center", ncols=3,
           frameon=False, fontsize="small")
fig.tight_layout(rect=(0, 0.1, 1, 1))
plt.show()

No painel de baixo, os caminhos são feitos de trechos retos, e, nesta grade, cada dobra é um ponto em que alguma coluna entra. Com a razão perto de 1, `limite` está acima de `pontuacao`, como nos mínimos quadrados, e mais à esquerda as duas curvas se cruzam. O chunk abaixo mede até que $\lambda$ `limite` fica acima e qual das duas fica acima no resto da grade:

In [ ]:
acima = caminho["limite"] > caminho["pontuacao"]
lam_cruza = float(caminho.index[acima].max())
print("limite > pontuacao até λ =", f"{lam_cruza:,.0f}".replace(",", "."))
print("em todo λ da grade até esse:", bool(acima.loc[:lam_cruza].all()))
print("pontuacao ≥ limite acima dele:",
      bool((~acima.loc[lam_cruza:].iloc[1:]).all()))
print("razão nesse λ:", f"{razao.loc[lam_cruza]:.2f}".replace(".", ","))

`limite` fica acima de `pontuacao` em todos os $\lambda$ da grade até 10.476; para $\lambda$ maiores que esse, `pontuacao` fica acima ou igual.

#### Por que `pontuacao` entra primeiro

A ordem de entrada tem uma regra, e ela sai de olhar o ponto em que todos os coeficientes são zero. Ali, com as colunas padronizadas $\tilde x_{ij}$ da seção 11.3, o intercepto é $\bar y$ e o RSS vale $\sum_i (y_i - \bar y)^2$. Dar à coluna $j$ um coeficiente pequeno, $\delta$, troca cada resíduo $y_i - \bar y$ por $y_i - \bar y - \delta\tilde x_{ij}$. Abrindo o quadrado, o RSS muda em $-2\delta\sum_i\tilde x_{ij}(y_i - \bar y) + \delta^2\sum_i\tilde x_{ij}^2$, e com $\delta$ pequeno o termo em $\delta^2$ é desprezível. O termo que sobra faz o RSS cair quando $\delta$ tem o mesmo sinal da soma $\sum_i\tilde x_{ij}(y_i - \bar y)$, e subir quando tem o sinal oposto. Com o sinal que ajuda, o RSS cai a uma taxa de **duas vezes** o módulo da soma por unidade de $|\delta|$.

A penalidade sobe $\lambda|\delta|$, a uma taxa $\lambda$, qualquer que seja o sinal de $\delta$. Enquanto a penalidade subir mais depressa do que o RSS cai, para toda coluna, o lasso fica no zero. Todos os coeficientes do lasso são zero, portanto, enquanto

$$
\lambda \ge \lambda_{\max} = 2\max_j\Big|\sum_{i=1}^{n}\tilde x_{ij}\,(y_i - \bar y)\Big|,
$$

isto é, enquanto $\lambda$ for pelo menos o dobro da maior dessas somas em módulo. A coluna que atinge esse máximo, a que baixa o RSS mais depressa, é a primeira a entrar. O chunk abaixo faz a conta em `Credit`:

In [ ]:
X_padr = mq_padr[0].transform(X)  # as 11 colunas padronizadas
somas = pd.Series(  # uma soma Σ x̃ᵢⱼ(yᵢ − ȳ) por coluna j, em módulo
    np.abs(X_padr.T @ (y - y.mean()).to_numpy()), index=X.columns
)
lam_max = 2 * somas.max()
proximo = lams[lams > entrada.iloc[0]].min()
print("coluna do máximo:", somas.idxmax())
print("λ_max:", f"{lam_max:,.0f}".replace(",", "."))
print("λ_max no intervalo da entrada:",
      bool(entrada.iloc[0] < lam_max <= proximo))

> **🔧 Função**
>
> **`modelo[0].transform(X)`**: o primeiro passo do `Pipeline`, o `StandardScaler` já ajustado, aplicado a `X`: as colunas padronizadas com as médias e os desvios do ajuste. Em `X_padr.T @ …`, a transposta e o produto de matriz por vetor dão uma soma por coluna.

A coluna do máximo é `pontuacao`, e o $\lambda_{\max}$ cai entre o $\lambda$ em que `pontuacao` entra na grade e o ponto seguinte, onde todos os coeficientes ainda são zero. A soma tem uma leitura mais familiar: numa coluna padronizada, ela é a correlação da coluna com a resposta multiplicada por $n$ vezes o desvio padrão de $y$, que se escreve $\mathrm{dp}(y)$. A correlação da coluna $j$ com $y$ é

$$
r_j = \frac{\sum_{i}\tilde x_{ij}\,(y_i - \bar y)}{n \cdot 1 \cdot \mathrm{dp}(y)},
$$

em que o 1 é o desvio padrão de $\tilde x_j$; os dois desvios padrão usam divisor $n$, como na padronização. Então $\lambda_{\max} = 2n\,\mathrm{dp}(y)\max_j|r_j|$: a primeira coluna a entrar é a de maior correlação com a resposta em valor absoluto.

In [ ]:
correlacao = X.corrwith(y).abs().sort_values(ascending=False)
print("maiores |correlações| com saldo:")
for coluna, r in correlacao.head(2).items():
    print(f"  {coluna}: {r:.4f}".replace(".", ","))
print("λ_max pela correlação igual:",
      bool(np.isclose(2 * n * y.std(ddof=0) * correlacao.iloc[0], lam_max)))
entre_elas = X["limite"].corr(X["pontuacao"])
print("corr(limite, pontuacao):", f"{entre_elas:.4f}".replace(".", ","))

> **🔧 Função**
>
> - **`X.corrwith(y)`**: a correlação de cada coluna de `X` com a série `y`, uma por coluna.
> - **`serie.items()`**: os pares (rótulo, valor) da série, um por vez, para percorrer com `for`.
> - **`np.isclose(a, b)`**: `True` se `a` e `b` são iguais a menos do arredondamento da conta em ponto flutuante.

`pontuacao` tem a maior correlação com `saldo`, por pouco: 0,8636 contra 0,8617 de `limite`. E as duas colunas quase se repetem, com correlação entre elas de 0,9969. Uma diferença pequena na correlação com `saldo` decide qual das duas entra primeiro, e é essa que fica sozinha no modelo com a penalidade forte: nesta grade, nos pontos acima daquele em que `limite` entra, 89.022, até $\lambda_{\max}$. Com diferença tão pequena, outra amostra de clientes pode inverter a ordem e deixar `limite` no lugar de `pontuacao`.

> **⚠️ Atenção — Coeficiente zero não quer dizer coluna sem relação com a resposta**
>
> `limite` tem correlação 0,8617 com `saldo`, e o lasso com $\lambda = 100.000$ põe o coeficiente de `limite` em zero. O zero diz que, com `pontuacao` no modelo e com esse $\lambda$, `limite` não acrescenta o bastante para pagar a penalidade; não diz que o saldo não depende do limite.

### Orçamento: outra forma de escrever as duas penalidades

A ridge e o lasso diferem só na troca do quadrado pelo módulo. Por que o módulo põe coeficientes em zero e o quadrado não? Para ver, ajuda escrever as duas penalidades de outro jeito: em vez de somar uma penalidade ao RSS, limita-se o tamanho dos coeficientes. Os coeficientes do lasso resolvem

$$
\begin{aligned}
&\text{minimizar } \mathrm{RSS} \text{ em } \beta\\
&\text{sujeito a } \textstyle\sum_{j=1}^{p}|\beta_j| \le s,
\end{aligned}
$$

e os da ridge resolvem

$$
\begin{aligned}
&\text{minimizar } \mathrm{RSS} \text{ em } \beta\\
&\text{sujeito a } \textstyle\sum_{j=1}^{p}\beta_j^2 \le s.
\end{aligned}
$$

Para cada $\lambda$ existe um $s$ com os mesmos coeficientes, e vice-versa. O número $s$ é um **orçamento**: os coeficientes podem ter o tamanho que quiserem, desde que a soma dos módulos, no lasso, ou dos quadrados, na ridge, não passe de $s$. Com $s$ grande o bastante para caberem os mínimos quadrados, a restrição não pesa, e a resposta é a dos mínimos quadrados, o que corresponde a $\lambda = 0$. Com $s$ pequeno, os coeficientes têm de ser pequenos.

### Por que o lasso zera coeficientes

Com duas colunas, o orçamento vira uma região do plano $(\beta_1, \beta_2)$. No lasso, com orçamento $s_L$, é o losango $|\beta_1| + |\beta_2| \le s_L$, com os bicos à distância $s_L$ da origem; na ridge, com orçamento $s_R$, é o disco $\beta_1^2 + \beta_2^2 \le s_R$, de raio $\sqrt{s_R}$. A solução é o ponto da região com o menor RSS.

Para ver onde esse ponto cai, o chunk abaixo sorteia 100 observações de duas colunas de uma normal bivariada, com variâncias 1 e correlação 0,5, e uma resposta $y = 1{,}0\,x_1 + 2{,}5\,x_2 + \varepsilon$, com $\varepsilon$ normal de desvio 1. Nas colunas padronizadas, ele ajusta os mínimos quadrados e o lasso com $\lambda = 300$, valor escolhido para este exemplo porque zera um dos dois coeficientes. O orçamento do lasso, $s_L$, é a norma $\ell_1$ da solução dele.

Para comparar as duas formas de região, a ridge precisa ter o mesmo tamanho. O chunk procura, numa grade fina de $\lambda$, a ridge cuja solução tem norma $\ell_2$ igual a $s_L$. O disco dela tem então raio $s_L$ e toca os eixos nos mesmos pontos que os bicos do losango, de modo que as duas regiões diferem só na forma.

In [ ]:
rng = np.random.default_rng(11)
Z = rng.multivariate_normal([0, 0], [[1, 0.5], [0.5, 1]], size=100)
y_z = 1.0 * Z[:, 0] + 2.5 * Z[:, 1] + rng.normal(0, 1, size=100)
n_z = len(Z)

mq_z = make_pipeline(StandardScaler(), LinearRegression()).fit(Z, y_z)
lasso_z = lasso(300, n_z).fit(Z, y_z)
s_L = float(np.abs(lasso_z[-1].coef_).sum())  # o orçamento do lasso


def ridge(lam):
    """A ridge nas colunas padronizadas."""
    return make_pipeline(StandardScaler(), Ridge(alpha=lam))


lams_z = np.logspace(1, 3, 2001)
normas = np.array(
    [np.linalg.norm(ridge(lam).fit(Z, y_z)[-1].coef_) for lam in lams_z]
)
lam_ridge = float(lams_z[np.argmin(np.abs(normas - s_L))])
ridge_z = ridge(lam_ridge).fit(Z, y_z)
raio = float(np.linalg.norm(ridge_z[-1].coef_))

solucoes = pd.DataFrame(
    [mq_z[-1].coef_, lasso_z[-1].coef_ + 0.0, ridge_z[-1].coef_],
    index=["mínimos quadrados", "lasso, λ = 300", "ridge"],
    columns=["β₁", "β₂"],
)
beta_mq = mq_z[-1].coef_
norma_1, norma_2 = np.abs(beta_mq).sum(), np.linalg.norm(beta_mq)
print("orçamento s_L do lasso:", f"{s_L:.3f}".replace(".", ","))
print("λ da ridge:", f"{lam_ridge:.1f}".replace(".", ","))
print("raio do disco da ridge:", f"{raio:.3f}".replace(".", ","))
print("s_R da ridge (raio²):", f"{raio**2:.3f}".replace(".", ","))
print("normas ℓ1 e ℓ2 de β̂:",
      f"{norma_1:.3f}; {norma_2:.3f}".replace(".", ","))
print("β̂ fora das duas regiões:", bool(norma_1 > s_L and norma_2 > raio))
solucoes.style.set_uuid("solucoes").format(decimal=",", precision=3)

> **🔧 Função**
>
> - **`rng.multivariate_normal(medias, covariancias, size)`**: `size` sorteios de uma normal em várias dimensões, aqui duas, com o vetor de médias e a matriz de covariâncias dados; com variâncias 1, a covariância 0,5 é a correlação entre as duas colunas.
> - **`np.argmin(arr)`**: a posição do menor elemento; aqui, a do $\lambda$ da grade cuja norma fica mais perto de `s_L`.

O lasso com $\lambda = 300$ põe $\beta_1$ em zero e fica com $\beta_2 = 1{,}374$: o orçamento $s_L$ é 1,374, todo gasto em $\beta_2$. A ridge com $\lambda = 110{,}9$ tem norma $\ell_2$ igual a 1,374, o valor de $s_L$, e o disco dela tem raio 1,374. O orçamento da ridge, na forma $\beta_1^2 + \beta_2^2 \le s_R$, é o quadrado do raio, $s_R = 1{,}889$; o chunk o calcula antes de arredondar o raio. A ridge reparte o peso: 0,712 e 1,176, os dois diferentes de zero. E $\hat\beta$, os mínimos quadrados, tem norma $\ell_1$ 3,235 e norma $\ell_2$ 2,557, e fica fora das duas regiões.

Para desenhar a região e o RSS juntos, calcula-se o RSS em cada ponto de uma malha de 200 × 200 pares $(\beta_1, \beta_2)$. Com as colunas padronizadas, o intercepto é a média da resposta, como a seção 11.3 mostrou para a ridge, e o resíduo de cada observação é $y_i - \bar y - \beta_1\tilde x_{i1} - \beta_2\tilde x_{i2}$. A função `rss_em` faz essa conta para um par $(\beta_1, \beta_2)$:

In [ ]:
Z_padr = mq_z[0].transform(Z)  # as duas colunas padronizadas
r = y_z - y_z.mean()             # a resposta centrada


def rss_em(beta):
    """O RSS de um par (β₁, β₂)."""
    return float(((r - Z_padr @ beta) ** 2).sum())

In [ ]:
b1s = np.linspace(-1.6, 1.6, 200)
b2s = np.linspace(-1.6, 2.8, 200)
rss = np.array([[rss_em((b1, b2)) for b1 in b1s] for b2 in b2s])

Cada curva de nível do RSS é uma elipse centrada em $\hat\beta$, e todos os pontos dela têm o mesmo RSS. Quanto mais longe de $\hat\beta$, maior o RSS. A solução com orçamento é o ponto da região com o menor RSS: a primeira elipse que, crescendo a partir de $\hat\beta$, encosta na região.

Antes da figura, um palpite: a elipse que cresce a partir de $\hat\beta$ encosta no losango num bico (sobre um eixo) ou num lado?

In [ ]:
# Figura: 'Curvas de nível do RSS (azul) e regiões de orçamento (cinza) do lasso, à esquerda, e da ridge, à direita, com duas colunas padronizadas. O "x" cinza é $\hat\beta$, a solução de mínimos quadrados. A região do lasso é o losango $|\beta_1| + |\beta_2| \le 1{,}374$; a da ridge é o disco $\beta_1^2 + \beta_2^2 \le 1{,}374^2$, de raio 1,374. O ponto laranja é a solução de cada método, onde a curva de nível mais externa desenhada encosta na região.'
fig, (ax_l, ax_r) = plt.subplots(1, 2, figsize=(5.0, 4.4), sharey=True)
angulos = np.linspace(0, 2 * np.pi, 400)
regiao = {"facecolor": (0.45, 0.45, 0.45, 0.3), "edgecolor": "0.45", "linewidth": 1.2}
ax_l.fill([s_L, 0, -s_L, 0], [0, s_L, 0, -s_L], **regiao)
ax_r.fill(raio * np.cos(angulos), raio * np.sin(angulos), **regiao)
com_menos = lambda valor, _: f"{valor:.1f}".replace(".", ",").replace("-", "−")
for ax, modelo, nome in [(ax_l, lasso_z, "lasso"), (ax_r, ridge_z, "ridge")]:
    beta = modelo[-1].coef_
    minimo, no_ponto = rss_em(beta_mq), rss_em(beta)
    niveis = minimo + (no_ponto - minimo) * np.array([0.1, 0.35, 0.65, 1.0])
    ax.contour(b1s, b2s, rss, levels=niveis, colors="C0", linewidths=1.2)
    ax.plot(*beta_mq, "x", color="0.45", markersize=9, markeredgewidth=2)
    ax.annotate("β̂", beta_mq, xytext=(0, -8), textcoords="offset points",
                ha="center", va="top")
    ax.plot(*beta, "o", color="C1", markersize=8, zorder=3)
    ax.set_title(nome, loc="left")
    ax.axhline(0, color="0.45", linewidth=0.8)
    ax.axvline(0, color="0.45", linewidth=0.8)
    ax.set_aspect("equal")
    ax.set_xlim(-1.6, 1.6)
    ax.set_ylim(-1.6, 3.2)
    ax.set_xlabel("β₁")
    ax.xaxis.set_major_formatter(com_menos)
    ax.yaxis.set_major_formatter(com_menos)
ax_l.set_ylabel("β₂")
fig.tight_layout()
plt.show()

O losango tem bicos, e os bicos estão sobre os eixos. A elipse que cresce a partir de $\hat\beta$ encosta no losango justamente no bico de cima, onde $\beta_1 = 0$: é o zero de $\beta_1$ na tabela das três soluções. O disco não tem bicos, e a elipse encosta nele num ponto fora dos eixos, com os dois coeficientes diferentes de zero.

> **🔧 Função**
>
> - **`np.linspace(a, b, k)`**: `k` números igualmente espaçados de `a` a `b`; aqui, os 400 ângulos que desenham o disco.
> - **`ax.fill(xs, ys, color, alpha)`**: pinta o polígono de vértices `(xs, ys)`; o losango tem quatro vértices, e o disco é um polígono de 400 lados.
> - **`np.cos(arr)`** e **`np.sin(arr)`**: o cosseno e o seno de cada ângulo, em radianos; $(r\cos\theta, r\sin\theta)$ percorre o círculo de raio $r$.
> - **`ax.contour(b1s, b2s, rss, levels, colors)`**: as curvas de nível de `rss` sobre a malha, uma para cada valor de `levels`; `colors` fixa a cor de todas.
> - **`ax.axvline(x)`**: uma linha vertical na posição `x`, de ponta a ponta do painel.
> - **`ax.set_aspect("equal")`**: a mesma escala nos dois eixos, para o disco sair redondo e o losango, com ângulos retos.

A elipse não precisa encostar num bico: com outro $\hat\beta$, ela pode tocar um dos lados do losango, e o lasso fica com as duas colunas. A intuição é que o bico é saliente: elipses de muitas posições e inclinações encostam nele antes de tocar qualquer lado. Com três colunas, a região do lasso é um octaedro, e a da ridge, uma esfera; com mais colunas, os bicos e as arestas da região do lasso podem pôr vários coeficientes em zero ao mesmo tempo.

A forma com orçamento descreve também a seleção de subconjuntos. Com $I(\beta_j \ne 0)$ valendo 1 quando o coeficiente não é zero e 0 quando é, o problema

$$
\begin{aligned}
&\text{minimizar } \mathrm{RSS} \text{ em } \beta\\
&\text{sujeito a } \textstyle\sum_{j=1}^{p}I(\beta_j \ne 0) \le s
\end{aligned}
$$

pede o menor RSS entre os modelos com no máximo $s$ coeficientes diferentes de zero. É a melhor seleção de subconjuntos da seção 11.1, com o orçamento $s$ no papel do tamanho $k$ de $\mathcal{M}_k$. Na forma direta, ela se resolve comparando os $\binom{p}{s}$ modelos de tamanho $s$, e isso fica inviável quando $p$ cresce. A ridge e o lasso trocam esse orçamento, que conta colunas, por outros que se resolvem com um ajuste por valor de $\lambda$. Dos dois, só o lasso tem bicos sobre os eixos, e por isso fica mais perto da seleção de subconjuntos: também zera coeficientes.

A tabela das três soluções mostra ainda o risco do zero. O $y$ deste exemplo depende das duas colunas: $x_1$ entra com coeficiente 1,0 na escala em que foi sorteada, e os mínimos quadrados dão a ela 0,810 nas colunas padronizadas, a escala em que estão os coeficientes da tabela. O lasso com $\lambda = 300$ descartou $x_1$. Zerar um coeficiente é uma aposta de que a coluna é dispensável, dadas as outras; quando a aposta erra, o lasso ganha viés em relação aos mínimos quadrados, e a decomposição a seguir separa o viés² da variância.

### Lasso ou ridge?

Qual dos dois prevê melhor? O simulador da seção 11.3, com $n = 50$ observações, $p = 45$ preditores, ruído de desvio $\sigma = 2$ e 200 réplicas do ruído sobre uma matriz de treino fixa, serve para comparar os dois em dois casos extremos. No cenário **denso**, os 45 coeficientes verdadeiros são sorteados da normal padrão, e todos os preditores importam. No **esparso**, só dois importam, com coeficientes 3 e −3, e os outros 43 valem zero. A função `decompoe` faz a conta da seção 11.3, com médias sobre os 2.000 pontos de teste, e devolve os dois termos separados, o viés² e a variância, junto com o número médio de coeficientes diferentes de zero e o número de colunas que ficam zeradas em algumas réplicas e não em outras; `mse_esperado` soma os dois termos e $\sigma^2$.

In [ ]:
rng = np.random.default_rng(11)
n_sim, p_sim, sigma = 50, 45, 2.0
X_sim = rng.normal(size=(n_sim, p_sim))            # matriz de treino, fixa
X_teste = rng.normal(size=(2000, p_sim))           # pontos de teste
beta_denso = rng.normal(0, 1, p_sim)               # todos os 45 preditores importam
ruidos = rng.normal(0, sigma, size=(200, n_sim))   # 200 réplicas do ruído
beta_esparso = np.zeros(p_sim)
beta_esparso[[0, 1]] = [3, -3]                     # só dois importam


def decompoe(beta, modelo):
    """Viés², variância, não nulos em média e colunas que oscilam."""
    f_treino, f_teste = X_sim @ beta, X_teste @ beta
    previsoes, nao_zero = [], []
    for ruido in ruidos:
        modelo.fit(X_sim, f_treino + ruido)
        previsoes.append(modelo.predict(X_teste))
        nao_zero.append(modelo[-1].coef_ != 0)
    previsoes, nao_zero = np.array(previsoes), np.array(nao_zero)
    vies2 = ((previsoes.mean(axis=0) - f_teste) ** 2).mean()
    variancia = previsoes.var(axis=0).mean()
    # colunas zeradas em algumas réplicas e não em outras
    oscilam = int((nao_zero.any(axis=0) & ~nao_zero.all(axis=0)).sum())
    return float(vies2), float(variancia), float(nao_zero.sum(axis=1).mean()), oscilam


def mse_esperado(beta, modelo):
    """MSE de teste esperado: viés² + variância + σ²."""
    vies2, variancia = decompoe(beta, modelo)[:2]
    return vies2 + variancia + sigma**2

> **🔧 Função**
>
> - **`np.zeros(k)`**: um array de `k` zeros; aqui, os 45 coeficientes do cenário esparso antes de os dois primeiros receberem 3 e −3.
> - **`arr.any(axis=0)`** e **`arr.all(axis=0)`**: para cada coluna de uma matriz de `True`/`False`, se algum ou se todos os valores são `True`; `&` é o "e" elemento a elemento, e `~` troca `True` por `False`.

Cada método percorre a sua grade de 31 valores de $\lambda$ em cada cenário: a da ridge vai de $10^{-2}$ a $10^{4}$, e a do lasso, de $10^{-1}$ a $10^{3}$. A tabela guarda o menor MSE de teste esperado de cada método em cada cenário. São $2 \times 2 \times 31 \times 200 = 24.800$ ajustes.

Vale arriscar uma resposta antes da tabela: algum dos dois métodos ganha nos dois cenários?

In [ ]:
lams_ridge = np.logspace(-2, 4, 31)
lams_lasso = np.logspace(-1, 3, 31)
linhas, melhor_lam, curvas = {}, {}, {}
for cenario, beta in [("denso", beta_denso), ("esparso", beta_esparso)]:
    curva_ridge = pd.Series(
        [mse_esperado(beta, ridge(lam)) for lam in lams_ridge], index=lams_ridge
    )
    curva_lasso = pd.Series(
        [mse_esperado(beta, lasso(lam, n_sim)) for lam in lams_lasso],
        index=lams_lasso,
    )
    linhas[cenario] = {"ridge": curva_ridge.min(), "lasso": curva_lasso.min()}
    melhor_lam[cenario] = {"ridge": curva_ridge.idxmin(),
                           "lasso": curva_lasso.idxmin()}
    curvas[cenario] = {"ridge": curva_ridge, "lasso": curva_lasso}
tabela = pd.DataFrame(linhas).T
tabela.index.name = "cenário"
print("ridge ganha no denso:",
      bool(tabela.loc["denso", "ridge"] < tabela.loc["denso", "lasso"]))
print("lasso ganha no esparso:",
      bool(tabela.loc["esparso", "lasso"] < tabela.loc["esparso", "ridge"]))
tabela.style.set_uuid("contra").set_table_attributes('style="width: auto"').format(
    decimal=",", precision=2
)

No cenário denso, a ridge chega a 14,52 e o lasso, a 19,29. No esparso, a ordem se inverte: o lasso chega a 4,96, 0,96 acima do piso $\sigma^2 = 4$ (4,96 − 4), e a ridge fica em 10,67. Neste simulador, nenhum dos dois ganha nos dois cenários: qual prevê melhor depende de quantas colunas importam de fato. A figura mostra as curvas inteiras, de onde saem esses mínimos.

In [ ]:
# Figura: 'MSE de teste esperado no simulador, contra $\lambda$ em escala logarítmica, para a ridge (azul) e o lasso (laranja), com o mínimo de cada curva marcado. Em cima, o cenário denso, com os 45 coeficientes verdadeiros diferentes de zero; embaixo, o esparso, com dois. A linha tracejada cinza é o piso $\sigma^2 = 4$.'
fig, eixos = plt.subplots(2, 1, figsize=(4.8, 5.8), sharex=True)
cores = {"ridge": "C0", "lasso": "C1"}
for ax, cenario in zip(eixos, ["denso", "esparso"]):
    for metodo, curva in curvas[cenario].items():
        ax.plot(curva.index, curva, color=cores[metodo], linewidth=2, label=metodo)
        ax.plot(curva.idxmin(), curva.min(), "o", color=cores[metodo], markersize=8)
    ax.axhline(sigma**2, color="0.45", linewidth=1, linestyle="--")
    ax.set_xscale("log")
    ax.set_xlim(lams_ridge.min(), lams_ridge.max())
    ax.set_ylim(0, None)
    ax.set_ylabel("MSE de teste esperado")
    ax.set_title(cenario, loc="left")
eixos[-1].set_xlabel("λ")
eixos[0].legend(frameon=False)
fig.tight_layout()
plt.show()

Para ver de onde vem cada vitória, o chunk abaixo separa o MSE de cada método, no $\lambda$ do mínimo, em viés² e variância, conta quantos coeficientes ficam diferentes de zero, em média nas 200 réplicas, e quantas colunas o lasso zera em algumas réplicas e não em outras:

In [ ]:
linhas = []
for cenario, beta in [("denso", beta_denso), ("esparso", beta_esparso)]:
    for metodo in ["ridge", "lasso"]:
        lam = melhor_lam[cenario][metodo]
        modelo = ridge(lam) if metodo == "ridge" else lasso(lam, n_sim)
        linhas.append((cenario, metodo, lam, *decompoe(beta, modelo)))
decomposicao = pd.DataFrame(
    linhas,
    columns=["cenário", "método", "λ", "viés²", "variância",
             "não nulos", "oscilam"],
).set_index(["cenário", "método"])
print("colunas que o lasso zera só às vezes:")
for cenario in ["denso", "esparso"]:
    print(f"  {cenario}:", decomposicao.loc[(cenario, "lasso"), "oscilam"])
decomposicao.drop(columns="oscilam").style.set_uuid("mecanismo").set_table_attributes(
    'style="width: auto"'
).format(decimal=",", thousands=".", precision=2)

No denso, o lasso perde pela variância. O viés² dos dois é parecido, 5,34 na ridge e 5,12 no lasso, e a variância do lasso, 10,17, é quase o dobro da da ridge, 5,18 (2 × 5,18 = 10,36). O lasso fica, em média, com 37,67 dos 45 coeficientes, e os que ele zera importavam. Quais são zerados muda de uma réplica para outra, e 43 das 45 colunas saem do modelo em algumas réplicas e ficam em outras. Cada troca de colunas muda as previsões, e é uma das fontes dessa variância.

No esparso, a ridge perde pelo viés². Ela mantém os 45 coeficientes, e o $\lambda$ que contém os 43 inúteis encolhe também os dois que importam: o viés² dela é 4,74, contra 0,40 do lasso. O lasso zera, em média, 45 − 9,24 = 35,76 dos 45 coeficientes, e ganha também na variância, 0,57 contra 1,92. Mesmo assim, ele guarda em média 9,24 colunas, mais que as 2 que importam; com o $\lambda$ do menor MSE, algumas das inúteis ficam no modelo.

Como ponto de partida, e não como regra: se a suspeita é que poucos preditores carregam quase todo o efeito, o lasso é o candidato natural, e ainda entrega um modelo mais fácil de ler; se muitos preditores contribuem um pouco cada, a ridge tende a prever melhor. Com colunas que quase se repetem, como `limite` e `pontuacao`, o lasso tende a ficar com uma só, e qual delas depende de detalhes da amostra.

Em `Credit`, com $\lambda = 10.000$, o lasso fica com seis colunas, as 11 menos as cinco zeradas na tabela dos três $\lambda$. Se esse modelo prevê melhor que a ridge, e com qual $\lambda$, depende de quantos preditores importam, e isso não se sabe de antemão num conjunto de dados real. É a validação cruzada que estima o erro de teste de cada escolha, na seção 11.5.

## Escolhendo o Parâmetro de Regularização

> **📌 Nota**
>
> Esta seção corresponde à seção 6.2.3 de James et al. (2023).

A ridge e o lasso produzem um modelo para cada valor de $\lambda$. Com $\lambda$ perto de zero, eles repetem os mínimos quadrados; com $\lambda$ grande, encolhem tudo até perto do modelo nulo, ou até ele. O RSS de treino não ajuda a escolher. Ele é mínimo nos mínimos quadrados e só piora com a penalidade, e por isso escolheria sempre $\lambda = 0$. Como escolher $\lambda$ sem um conjunto de teste?

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.lines import Line2D
from sklearn.linear_model import Lasso, LinearRegression, Ridge
from sklearn.model_selection import GridSearchCV, KFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

plt.style.use("estilo-figuras.mplstyle")

credit = pd.read_csv("dados/Credit.csv")
X = pd.get_dummies(credit.drop(columns="saldo"), drop_first=True, dtype=float)
y = credit["saldo"]
n = len(X)

numero_br = lambda valor, casas=2: (
    f"{valor:,.{casas}f}".replace(",", "_").replace(".", ",").replace("_", ".")
    .replace("-", "−")
)

### Uma grade e a validação cruzada

A resposta é a do capítulo 10: estimar o erro de teste com validação cruzada e escolher pelo erro estimado. O procedimento tem quatro passos:

1. escolher uma grade de valores de $\lambda$;
2. para cada $\lambda$ da grade, calcular o erro de validação cruzada;
3. ficar com o $\lambda$ de menor erro;
4. reajustar o modelo com esse $\lambda$, agora com todas as observações.

O último passo existe porque a validação cruzada serviu só para escolher $\lambda$. Os modelos dos grupos treinam com nove décimos das linhas cada; o modelo que vai ser usado aprende com todas.

A grade precisa ir de um $\lambda$ que quase não encolhe a um que encolhe muito, e em escala logarítmica, porque o efeito de $\lambda$ muda quando ele é multiplicado por 10, não quando se soma 1 a ele. Em `Credit`, a grade da ridge tem 51 valores, de $10^{-3}$ a $10^{2}$, e a validação cruzada usa os 10 grupos da seção 11.2, com a mesma semente. O `GridSearchCV` faz os quatro passos de uma vez. O `StandardScaler` vai dentro do `Pipeline` para que, em cada grupo, a padronização seja ajustada só com as linhas de treino; padronizar `X` antes da busca deixaria as linhas de validação entrarem na escala, o vazamento da seção 10.6. Para comparar, entra também o erro de validação cruzada dos mínimos quadrados, com os mesmos grupos.

Antes da saída, um palpite: quanto você espera que a ridge, com o $\lambda$ escolhido, baixe o erro de validação cruzada dos mínimos quadrados? 10%? 1%?

In [ ]:
grupos = KFold(10, shuffle=True, random_state=11)
lams_ridge = np.logspace(-3, 2, 51)
busca_ridge = GridSearchCV(
    make_pipeline(StandardScaler(), Ridge()),
    {"ridge__alpha": lams_ridge},
    cv=grupos,
    scoring="neg_mean_squared_error",
)
busca_ridge.fit(X, y)
cv_ridge = pd.Series(
    -busca_ridge.cv_results_["mean_test_score"], index=lams_ridge
)
lam_ridge = float(busca_ridge.best_params_["ridge__alpha"])
cv_mq = -cross_val_score(
    LinearRegression(), X, y, cv=grupos,
    scoring="neg_mean_squared_error",
).mean()
perto = cv_ridge[cv_ridge <= 1.01 * cv_ridge.min()]
trecho = cv_ridge.loc[perto.index.min():perto.index.max()]
print(f"{X.shape[0]} linhas, {X.shape[1]} colunas")
print("λ escolhido:", numero_br(lam_ridge, 3))
print("erro de validação cruzada:", numero_br(cv_ridge.min()))
print("mínimos quadrados:", numero_br(cv_mq))
print("λ a 1% do mínimo:", len(perto), "de", len(lams_ridge))
print("  de", numero_br(perto.index.min(), 3),
      "a", numero_br(perto.index.max(), 3))
print("  sem buracos:", len(trecho) == len(perto))
print("erro em λ = 100:", numero_br(cv_ridge.iloc[-1]))

In [ ]:
# Figura: 'A ridge em `Credit`. Em cima, em cinza, o erro de validação cruzada em 10 grupos contra $\lambda$, em escala logarítmica; embaixo, os coeficientes padronizados ajustados com as 400 linhas, na mesma grade. A linha tracejada marca o $\lambda$ de menor erro de validação cruzada.'
caminho_ridge = pd.DataFrame(
    [make_pipeline(StandardScaler(), Ridge(alpha=lam))
     .fit(X, y)[-1].coef_ for lam in lams_ridge],
    index=lams_ridge, columns=X.columns,
)
destaque = {
    "renda": "C0", "limite": "C1",
    "pontuacao": "C2", "estudante_sim": "C3",
}
fig, (ax_cv, ax_coef) = plt.subplots(
    2, 1, figsize=(4.8, 5.8), sharex=True, layout="constrained"
)
ax_cv.plot(lams_ridge, cv_ridge / 1000, color="0.45", linewidth=2)
ax_cv.set_ylabel("erro de validação\ncruzada (mil dólares²)")
for coluna in X.columns:
    em_destaque = coluna in destaque
    ax_coef.plot(lams_ridge, caminho_ridge[coluna],
                 color=destaque.get(coluna, "0.45"),
                 linewidth=2 if em_destaque else 1,
                 zorder=3 if em_destaque else 1)
ax_coef.axhline(0, color="0.45", linewidth=0.8)
ax_coef.set_ylim(-320, 480)
ax_coef.set_ylabel("coeficiente padronizado")
for ax in (ax_cv, ax_coef):
    ax.axvline(lam_ridge, color="0.45", linestyle="--", linewidth=1.2)
ax_coef.set_xscale("log")
ax_coef.set_xlim(lams_ridge.min(), lams_ridge.max())
ax_coef.set_xlabel("λ")
entradas = [Line2D([], [], color=c, linewidth=2)
            for c in destaque.values()]
entradas += [Line2D([], [], color="0.45", linewidth=1),
             Line2D([], [], color="0.45", linestyle="--")]
rotulos = [*destaque, "outras sete", "menor erro"]
fig.legend(entradas, rotulos, loc="outside lower center", ncols=3,
           frameon=False, fontsize="small")
plt.show()

> **🔧 Função**
>
> - **`GridSearchCV(estimador, grade, cv, scoring)`**: o erro de validação cruzada do `estimador` em cada valor da `grade`, com os grupos de `cv`. A grade é um dicionário `{"passo__parametro": valores}`.
> - **`busca.cv_results_`**: as pontuações da busca; em `"mean_test_score"`, a média nos 10 grupos, com o sinal trocado (seção 11.1).
> - **`busca.best_params_`**: o valor da grade de melhor pontuação.
> - **`fig.legend(..., loc="outside lower center")`**: a legenda da figura inteira, embaixo e fora dos painéis.

O menor erro de validação cruzada está no ponto da grade $\lambda = 0{,}316$, um encolhimento pequeno. O erro nele é 10.101,15 e o dos mínimos quadrados, 10.108,53; a ridge fica só 7,38 abaixo (10.108,53 − 10.101,15), menos de 0,1% (7,38 ÷ 10.108,53 ≈ 0,0007). O vale é raso, e 36 dos 51 valores da grade, todos os de 0,001 a 3,162, ficam a menos de 1% do erro mínimo. Na figura, é a parte plana da curva de cima; a linha tracejada cai nela, e ali os coeficientes, embaixo, já se afastaram um pouco dos da ponta esquerda.

O 10.101,15 é um pouco otimista como estimativa do erro da ridge escolhida. Os mesmos grupos mediram o erro de 51 candidatos, e o menor de 51 números medidos com ruído tende a ficar abaixo do erro verdadeiro do modelo que o produziu. Nestes dados, com 400 clientes e 11 colunas, usar os mínimos quadrados daria um erro estimado quase igual.

O quarto passo do procedimento, o reajuste com as 400 linhas, fica guardado em `best_estimator_`. Quanto os coeficientes e as previsões dele se afastam dos mínimos quadrados?

In [ ]:
ridge_final = make_pipeline(
    StandardScaler(), Ridge(alpha=lam_ridge)
).fit(X, y)
melhor = busca_ridge.best_estimator_
print("reajuste = ajuste com tudo:",
      bool(np.allclose(melhor[-1].coef_, ridge_final[-1].coef_)))
mq_padr = make_pipeline(StandardScaler(), LinearRegression()).fit(X, y)
mudanca = pd.Series(
    ridge_final[-1].coef_ - mq_padr[-1].coef_, index=X.columns
)
print("maiores mudanças nos coeficientes:")
for coluna in mudanca.abs().nlargest(2).index:
    print(f"  {coluna}: {numero_br(mudanca[coluna])}")
dif = np.abs(ridge_final.predict(X) - mq_padr.predict(X))
print("maior diferença nas previsões:", numero_br(dif.max()))
print("desvio padrão de saldo:", numero_br(y.std()))

> **🔧 Função**
>
> - **`busca.best_estimator_`**: o `Pipeline` reajustado com todas as linhas e o valor escolhido; é o modelo que se usa para prever. O reajuste é o padrão do `GridSearchCV` (`refit=True`).
> - **`np.allclose(a, b)`**: `True` se os dois arrays são iguais posição a posição, a menos do arredondamento de ponto flutuante.
> - **`cross_val_score(estimador, X, y, cv, scoring)`**, usado no chunk `grid-ridge` para os mínimos quadrados: a pontuação em cada grupo de `cv`, sem busca nem reajuste; o `.mean()` tira a média.

Os coeficientes mudaram. Em relação aos mínimos quadrados, `limite` perdeu 36,22 e `pontuacao` ganhou 35,22, em coeficientes padronizados. As duas colunas quase se repetem, como a seção 11.3 mediu, e o peso passou de uma para a outra sem mexer muito nas previsões. Nas 400 linhas, a maior diferença entre as previsões da ridge e as dos mínimos quadrados é de 7,06 dólares, contra um desvio padrão de 459,76 no `saldo`.

### O lasso em `Credit`

O mesmo procedimento vale para o lasso, com a conversão da seção 11.4, `alpha` $= \lambda/(2n)$, e a grade na escala do ajuste com as 400 linhas. Ela vai de $\lambda = 1$ a $\lambda = 10^{5}$, com 51 valores. A busca mede também o erro em $\lambda = 10.000$, o valor em que, na seção 11.4, o lasso ficou com seis colunas.

Quando vários $\lambda$ dão erros parecidos, cabe também a regra do um desvio da seção 11.2. Aqui o modelo mais simples é o de **maior** $\lambda$, e a regra fica com o maior $\lambda$ cujo erro não passa do teto: o menor erro mais a margem $s/\sqrt{10}$, com $s$ o desvio padrão dos erros dos 10 grupos no $\lambda$ de menor erro, o mesmo desvio da seção 11.2. A função abaixo calcula o teto a partir dos erros de cada grupo.

In [ ]:
def por_grupo(busca):
    """Erro de cada grupo (linhas) em cada λ (colunas)."""
    res = busca.cv_results_
    return -np.array(
        [res[f"split{g}_test_score"] for g in range(10)]
    )


def regra_do_um_desvio(busca, lams):
    """O maior λ da grade cujo erro não passa do teto, e o teto."""
    erros = por_grupo(busca)
    cv = erros.mean(axis=0)
    i = np.argmin(cv)
    teto = cv[i] + erros[:, i].std(ddof=1) / np.sqrt(10)
    return float(lams[cv <= teto].max()), teto

In [ ]:
lams_lasso = np.logspace(0, 5, 51)
lasso = Lasso(max_iter=100_000, tol=1e-8)
busca_lasso = GridSearchCV(
    make_pipeline(StandardScaler(), lasso),
    {"lasso__alpha": lams_lasso / (2 * n)},
    cv=grupos,
    scoring="neg_mean_squared_error",
)
busca_lasso.fit(X, y)
cv_lasso = pd.Series(
    -busca_lasso.cv_results_["mean_test_score"], index=lams_lasso
)
abaixo_de_1 = [
    -cross_val_score(
        make_pipeline(StandardScaler(),
                      Lasso(alpha=lam / (2 * n), max_iter=100_000,
                            tol=1e-8)),
        X, y, cv=grupos, scoring="neg_mean_squared_error",
    ).mean()
    for lam in np.logspace(-3, 0, 7)
]
lam_regra, teto = regra_do_um_desvio(busca_lasso, lams_lasso)
print("erro só sobe com λ:", bool(np.all(np.diff(cv_lasso) > 0)))
print("de λ = 0,001 a 1, só sobe:",
      bool(np.all(np.diff(abaixo_de_1) > 0)))
print("erro em λ = 1:", numero_br(cv_lasso.iloc[0]))
print("erro em λ = 10.000:", numero_br(cv_lasso.loc[1e4]))
print("teto:", numero_br(teto))
print("λ pela regra:", numero_br(lam_regra, 0))
print("erro no λ da regra:", numero_br(cv_lasso.loc[lam_regra]))

In [ ]:
# Figura: 'O lasso em `Credit`: o erro de validação cruzada em 10 grupos contra $\lambda$, em escala logarítmica, de 1 a 100 mil. A linha horizontal tracejada é o erro dos mínimos quadrados, com os mesmos grupos, e a traço-ponto, o teto da regra do um desvio; a vertical pontilhada marca o $\lambda$ que a regra escolhe. O eixo vertical vai só até 13 mil, e a curva sai pelo topo do painel depois de 10.000.'
fig, ax = plt.subplots(figsize=(4.8, 3.8), layout="constrained")
ax.plot(lams_lasso, cv_lasso / 1000, color="C0", linewidth=2)
ax.axhline(cv_mq / 1000, color="0.45", linestyle="--", linewidth=1.2)
ax.axhline(teto / 1000, color="0.45", linestyle="-.", linewidth=1.2)
ax.axvline(lam_regra, color="0.45", linestyle=":", linewidth=2)
ax.set_xscale("log")
ax.set_xlim(lams_lasso.min(), lams_lasso.max())
ax.set_ylim(9.8, 13)
ax.yaxis.set_major_formatter(
    lambda valor, _: f"{valor:.1f}".replace(".", ","))
ax.set_xlabel("λ")
ax.set_ylabel("erro de validação\ncruzada (mil dólares²)")
entradas = [
    Line2D([], [], color="C0", linewidth=2),
    Line2D([], [], color="0.45", linestyle="--"),
    Line2D([], [], color="0.45", linestyle="-."),
    Line2D([], [], color="0.45", linestyle=":", linewidth=2),
]
rotulos = ["lasso", "mínimos quadrados", "teto da regra",
           "λ da regra"]
fig.legend(entradas, rotulos, loc="outside lower center", ncols=2,
           frameon=False, fontsize="small")
plt.show()

> **🔧 Função**
>
> - **`busca.cv_results_[f"split{g}_test_score"]`**: a pontuação no grupo `g`, de 0 a 9, uma por valor da grade. Empilhadas, formam uma matriz de 10 linhas, uma por grupo, e uma coluna por $\lambda$.
> - **`lams[cv <= teto].max()`**: entre os $\lambda$ da grade com erro no teto ou abaixo dele, o maior.

Na grade inteira, o erro só sobe quando $\lambda$ cresce, e o menor fica na ponta esquerda, $\lambda = 1$, com 10.108,55. Um mínimo na borda da grade costuma pedir uma grade mais longa, porque o melhor $\lambda$ pode estar do lado de fora. Aqui, do lado de fora, o ganho é desprezível: de $\lambda = 0{,}001$ a 1 o erro também só sobe, e à esquerda o lasso encolhe cada vez menos, até virar os mínimos quadrados em $\lambda = 0$, cujo erro, 10.108,53, fica só 0,02 abaixo (10.108,55 − 10.108,53). No $\lambda$ de menor erro, o lasso também quase não encolhe em `Credit`.

O teto fica em 10.769,22, e a regra escolhe o ponto da grade $\lambda = 5.012$, com erro de validação cruzada 10.550,11, abaixo dele. Com $\lambda = 10.000$, o erro chega a 12.067,70. Quais colunas o lasso guarda com o $\lambda$ da regra, e são as mesmas que ele mantém com $\lambda = 10.000$?

In [ ]:
lasso_regra = make_pipeline(
    StandardScaler(),
    Lasso(alpha=lam_regra / (2 * n), max_iter=100_000, tol=1e-8),
).fit(X, y)
colunas_regra = X.columns[lasso_regra[-1].coef_ != 0]
lasso_10mil = make_pipeline(
    StandardScaler(),
    Lasso(alpha=1e4 / (2 * n), max_iter=100_000, tol=1e-8),
).fit(X, y)
colunas_10mil = X.columns[lasso_10mil[-1].coef_ != 0]
print("colunas pela regra:", len(colunas_regra))
for coluna in colunas_regra:
    print(" ", coluna)
print("mesmas de λ = 10.000:", set(colunas_10mil) == set(colunas_regra))

O modelo da regra fica com seis colunas, `renda`, `limite`, `pontuacao`, `cartoes`, `idade` e `estudante_sim`, as mesmas de $\lambda = 10.000$. Mesmo assim, o erro de validação cruzada é bem diferente, 12.067,70 com $\lambda = 10.000$ contra 10.550,11 com $\lambda = 5.012$. O $\lambda$ decide quais colunas ficam e quanto os coeficientes delas são encolhidos, e com $\lambda = 10.000$ o encolhimento custa 1.517,59 de erro a mais (12.067,70 − 10.550,11).

Entre os dois métodos, em `Credit`, a ridge tem o menor erro de validação cruzada, 10.101,15, contra 10.108,55 do melhor lasso. A diferença, 7,40 (10.108,55 − 10.101,15), é do tamanho da que separa a ridge dos mínimos quadrados, e as duas são pequenas diante da margem da regra do um desvio, 660,67 (10.769,22 − 10.108,55), que dá a escala da incerteza do próprio erro estimado. Nenhum dos dois ganha de fato dos mínimos quadrados. O que o lasso oferece aqui é outra coisa: pela regra do um desvio, um modelo com seis das 11 colunas, a um erro 448,96 maior que o da ridge (10.550,11 − 10.101,15), cerca de 4% (448,96 ÷ 10.101,15 ≈ 0,044).

### O lasso no cenário esparso

Em `Credit`, o erro de teste é desconhecido, e não há como saber quais colunas importam de fato. No cenário esparso do simulador das seções 11.3 e 11.4, sabe-se. Dos 45 preditores, só os dois primeiros entram na $f$ verdadeira, com coeficientes 3 e −3. Os dois são os **preditores de sinal**; os outros 43, os de **ruído**. Aqui não há 200 réplicas: a validação cruzada trabalha com um conjunto de dados só, o da primeira réplica do ruído, como se fosse o único disponível.

In [ ]:
rng = np.random.default_rng(11)
n_sim, p_sim, sigma = 50, 45, 2.0
X_sim = rng.normal(size=(n_sim, p_sim))      # matriz de treino, fixa
X_teste = rng.normal(size=(2000, p_sim))     # 2.000 pontos novos
# beta_denso não é usado aqui: é sorteado só para que os
# sorteios seguintes coincidam com os das seções 11.3 e 11.4
beta_denso = rng.normal(0, 1, p_sim)
ruidos = rng.normal(0, sigma, size=(200, n_sim))   # 200 réplicas
beta_esparso = np.zeros(p_sim)
beta_esparso[[0, 1]] = [3, -3]               # só dois importam

y_sim = X_sim @ beta_esparso + ruidos[0]     # um conjunto de dados

A grade tem 61 valores de $\lambda$, de 1 a $10^{3}$, e `alpha` $= \lambda/(2 \cdot 50)$. A divisão em 10 grupos usa o mesmo `KFold`, agora sobre as 50 linhas do conjunto simulado. Os resultados vêm primeiro em números; a figura, depois do caminho dos coeficientes, junta o erro e os coeficientes num só eixo.

In [ ]:
lams_sim = np.logspace(0, 3, 61)
busca_sim = GridSearchCV(
    make_pipeline(StandardScaler(), lasso),
    {"lasso__alpha": lams_sim / (2 * n_sim)},
    cv=grupos,
    scoring="neg_mean_squared_error",
)
busca_sim.fit(X_sim, y_sim)
cv_sim = -busca_sim.cv_results_["mean_test_score"]
lam_min = float(lams_sim[np.argmin(cv_sim)])
coef_min = busca_sim.best_estimator_[-1].coef_
ordem = np.argsort(-np.abs(coef_min))   # do maior |coef| ao menor
print("λ de menor erro:", numero_br(lam_min))
print("erro de validação cruzada:", numero_br(cv_sim.min()))
print("não nulos:", int((coef_min != 0).sum()))
print("preditores de sinal:", numero_br(coef_min[0]),
      "e", numero_br(coef_min[1]))
print("os dois maiores são os de sinal:",
      set(ordem[:2].tolist()) == {0, 1})
print("maior ruído, |coef|:", numero_br(np.abs(coef_min[2:]).max()))

> **🔧 Função**
>
> **`np.argsort(arr)`**: as posições que põem `arr` em ordem crescente; com `-np.abs(coef)`, a ordem vai do maior coeficiente em valor absoluto ao menor, e `ordem[:2]` são as posições dos dois maiores.

Com o menor erro de validação cruzada, 2,76, no ponto da grade $\lambda = 17{,}78$, o lasso fica com 13 coeficientes não nulos. Os dois maiores em valor absoluto são os dos preditores de sinal, 3,48 e −2,08. Os outros 11 (13 − 2) são preditores de ruído que o lasso não descartou, com coeficientes que não passam de 0,64 em valor absoluto.

O 2,76 fica abaixo até de $\sigma^2 = 4$, a variância do ruído (`sigma = 2.0` no simulador), e o erro de teste esperado de nenhum modelo fica abaixo de 4. De onde vem a diferença? Duas medidas ajudam. Nas 50 linhas, a média dos quadrados do ruído sorteado é o erro que a própria $f$ verdadeira teria nelas. E o erro de teste esperado do lasso escolhido é $\sigma^2$ mais a média, sobre pontos novos, do quadrado da diferença entre a previsão dele e a $f$ verdadeira, estimada aqui nos 2.000 pontos de `X_teste`.

In [ ]:
ruido2 = (ruidos[0] ** 2).mean()
print("ruído², média nas 50 linhas:", numero_br(ruido2))
print("λ da grade com erro abaixo dele:", int((cv_sim < ruido2).sum()),
      "de", len(lams_sim))
f_teste = X_teste @ beta_esparso
erro_teste = sigma**2 + (
    (f_teste - busca_sim.predict(X_teste)) ** 2
).mean()
print("erro de teste esperado dele:", numero_br(erro_teste))

A diferença tem duas partes. Uma é o sorteio: nestas 50 linhas, o ruído saiu pequeno, com média dos quadrados 3,06, abaixo de 4, e isso explica 0,94 (4 − 3,06). A outra, em boa parte, é a escolha: só 8 dos 61 $\lambda$ da grade têm erro de validação cruzada abaixo de 3,06, e o 2,76 é o menor deles, 0,30 abaixo (3,06 − 2,76). É o otimismo já apontado em `Credit`. O erro de teste esperado do lasso escolhido, estimado com os 2.000 pontos de teste em 4,97, fica acima de 4.

A regra do um desvio, com a mesma função de `Credit`:

In [ ]:
lam_regra_sim, teto_sim = regra_do_um_desvio(busca_sim, lams_sim)
coef_regra = make_pipeline(
    StandardScaler(),
    Lasso(alpha=lam_regra_sim / (2 * n_sim),
          max_iter=100_000, tol=1e-8),
).fit(X_sim, y_sim)[-1].coef_
ordem_regra = np.argsort(-np.abs(coef_regra))
print("teto:", numero_br(teto_sim))
print("λ pela regra:", numero_br(lam_regra_sim))
print("erro no λ da regra:",
      numero_br(cv_sim[lams_sim == lam_regra_sim][0]))
print("não nulos:", int((coef_regra != 0).sum()))
print("preditores de sinal:", numero_br(coef_regra[0]),
      "e", numero_br(coef_regra[1]))
print("os dois maiores são os de sinal:",
      set(ordem_regra[:2].tolist()) == {0, 1})
print("maior ruído, |coef|:", numero_br(np.abs(coef_regra[2:]).max()))

A regra vai até $\lambda = 35{,}48$. O erro de validação cruzada sobe de 2,76 para 3,20, abaixo do teto de 3,27, e em troca sobram 6 coeficientes não nulos, os dois de sinal, 3,18 e −2,00, ainda os maiores, e quatro preditores de ruído (6 − 2), o maior deles com 0,38 em valor absoluto. Como ficam os mínimos quadrados, com os 45 preditores?

In [ ]:
mq_sim = make_pipeline(StandardScaler(), LinearRegression())
cv_mq_sim = -cross_val_score(
    mq_sim, X_sim, y_sim, cv=grupos,
    scoring="neg_mean_squared_error",
).mean()
interpola = all(
    np.abs(mq_sim.fit(X_sim[tr], y_sim[tr]).predict(X_sim[tr])
           - y_sim[tr]).max() < 1e-6
    for tr, _ in grupos.split(X_sim)
)
coef_mq_sim = mq_sim.fit(X_sim, y_sim)[-1].coef_
ordem_mq = np.argsort(-np.abs(coef_mq_sim)).tolist()
print("erro de validação cruzada:", numero_br(cv_mq_sim))
print("passa pelo treino em cada grupo:", interpola)
print("preditores de sinal:", numero_br(coef_mq_sim[0]),
      "e", numero_br(coef_mq_sim[1]))
print("posição do 1.º de sinal:", ordem_mq.index(0) + 1)
print("posição do 2.º de sinal:", ordem_mq.index(1) + 1)
print("maior ruído, |coef|:",
      numero_br(np.abs(coef_mq_sim[2:]).max()))

No erro de validação cruzada, são 603,84 nos mínimos quadrados contra 2,76 do lasso no menor erro, cerca de 219 vezes menor (603,84 ÷ 2,76 ≈ 219). Em cada grupo, o treino tem 45 das 50 linhas, e os mínimos quadrados têm 46 coeficientes a ajustar, um para cada preditor e o intercepto, mais coeficientes que linhas de treino. Em todos os 10 grupos, o ajuste passa pelos pontos de treino, ruído incluído, e as previsões nas 5 linhas de fora saem muito erradas.

Nos coeficientes ajustados com as 50 linhas, o 1.º preditor de sinal tem o maior em valor absoluto, 4,30, mas o 2.º, −1,53, fica só em 4.º lugar, com dois preditores de ruído à frente dele, a maior com 1,99. Olhando só o tamanho dos coeficientes, os mínimos quadrados misturam o 2.º preditor de sinal com o ruído.

A figura mostra o erro de validação cruzada e os coeficientes do lasso ajustados com as 50 linhas, contra a razão $\lVert\hat\beta^L_\lambda\rVert_1/\lVert\hat\beta\rVert_1$ da seção 11.4, e não contra $\lambda$. A razão vai de 0, no modelo nulo, a 1, nos mínimos quadrados; num eixo de $\lambda$ em escala logarítmica, os mínimos quadrados, com $\lambda = 0$, não teriam lugar. Os coeficientes usam uma grade mais longa, de $10^{3}$ a $10^{-2}$, com 101 valores, que chega perto dos mínimos quadrados, e nela se vê em que $\lambda$ cada coluna sai do zero.

In [ ]:
lams_caminho = np.logspace(3, -2, 101)        # do maior λ ao menor
caminho_sim = np.array([
    make_pipeline(
        StandardScaler(),
        Lasso(alpha=lam / (2 * n_sim), max_iter=100_000, tol=1e-8),
    ).fit(X_sim, y_sim)[-1].coef_
    for lam in lams_caminho
])
razao_sim = (np.abs(caminho_sim).sum(axis=1)
             / np.abs(coef_mq_sim).sum())
razao_de = lambda lam: razao_sim[np.argmin(np.abs(lams_caminho - lam))]
# λ de entrada de cada coluna: o maior λ com coeficiente não nulo
entrada = np.where(caminho_sim != 0, lams_caminho[:, None], 0)
primeiras = np.argsort(-entrada.max(axis=0))[:2]
print("os de sinal entram primeiro:",
      set(primeiras.tolist()) == {0, 1})
print("razão no menor λ:", numero_br(razao_sim[-1], 3))
print("razão no menor erro:", numero_br(razao_de(lam_min)))
print("razão pela regra:", numero_br(razao_de(lam_regra_sim)))
print("razão em λ = 1:", numero_br(razao_de(1.0)))
x_coef = np.append(razao_sim, 1.0)        # os mínimos quadrados em 1
coef_todos = np.vstack([caminho_sim, coef_mq_sim])

> **🔧 Função**
>
> - **`np.where(condicao, a, b)`**: posição a posição, `a` onde a condição vale e `b` onde não vale. Aqui, `lams_caminho[:, None]` vira uma coluna, repetida para os 45 coeficientes: cada coeficiente não nulo é trocado pelo $\lambda$ da sua linha, e cada zero, por 0; o `.max(axis=0)` pega, por coluna, o maior $\lambda$ em que ela não é zero.
> - **`np.append(arr, valor)`**: um array novo, com `valor` acrescentado no fim; aqui, a razão 1 dos mínimos quadrados depois das razões da grade.
> - **`np.vstack([matriz, linha])`**: empilha as linhas, uma matriz embaixo da outra; aqui, os coeficientes dos mínimos quadrados como a última linha do caminho.

No menor $\lambda$ do caminho, a razão já é 0,994, e os mínimos quadrados entram na figura em 1. O menor erro de validação cruzada fica na razão 0,25, e a regra do um desvio, em 0,19. A curva do erro para antes, na razão 0,62, a de $\lambda = 1$, o fim da grade da validação cruzada.

In [ ]:
# Figura: 'O lasso no cenário esparso ($n = 50$, $p = 45$). Em cima, o erro de validação cruzada em 10 grupos, que termina na razão 0,62; embaixo, os coeficientes padronizados ajustados com as 50 linhas, com os mínimos quadrados em 1.'
fig, (ax_cv, ax_coef) = plt.subplots(
    2, 1, figsize=(4.8, 5.8), sharex=True, layout="constrained"
)
ax_cv.plot([razao_de(lam) for lam in lams_sim], cv_sim,
           color="0.45", linewidth=2)
ax_cv.set_ylabel("erro de validação\ncruzada (MSE)")
for j in range(2, p_sim):
    ax_coef.plot(x_coef, coef_todos[:, j], color="0.45", linewidth=0.8)
for j, cor in [(0, "C0"), (1, "C1")]:
    ax_coef.plot(x_coef, coef_todos[:, j], color=cor, linewidth=2,
                 zorder=3)
ax_coef.axhline(0, color="0.45", linewidth=0.8)
ax_coef.set_ylabel("coeficiente padronizado")
for ax in (ax_cv, ax_coef):
    ax.axvline(razao_de(lam_min), color="0.45", linestyle="--",
               linewidth=1.2)
    ax.axvline(razao_de(lam_regra_sim), color="0.45", linestyle=":",
               linewidth=2)
ax_coef.set_xlim(0, 1.02)
ax_coef.xaxis.set_major_formatter(
    lambda valor, _: f"{valor:.1f}".replace(".", ","))
ax_coef.yaxis.set_major_formatter(
    lambda valor, _: f"{valor:.0f}".replace("-", "−"))
ax_coef.set_xlabel(
    r"$\|\hat{\beta}^{\mathrm{L}}_{\lambda}\|_1 \;/\; \|\hat{\beta}\|_1$",
    fontsize=12)
entradas = [Line2D([], [], color=c, linewidth=2) for c in ["C0", "C1"]]
entradas += [
    Line2D([], [], color="0.45", linewidth=1),
    Line2D([], [], color="0.45", linestyle="--"),
    Line2D([], [], color="0.45", linestyle=":", linewidth=2),
]
rotulos = ["1.º de sinal (β = 3)", "2.º de sinal (β = −3)",
           "ruído (43)", "menor erro", "regra do um desvio"]
fig.legend(entradas, rotulos, loc="outside lower center", ncols=3,
           frameon=False, fontsize="small")
plt.show()

Da esquerda para a direita, com $\lambda$ caindo, os dois preditores de sinal são os primeiros a sair do zero, e os de ruído entram depois. Na ponta direita, nos mínimos quadrados, o 2.º preditor de sinal fica em 4.º lugar, no meio do ruído. Nas duas linhas verticais, os dois de sinal têm os maiores coeficientes. Na pontilhada, a da regra, o modelo guarda menos preditores de ruído, 4 contra 11, e os coeficientes dos preditores de sinal ficam um pouco mais encolhidos, 3,18 contra 3,48 no 1.º.

Sem um conjunto de teste, o $\lambda$ sai do erro de validação cruzada: o de menor erro na grade, ou o maior $\lambda$ que a regra do um desvio aceita. Nos dois exemplos, quanto encolher acompanha quantas linhas há para cada coluna, e também quantas colunas carregam sinal. Em `Credit`, com 400 linhas para 11 colunas, os mínimos quadrados já estimam cada coeficiente com folga, e o encolhimento tem pouca variância a cortar: o $\lambda$ de menor erro encolhe pouco, e o ganho sobre os mínimos quadrados fica abaixo de 0,1%. O que o lasso oferece ali é um modelo menor, pela regra do um desvio.

No conjunto simulado, com 50 linhas para 45 colunas e só dois preditores de sinal, é o contrário. Os mínimos quadrados passam pelos pontos de treino de cada grupo, ruído incluído, e o encolhimento corta essa variância. O lasso escolhido erra muito menos na validação cruzada, mesmo descontado o otimismo do 2,76, e dá aos dois preditores de sinal os maiores coeficientes, que os mínimos quadrados não davam.

## Regressão por Componentes Principais

> **📌 Nota**
>
> Esta seção corresponde às seções 6.3 e 6.3.1 de James et al. (2023).

Em `Credit`, `limite` e `pontuacao` quase se repetem, com correlação de 0,997 (seção 8.6), e `renda` anda junto com `limite`, com correlação de 0,792. Colunas que variam juntas repetem parte da mesma informação, e os mínimos quadrados gastam um coeficiente com cada uma. A seleção de subconjuntos escolhe algumas colunas; a ridge e o lasso encolhem os coeficientes, e o lasso leva alguns a zero. Há um terceiro caminho: trocar as colunas originais por poucas **combinações** delas e ajustar os mínimos quadrados sobre as combinações. Quando isso funciona, e quando não funciona?

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.lines import Line2D
from sklearn.decomposition import PCA
from sklearn.linear_model import Lasso, LinearRegression, Ridge
from sklearn.model_selection import GridSearchCV, KFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

plt.style.use("estilo-figuras.mplstyle")

credit = pd.read_csv("dados/Credit.csv")
X = pd.get_dummies(credit.drop(columns="saldo"), drop_first=True, dtype=float)
y = credit["saldo"]

numero_br = lambda valor, casas=2: (
    f"{valor:,.{casas}f}".replace(",", "_").replace(".", ",").replace("_", ".")
    .replace("-", "−")
)
virgula = lambda casas: (
    lambda valor, _: f"{valor:.{casas}f}".replace(".", ",").replace("-", "−")
)

### Combinações lineares das colunas

Em `Credit`, uma coluna nova pode ser a média de `limite` e `pontuacao`, as duas padronizadas: para cada cliente, metade de um valor mais metade do outro. Como as duas quase se repetem, essa média guarda quase tudo o que elas dizem, numa coluna só. Ela é uma **combinação linear** das colunas: uma soma das colunas originais, cada uma multiplicada por um peso, aqui $1/2$ e $1/2$. A média serve como uma coluna só?

In [ ]:
padronizadas = StandardScaler().fit_transform(X[["limite", "pontuacao"]])
media = padronizadas.mean(axis=1)    # metade de cada, cliente a cliente
for nome, coluna in [("limite", padronizadas[:, 0]),
                     ("pontuacao", padronizadas[:, 1]),
                     ("média", media)]:
    print(f"corr({nome}, saldo):",
          numero_br(np.corrcoef(coluna, y)[0, 1], 3))

> **🔧 Função**
>
> - **`StandardScaler().fit_transform(df)`**: ajusta o padronizador às colunas de `df` e já devolve as colunas padronizadas, numa chamada só, como um array.
> - **`np.corrcoef(a, b)`**: a matriz de correlações entre `a` e `b`, de $2 \times 2$; o elemento `[0, 1]` é a correlação entre os dois.

Com `saldo`, a média tem correlação de 0,863, contra 0,862 de `limite` e 0,864 de `pontuacao`: a coluna nova guarda a relação das duas com a resposta.

Em geral, com os preditores $X_1, \dots, X_p$, a combinação de número $m$ se chama $Z_m$, e o peso de $X_j$ nela se chama $\phi_{jm}$:

$$
Z_m = \sum_{j=1}^{p}\phi_{jm}X_j.
$$

São $M$ combinações, $m = 1, \dots, M$, com $M \le p$; para reduzir a dimensão, $M < p$. No exemplo, $M = 1$, os pesos de `limite` e `pontuacao` valem $1/2$, e os das outras colunas, zero.

Escolhidos os pesos, o modelo é a regressão linear de $Y$ sobre as $M$ colunas novas, ajustada por mínimos quadrados. O valor de $Z_m$ na observação $i$ é $z_{im}$, o intercepto é $\theta_0$, e $\theta_m$ é o coeficiente de $Z_m$:

$$
y_i = \theta_0 + \sum_{m=1}^{M}\theta_m z_{im} + \varepsilon_i,
$$

para $i = 1, \dots, n$. Em vez de $p + 1$ coeficientes, $\beta_0, \beta_1, \dots, \beta_p$, o ajuste estima $M + 1$, $\theta_0, \theta_1, \dots, \theta_M$. É a **redução de dimensão**: o problema passa de $p$ dimensões para $M$.

O modelo é linear nas colunas $Z_m$. Continua linear nas colunas originais $X_j$? Substituindo $z_{im}$ pela soma que define $Z_m$ e juntando os termos de cada coluna,

$$
\begin{aligned}
\sum_{m=1}^{M}\theta_m z_{im}
&= \sum_{m=1}^{M}\theta_m\sum_{j=1}^{p}\phi_{jm}x_{ij} \\
&= \sum_{j=1}^{p}\beta_j x_{ij},
\end{aligned}
$$

com

$$
\beta_j = \sum_{m=1}^{M}\theta_m\phi_{jm}.
$$

No exemplo da média, com $M = 1$ e pesos $1/2$ em `limite` e `pontuacao`, a soma tem um termo só: $\beta_{\text{limite}} = \beta_{\text{pontuacao}} = \theta_1/2$, e o $\beta_j$ das outras colunas é zero. Qualquer que seja o $\theta_1$ ajustado, as duas colunas ficam com o mesmo coeficiente.

Cada $\beta_j$ ainda existe, mas não é livre: os $p$ coeficientes saem de só $M$ números, os $\theta_m$. Essa amarra introduz viés quando a melhor combinação de coeficientes não tem essa forma. Em troca, com $M$ bem menor que $p$, há menos coeficientes a estimar com os mesmos $n$ pontos, e a variância do ajuste cai. Com $M = p$ e combinações linearmente independentes, isto é, nenhuma delas uma soma com pesos das outras, a amarra some e o ajuste é o dos mínimos quadrados.

Falta dizer de onde vêm os pesos $\phi_{jm}$. Um jeito de escolhê-los é pelas **componentes principais**.

### A direção de maior variância

Com duas colunas, dá para ver de onde vêm os pesos. As duas aqui são `renda` e `limite`, padronizadas como na seção 11.3. Cada cliente é um ponto no plano. Uma combinação $Z_1 = \phi_{11}\,\text{renda} + \phi_{21}\,\text{limite}$, com $\phi_{11}^2 + \phi_{21}^2 = 1$, dá a cada ponto a sua posição ao longo de uma reta que passa pela origem: o pé do segmento perpendicular que vai do ponto até a reta. Sem a condição $\phi_{11}^2 + \phi_{21}^2 = 1$, a variância dessas posições cresceria sem limite só com pesos maiores. Com ela, cada reta dá um espalhamento diferente, e uma delas espalha mais que todas as outras.

Os pesos $\phi_{j1}$ dessa reta se chamam **cargas**, e a posição $z_{i1}$ de cada ponto ao longo dela, o **escore** do ponto. Antes de rodar a célula: com `renda` e `limite` subindo juntas, para onde você espera que aponte a reta de maior espalhamento?

In [ ]:
duas = StandardScaler().fit_transform(X[["renda", "limite"]])
pca2 = PCA().fit(duas)
escores2 = pca2.transform(duas)
cargas = np.abs(pca2.components_)
print("corr(renda, limite):",
      numero_br(X["renda"].corr(X["limite"]), 3))
print("corr(limite, pontuacao):",
      numero_br(X["limite"].corr(X["pontuacao"]), 3))
print("|cargas| da 1.ª:", numero_br(cargas[0, 0], 3),
      numero_br(cargas[0, 1], 3))
print("|cargas| da 2.ª:", numero_br(cargas[1, 0], 3),
      numero_br(cargas[1, 1], 3))
print("mesmo sinal na 1.ª:",
      bool(np.sign(pca2.components_[0, 0])
           == np.sign(pca2.components_[0, 1])))
print("sinais opostos na 2.ª:",
      bool(np.sign(pca2.components_[1, 0])
           != np.sign(pca2.components_[1, 1])))
print("variância explicada:",
      *[numero_br(v, 3) for v in pca2.explained_variance_ratio_])
print("variância dos escores:",
      *[numero_br(v, 3) for v in escores2.var(axis=0)])
print("corr(z1, z2) ≈ 0:",
      bool(np.isclose(np.corrcoef(escores2.T)[0, 1], 0)))

> **🔧 Função**
>
> - **`PCA(n_components)`**: a análise de componentes principais (PCA, tema do capítulo 15); sem `n_components`, calcula todas as componentes, uma por coluna. A `PCA` só centra as colunas; quem divide pelo desvio padrão é o `StandardScaler`.
> - **`pca.components_`**: as cargas, uma linha por componente. **`pca.explained_variance_ratio_`**: a fração da variância total que cada componente carrega. **`pca.transform(X)`**: os escores, uma coluna por componente.
> - **`np.sign(x)`**: 1, −1 ou 0, conforme o sinal de `x`. **`np.corrcoef(A.T)`**: com uma matriz só, as correlações entre as colunas de `A`.

In [ ]:
# Figura: 'As 400 observações de `Credit` em `renda` e `limite`, as duas padronizadas. Em cima, a linha sólida laranja é a direção da primeira componente principal, e a tracejada cinza, a da segunda. Embaixo, os mesmos pontos com os escores das duas componentes nos eixos: a nuvem de cima girada até a primeira direção ficar na horizontal, e a linha laranja e a tracejada de baixo são as mesmas duas direções, giradas junto. Os dois pontos roxos são os mesmos clientes nos dois painéis; o segmento de cada um vai até a direção da primeira componente. Em cada painel, os dois eixos têm a mesma escala.'
# o sinal de cada componente é arbitrário; para o painel de baixo ser o de
# cima girado, o 2.º escore usa o sinal da 1.ª direção girada de 90 graus
girada = np.array([-pca2.components_[0, 1], pca2.components_[0, 0]])
giro = escores2 * [1, np.sign(pca2.components_[1] @ girada)]
destacados = [giro[:, 1].argmax(), giro[:, 1].argmin()]
fig, (ax_orig, ax_giro) = plt.subplots(
    2, 1, figsize=(4.8, 5.6), layout="constrained",
    gridspec_kw={"height_ratios": [3, 2]},
)
ax_orig.scatter(duas[:, 0], duas[:, 1], s=12, alpha=0.6,
                color="C0", linewidths=0)
for k, estilo in [(0, {"color": "C1", "linewidth": 2}),
                  (1, {"color": "0.45", "linewidth": 1.5,
                       "linestyle": "--"})]:
    direcao = pca2.components_[k]
    ax_orig.axline((0, 0), direcao, **estilo)
ax_giro.scatter(giro[:, 0], giro[:, 1], s=12, alpha=0.6,
                color="C0", linewidths=0)
ax_giro.axhline(0, color="C1", linewidth=2)
ax_giro.axvline(0, color="0.45", linewidth=1.5, linestyle="--")
for i in destacados:
    z1, z2 = giro[i]
    pe = z1 * pca2.components_[0]      # o ponto projetado na direção
    ax_orig.plot([duas[i, 0], pe[0]], [duas[i, 1], pe[1]],
                 color="C3", linewidth=1.5, zorder=3)
    ax_orig.scatter(*duas[i], s=45, color="C3", zorder=4)
    ax_giro.plot([z1, z1], [z2, 0], color="C3", linewidth=1.5, zorder=3)
    ax_giro.scatter(z1, z2, s=45, color="C3", zorder=4)
ax_orig.set_xlabel("renda (padronizada)")
ax_orig.set_ylabel("limite (padronizado)")
ax_giro.set_xlabel("escore da 1.ª componente, $z_{i1}$")
ax_giro.set_ylabel("escore da 2.ª componente")
for ax in (ax_orig, ax_giro):
    ax.set_aspect("equal")
    ax.xaxis.set_major_formatter(virgula(0))
    ax.yaxis.set_major_formatter(virgula(0))
ax_orig.set_xlim(-2.5, 6)
ax_orig.set_ylim(-2.5, 4.5)
ax_giro.set_xlim(giro[:, 0].min() - 0.5, giro[:, 0].max() + 0.5)
ax_giro.set_ylim(-1.5, 1.5)
plt.show()

> **🔧 Função**
>
> - **`ax.axline(ponto, direcao)`**: a reta que passa por `ponto` na `direcao` dada, de ponta a ponta do painel; trocar o sinal da direção desenha a mesma reta.
> - **`gridspec_kw={"height_ratios": [3, 2]}`**, em `plt.subplots`: as alturas relativas dos painéis. **`arr.argmax()`** e **`arr.argmin()`**: a posição do maior e do menor valor.

A reta laranja de cima segue o comprimento da nuvem. Em cada ponto roxo, o pé do segmento marca a posição do cliente ao longo dessa reta, e é essa posição que vira a coordenada horizontal embaixo; o comprimento do segmento, positivo de um lado da reta e negativo do outro, vira a coordenada vertical.

> **🔷 Conceito**
>
> A **primeira componente principal** de um conjunto de colunas é a combinação $Z_1 = \sum_j\phi_{j1}X_j$, com $\sum_j\phi_{j1}^2 = 1$, de maior variância: a direção ao longo da qual os pontos mais se espalham. Os pesos $\phi_{j1}$ são as **cargas** da componente, e o valor $z_{i1}$ de cada observação é o **escore** dela.
>
> A segunda componente é a combinação de maior variância entre as que não têm correlação com a primeira, e cada componente seguinte é a de maior variância entre as que não têm correlação com as anteriores, até $p$ componentes.

As duas cargas da primeira componente têm o mesmo tamanho, 0,707, e o mesmo sinal: o escore $z_{i1}$ é, a menos da escala e do sinal, a média de `renda` e `limite` padronizadas do cliente $i$. Numa ponta da escala do escore ficam os clientes com renda e limite acima da média; na outra, os com os dois abaixo. Com as duas colunas juntas, a primeira componente carrega 0,896 da variância total, e a segunda, 0,104.

Com duas colunas, não ter correlação com a primeira componente equivale a ter a direção perpendicular à dela, a tracejada. As cargas da segunda têm também o tamanho 0,707, com sinais opostos: o escore $z_{i2}$ mede o quanto `renda` e `limite` do cliente se afastam uma da outra. Os escores da segunda ficam bem mais perto de zero que os da primeira: com correlação de 0,792 entre renda e limite, a segunda componente carrega só 0,104 da variância.

Com duas colunas padronizadas, as cargas saem 0,707 $= 1/\sqrt{2}$ para qualquer correlação diferente de zero entre elas; a correlação decide só a divisão da variância. As variâncias dos escores, 1,792 e 0,208, são $1 + 0{,}792$ e $1 - 0{,}792$, e 1,792 ÷ 2 = 0,896 é a fração da primeira.

### A regressão sobre as componentes

Em `renda` e `limite`, a primeira componente sozinha carrega 0,896 da variância. Quando as primeiras componentes de $X$ concentram quase toda a variância, dá para trocar as $p$ colunas por elas e ajustar a regressão só sobre as $M$ primeiras.

> **🔷 Conceito**
>
> A **regressão por componentes principais** (PCR, do inglês *principal components regression*) usa os escores das $M$ primeiras componentes principais de $X$ como as colunas $Z_1, \dots, Z_M$ da regressão acima e ajusta $Y$ sobre elas por mínimos quadrados. As colunas de $X$ são padronizadas antes, como na ridge; sem isso, a coluna de maior variância na sua unidade dominaria as primeiras componentes.
>
> As componentes são calculadas só com $X$: a resposta não entra na escolha das direções. A PCR aposta que as direções em que $X$ mais varia são também as que importam para prever $Y$.

A aposta pode falhar. Nada obriga a resposta a depender das direções em que $X$ mais varia; uma direção de pouca variância pode ser justamente a que prevê $Y$.

No `scikit-learn`, a PCR é um `Pipeline` de três passos: `StandardScaler`, `PCA(n_components=M)` e `LinearRegression`. O número de componentes $M$ faz o papel do $\lambda$ da ridge, no sentido inverso: com $M$ pequeno, o modelo é rígido e tem viés; com $M = p$, é o ajuste dos mínimos quadrados.

Vale o mesmo simulador das seções 11.3 a 11.5, agora medindo o viés² e a variância da PCR para cada $M$: $n = 50$ observações, $p = 45$ preditores sorteados de forma independente, a mesma matriz de treino em todas as réplicas e 200 réplicas do ruído, de desvio $\sigma = 2$. Os dois cenários diferem só nos coeficientes verdadeiros:

| Cenário | Coeficientes verdadeiros |
|---|---|
| **denso** | os 45 sorteados da normal padrão |
| **esparso** | $\beta_1 = 3$ e $\beta_2 = -3$; os outros 43 são zero |

O viés² e a variância são estimados com as 200 réplicas e 2.000 pontos de teste, como na seção 11.3, e o MSE de teste é a soma dos dois mais $\sigma^2 = 4$.

In [ ]:
rng = np.random.default_rng(11)
n_sim, p_sim, sigma = 50, 45, 2.0
X_sim = rng.normal(size=(n_sim, p_sim))      # matriz de treino, fixa
X_teste = rng.normal(size=(2000, p_sim))     # 2.000 pontos novos
beta_denso = rng.normal(0, 1, p_sim)         # os 45 importam
ruidos = rng.normal(0, sigma, size=(200, n_sim))   # 200 réplicas
beta_esparso = np.zeros(p_sim)
beta_esparso[[0, 1]] = [3, -3]               # só dois importam
cenarios = {"denso": beta_denso, "esparso": beta_esparso}


def decompoe(beta, modelo, treino=X_sim, teste=X_teste):
    """Viés², variância e MSE de teste sobre as 200 réplicas."""
    f_treino, f_teste = treino @ beta, teste @ beta
    previsoes = np.array([
        modelo.fit(treino, f_treino + ruido).predict(teste)
        for ruido in ruidos
    ])
    vies2 = float(((previsoes.mean(axis=0) - f_teste) ** 2).mean())
    variancia = float(previsoes.var(axis=0).mean())
    return {"viés²": vies2, "variância": variancia,
            "MSE de teste": vies2 + variancia + sigma**2}


def pcr(M):
    """A PCR com M componentes, nas colunas padronizadas."""
    return make_pipeline(
        StandardScaler(),
        PCA(n_components=M, svd_solver="covariance_eigh"),
        LinearRegression(),
    )

> **🔧 Função**
>
> **`PCA(..., svd_solver="covariance_eigh")`**: calcula as componentes pela matriz de covariâncias das colunas, uma matriz de $p \times p$. Dá as mesmas direções que o cálculo padrão, a menos do sinal, com uma conta menor, o que pesa em milhares de ajustes.

A grade vai de $M = 1$ a $M = 45$, de dois em dois: em cada cenário, 23 valores de $M$ × 200 réplicas = 4.600 ajustes. A célula demora a terminar, e o tempo depende da máquina: espere a saída aparecer. Antes de rodar a célula: onde você espera o menor MSE de teste, com $M = 1$, com $M = 45$ ou no meio?

In [ ]:
Ms = range(1, p_sim + 1, 2)
print("valores de M na grade:", len(Ms))
curvas = {
    nome: pd.DataFrame([decompoe(beta, pcr(M)) for M in Ms], index=Ms)
    for nome, beta in cenarios.items()
}
for nome, curva in curvas.items():
    mse = curva["MSE de teste"]
    print(f"{nome}: menor MSE {numero_br(mse.min())}"
          f" em M = {mse.idxmin()}")
    print("  MSE em M = 45:", numero_br(mse.loc[45]))
    print("  viés², M = 1 e 45:",
          numero_br(curva.loc[1, "viés²"]),
          numero_br(curva.loc[45, "viés²"]))
    print("  variância, M = 1 e 45:",
          numero_br(curva.loc[1, "variância"]),
          numero_br(curva.loc[45, "variância"]))
    print("  variância só sobe:",
          bool((curva["variância"].diff().dropna() > 0).all()))
print("variância igual nos dois:",
      bool(np.allclose(curvas["denso"]["variância"],
                       curvas["esparso"]["variância"])))
mq = make_pipeline(StandardScaler(), LinearRegression())
print("M = 45 = mín. quadrados:",
      all(np.isclose(decompoe(beta, mq)["MSE de teste"],
                     curvas[nome].loc[45, "MSE de teste"])
          for nome, beta in cenarios.items()))

In [ ]:
# Figura: 'Viés², variância e MSE de teste da PCR no simulador ($n = 50$, $p = 45$), contra o número de componentes $M$, de 1 a 45, de dois em dois. Em cima, o cenário denso; embaixo, o esparso. A linha tracejada cinza é o piso $\sigma^2 = 4$. Em $M = 45$, a PCR é o ajuste dos mínimos quadrados.'
fig, eixos = plt.subplots(2, 1, figsize=(4.8, 5.8), sharex=True,
                          layout="constrained")
cores = {"viés²": "C0", "variância": "C2", "MSE de teste": "C3"}
for ax, (nome, curva) in zip(eixos, curvas.items()):
    for coluna, cor in cores.items():
        ax.plot(curva.index, curva[coluna], color=cor, linewidth=2)
    ax.axhline(sigma**2, color="0.45", linestyle="--", linewidth=1.2)
    ax.set_ylim(0, 60)
    ax.set_ylabel("erro quadrático médio")
    ax.set_title(nome, loc="left")
eixos[-1].set_xlim(0, 46)
eixos[-1].set_xlabel("número de componentes, M")
entradas = [Line2D([], [], color=c, linewidth=2) for c in cores.values()]
entradas.append(Line2D([], [], color="0.45", linestyle="--"))
fig.legend(entradas, [*cores, "piso, σ² = 4"],
           loc="outside lower center", ncols=2, frameon=False,
           fontsize="small")
plt.show()

Lida da direita para a esquerda, a figura repete a troca entre viés e variância que a ridge mostrou. Com $M = 45$, as 45 componentes refazem as 45 colunas, e o MSE de teste, 53,50 nos dois cenários, é o dos mínimos quadrados. A variância sobe a cada passo da grade, de 0,10 em $M = 1$ a 49,38 em $M = 45$, e é a mesma nos dois cenários: a variância das previsões depende da matriz de treino, das componentes usadas e do ruído, não dos coeficientes verdadeiros. O viés² é o que as poucas componentes custam: 44,02 no denso e 15,93 no esparso, em $M = 1$, contra 0,12 nos dois, em $M = 45$; com poucas componentes, o modelo não alcança a $f$ verdadeira.

O menor MSE de teste da grade fica em $M = 41$ no denso, 18,50, e em $M = 29$ no esparso, 12,21. Nos dois, a PCR com esse $M$ fica muito abaixo dos 53,50 dos mínimos quadrados. Mas precisa de muitas componentes: no denso, 41 das 45. Esse $M$ é uma escolha de oráculo: ele sai de 2.000 pontos novos com a $f$ verdadeira conhecida, o que nenhum dado real oferece. Serve para comparar métodos julgados pelo mesmo critério; num dado real, quem escolhe $M$ é a validação cruzada, como em `Credit`, adiante.

### A PCR contra a ridge e o lasso

Nesses mesmos dados, a ridge e o lasso fazem melhor? Cada um entra com as grades de $\lambda$ da seção 11.4 e é julgado pelo menor MSE de teste da sua grade, o mesmo critério de oráculo da PCR. Esta célula também demora.

In [ ]:
def ridge(lam):
    return make_pipeline(StandardScaler(), Ridge(alpha=lam))


def lasso(lam):
    return make_pipeline(
        StandardScaler(),
        # alpha = λ/(2n), como na seção 11.4
        Lasso(alpha=lam / (2 * n_sim), max_iter=100_000, tol=1e-8),
    )


grades = {"ridge": (ridge, np.logspace(-2, 4, 31)),
          "lasso": (lasso, np.logspace(-1, 3, 31))}
tabela = pd.DataFrame({
    nome: {
        "mínimos quadrados": curvas[nome].loc[45, "MSE de teste"],
        "PCR": curvas[nome]["MSE de teste"].min(),
        **{metodo: min(decompoe(beta, faz(lam))["MSE de teste"]
                       for lam in lams)
           for metodo, (faz, lams) in grades.items()},
    }
    for nome, beta in cenarios.items()
})
for nome in tabela.columns:
    print(f"MSE crescente, {nome}:")
    print(", ".join(tabela[nome].sort_values().index))
tabela.style.set_uuid("pcr-contra").format(numero_br)

A PCR fica atrás do melhor método nos dois cenários. No denso, a ridge chega a 14,52, contra 18,50 da PCR, e a PCR ganha do lasso, que chega a 19,29. No esparso, o lasso chega a 4,96, contra 12,21, e a PCR fica atrás também da ridge, que chega a 10,67.

Por que tantas componentes? A PCR aposta que a $f$ verdadeira mora nas primeiras direções de $X$. Nos dados do simulador, os 45 preditores foram sorteados de forma independente. Duas frações, medidas nas 50 linhas de treino para cada $M$, mostram onde a $f$ mora. A primeira é a fração da variância de $X$ que as $M$ primeiras componentes carregam. A segunda é a fração da variância da $f$ verdadeira que uma regressão sobre essas componentes alcança, o $R^2$ dessa regressão.

In [ ]:
pca_sim = make_pipeline(
    StandardScaler(), PCA(svd_solver="covariance_eigh")
).fit(X_sim)
escores = pca_sim.transform(X_sim)
fracoes = pd.DataFrame(
    {"variância de X": pca_sim[-1].explained_variance_ratio_.cumsum()},
    index=range(1, p_sim + 1),
)
fracoes.index.name = "M"
for nome, beta in cenarios.items():
    f = X_sim @ beta                      # a f verdadeira, sem ruído
    fracoes[f"f, {nome}"] = [
        LinearRegression().fit(escores[:, :M], f)
        .score(escores[:, :M], f)
        for M in fracoes.index
    ]
print("variância de X na 1.ª:",
      numero_br(fracoes.iloc[0, 0], 3))
fracoes.loc[[5, 15, 25, 35]].style.set_uuid("onde-mora-f").format(
    lambda v: numero_br(v, 3))

> **🔧 Função**
>
> - **`arr.cumsum()`**: a soma acumulada, posição a posição; a linha $M$ da tabela soma as frações das $M$ primeiras componentes.
> - **`escores[:, :M]`**: todas as linhas e as $M$ primeiras colunas, os escores das $M$ primeiras componentes.
> - **`modelo.score(X, y)`**: numa regressão, o $R^2$ das previsões de `modelo` em `X` contra `y` (seção 8.2).

Nenhuma direção concentra muito da variância de $X$: a primeira componente carrega só 0,081, e as cinco primeiras, 0,338. A $f$ verdadeira acompanha essa divisão: com $M = 5$, a regressão sobre as componentes alcança 0,405 da variância de $f$ no denso e 0,295 no esparso, frações da mesma ordem da de $X$. Com $M = 25$, mais da metade das componentes, a regressão alcança 0,934 da $f$ no denso e 0,917 no esparso; o resto está espalhado pelas outras 20, e buscá-lo traz a variância de volta.

No esparso, a $f$ depende de duas colunas originais, $3X_1 - 3X_2$, e essa direção se espalha por muitas componentes. A PCR não seleciona colunas: cada componente é uma combinação das 45, e o modelo não tem como ficar só com $X_1$ e $X_2$, como o lasso pode ficar, ao zerar coeficientes.

### Quando a PCR funciona

A aposta da PCR dá certo quando as colunas de $X$ variam juntas porque vêm de poucas fontes comuns, e a $f$ depende de $X$ por essas mesmas fontes. Para ver isso, as 45 colunas do simulador passam a ser geradas de outro jeito: cada linha sorteia três **fatores** escondidos, e cada coluna é uma mistura dos três, com pesos de mistura sorteados uma vez para todas as linhas, mais um ruído de desvio 0,5. Os coeficientes verdadeiros somam os pesos de mistura dos três fatores, multiplicados por 2, −1 e 1, para a $f$ depender de todos eles.

O ruído da resposta e as 200 réplicas continuam os mesmos; os 2.000 pontos de teste são novos, sorteados do mesmo jeito que as linhas de treino. A grade de $M$ agora vai de um em um, de 1 a 45, e a célula também demora. Quantas componentes a PCR vai usar?

In [ ]:
rng_fatores = np.random.default_rng(12)
pesos_fatores = rng_fatores.normal(size=(3, p_sim))   # 3 fatores × 45 colunas


def com_fatores(linhas):
    """Linhas de X geradas por 3 fatores escondidos, mais um ruído pequeno."""
    fatores = rng_fatores.normal(size=(linhas, 3))
    return (fatores @ pesos_fatores
            + 0.5 * rng_fatores.normal(size=(linhas, p_sim)))


X_fatores, X_fatores_teste = com_fatores(n_sim), com_fatores(2000)
# a f combina os três fatores com pesos fixos 2, −1 e 1;
# o / 10 só reduz a escala da f
beta_fatores = pesos_fatores.T @ np.array([2, -1, 1]) / 10
dados_fatores = {"treino": X_fatores, "teste": X_fatores_teste}
Ms_todos = range(1, p_sim + 1)          # a grade inteira, de 1 em 1
mse_fatores = pd.Series({
    M: decompoe(beta_fatores, pcr(M), **dados_fatores)["MSE de teste"]
    for M in Ms_todos
})
mse_ridge_fatores = min(
    decompoe(beta_fatores, ridge(lam), **dados_fatores)["MSE de teste"]
    for lam in grades["ridge"][1]
)
pca_fatores = make_pipeline(StandardScaler(), PCA()).fit(X_fatores)
escores_fatores = pca_fatores.transform(X_fatores)
f_fatores = X_fatores @ beta_fatores     # a f verdadeira no treino
r2_f = [LinearRegression().fit(escores_fatores[:, :M], f_fatores)
        .score(escores_fatores[:, :M], f_fatores) for M in (1, 2, 3)]
print("variância das 3 primeiras:", numero_br(
    pca_fatores[-1].explained_variance_ratio_[:3].sum(), 3))
print("R² da f com M = 1, 2 e 3:")
print(" ", *[numero_br(r2, 3) for r2 in r2_f])
print("PCR, M de menor MSE:", mse_fatores.idxmin())
print("PCR, menor MSE:", numero_br(mse_fatores.min()))
print("PCR, M = 1:", numero_br(mse_fatores.loc[1]))
print("mínimos quadrados:", numero_br(mse_fatores.loc[45]))
print("ridge, menor MSE:", numero_br(mse_ridge_fatores))
print("PCR abaixo da ridge:", bool(mse_fatores.min() < mse_ridge_fatores))

> **🔧 Função**
>
> **`decompoe(beta, modelo, **dados_fatores)`**: o `**` desempacota o dicionário em argumentos nomeados, aqui `treino=X_fatores` e `teste=X_fatores_teste`; sem eles, `decompoe` usa as matrizes do simulador.

In [ ]:
# Figura: 'MSE de teste da PCR no caso dos três fatores escondidos, contra o número de componentes $M$, de 1 a 45, de um em um. O ponto laranja marca o menor valor da grade, 4,79, em $M = 3$; a linha tracejada cinza é o piso $\sigma^2 = 4$. Em $M = 45$, a PCR é o ajuste dos mínimos quadrados, com 79,98.'
fig, ax = plt.subplots(figsize=(4.8, 3.4), layout="constrained")
ax.plot(mse_fatores.index, mse_fatores, color="C0", linewidth=2)
ax.axhline(sigma**2, color="0.45", linestyle="--", linewidth=1.2)
M_min = mse_fatores.idxmin()
ax.plot(M_min, mse_fatores.min(), "o", color="C1", markersize=8, zorder=3)
ax.annotate(f"M = {M_min}: {numero_br(mse_fatores.min())}",
            (M_min, mse_fatores.min()), xytext=(12, 28),
            textcoords="offset points", fontsize="small",
            arrowprops={"arrowstyle": "-", "color": "0.45"})
ax.text(44, sigma**2 - 1, "piso, σ² = 4", ha="right", va="top",
        fontsize="small", color="0.45")
ax.set_xlim(0, 46)
ax.set_ylim(0, 85)
ax.set_xlabel("número de componentes, M")
ax.set_ylabel("MSE de teste")
ax.yaxis.set_major_formatter(virgula(0))
plt.show()

As três primeiras componentes carregam 0,864 da variância de $X$, e a $f$ precisa das três: a regressão sobre a primeira alcança 0,633 da variância da $f$; sobre as duas primeiras, 0,801; sobre as três, 0,998. Na grade inteira, a PCR chega ao menor MSE de teste com $M = 3$, uma componente por fator: 4,79, perto do piso $\sigma^2 = 4$, contra 62,04 com $M = 1$ e 79,98 dos mínimos quadrados, com $M = 45$. Aqui a PCR fica abaixo até da ridge, que chega a 5,02. Quando as colunas vêm de poucos fatores e a $f$ depende deles, poucas componentes guardam o que serve para prever, e o resto das direções só traz variância.

### PCR em `Credit`

`Credit` tem 400 clientes, e a resposta é `saldo`, a dívida no cartão. Os preditores são 11 colunas: seis numéricas (`renda`, `limite`, `pontuacao`, `cartoes`, `idade`, `escolaridade`) e cinco indicadoras, `imovel_proprio_sim`, `estudante_sim` e `casado_sim`, mais duas para `regiao`, `regiao_Oeste` e `regiao_Sul`, com `Leste` como base.

A PCR é ajustada com $M$ de 1 a 11 componentes e avaliada pelo erro de validação cruzada dos 10 grupos das seções 11.2 e 11.5, com a mesma semente, num `GridSearchCV` como o da seção 11.5, agora com a grade em `pca__n_components`. Como o `StandardScaler` e a `PCA` estão dentro do `Pipeline`, a padronização e as direções são refeitas em cada grupo só com as linhas de treino, e as linhas de validação não entram na escolha das componentes, o que evita o vazamento da seção 10.6. Para comparar, entra o erro dos mínimos quadrados nos mesmos grupos. Quantas componentes a validação cruzada vai escolher?

In [ ]:
grupos = KFold(10, shuffle=True, random_state=11)
Ms_credit = range(1, X.shape[1] + 1)
busca = GridSearchCV(
    make_pipeline(StandardScaler(), PCA(), LinearRegression()),
    {"pca__n_components": list(Ms_credit)},
    cv=grupos, scoring="neg_mean_squared_error",
).fit(X, y)
cv_pcr = pd.Series(-busca.cv_results_["mean_test_score"],
                   index=Ms_credit)
cv_mq = -cross_val_score(LinearRegression(), X, y, cv=grupos,
                         scoring="neg_mean_squared_error").mean()
quedas = -cv_pcr.diff()        # quanto o erro cai de M − 1 para M
print("preditores:", X.shape[1], "| clientes:", X.shape[0])
print("M de menor erro:", cv_pcr.idxmin())
print("igual aos mín. quadrados:",
      bool(np.isclose(cv_pcr.min(), cv_mq)))
print("menor erro com M ≤ 9:", numero_br(cv_pcr.loc[:9].min()))
print("  em M =", cv_pcr.loc[:9].idxmin())
print("maior queda: de M =", quedas.idxmax() - 1,
      "a", quedas.idxmax())
for M in [1, 2, 7, 8, 9, 10, 11]:
    print(f"M = {M:>2}: erro {numero_br(cv_pcr[M])}")

O menor erro de validação cruzada está em $M = 11$, com todas as componentes, e é igual ao dos mínimos quadrados. Antes disso, o erro tem um trecho de queda e um patamar. Em $M = 2$, ele sobe um pouco, de 89.212,37 para 89.695,30, antes de cair; de $M = 1$ a $M = 7$, cai no total para 69.645,09, cerca de 22% a menos ((89.212,37 − 69.645,09) ÷ 89.212,37); em $M = 8$ e $M = 9$, fica em torno desse nível, com 69.736,74 e 70.670,76. De $M = 9$ para $M = 10$, ele cai para 10.130,53, a maior queda da curva, e em $M = 11$ ainda desce 22,00 (10.130,53 − 10.108,53). Em `Credit`, a PCR não reduz a dimensão: o melhor modelo usa as 11 componentes.

O que muda de $M = 9$ para $M = 10$ aparece nos coeficientes. De cada ajuste com os 400 clientes saem os $\beta_j = \sum_m\theta_m\phi_{jm}$, reconstruídos a partir dos $\theta_m$ e das cargas: são os coeficientes nas colunas padronizadas, comparáveis aos da ridge.

In [ ]:
modelos = {M: make_pipeline(StandardScaler(), PCA(n_components=M),
                            LinearRegression()).fit(X, y)
           for M in Ms_credit}
# β_j = soma de θ_m φ_jm: cargas transpostas vezes os θ
coef_pcr = pd.DataFrame({
    M: modelo[1].components_.T @ modelo[-1].coef_
    for M, modelo in modelos.items()
}, index=X.columns).T
coef_pcr.index.name = "M"
destaque = {"renda": "C0", "limite": "C1",
            "pontuacao": "C2", "estudante_sim": "C3"}
outras = coef_pcr.drop(columns=list(destaque))
print("algum β_j igual a zero:", bool(np.isclose(coef_pcr, 0).any()))
print("outras sete, de", numero_br(outras.min().min(), 1),
      "a", numero_br(outras.max().max(), 1))
diferenca = (coef_pcr.loc[7:9, "renda"]
             - coef_pcr.loc[7:9, "estudante_sim"]).abs().max()
print("renda × estudante_sim, M = 7 a 9:")
print("  maior diferença:", numero_br(diferenca, 1))
coef_pcr.loc[[1, 9, 10, 11], list(destaque)].style.set_uuid(
    "coef-pcr").format(lambda v: numero_br(v, 1))

> **🔧 Função**
>
> - **`modelo[1]`**: o segundo passo do `Pipeline`, a `PCA` ajustada; `components_.T @ coef_` multiplica as cargas pelos $\theta_m$ da regressão e dá um $\beta_j$ por coluna original.
> - **`df.drop(columns=lista)`**: uma cópia de `df` sem as colunas de `lista`; numa `Series`, **`serie.drop(rotulos)`** tira os elementos com esses rótulos.

In [ ]:
# Figura: 'A PCR em `Credit`, contra o número de componentes $M$. Em cima, os coeficientes padronizados $\beta_j$ reconstruídos das componentes, com `renda`, `limite` (tracejada, por cima), `pontuacao` e `estudante_sim` (mais grossa, por baixo) em cor; de $M = 7$ a $M = 9$, a linha de `renda` corre sobre a de `estudante_sim`, a no máximo 3,0 dela; embaixo, o erro de validação cruzada em 10 grupos. Em $M = 11$, a PCR é o ajuste dos mínimos quadrados.'
fig, (ax_coef, ax_cv) = plt.subplots(
    2, 1, figsize=(4.8, 5.8), sharex=True, layout="constrained"
)
tracejado = (0, (4, 3))
# estudante_sim, mais grossa e por baixo, fica visível como borda da
# renda de M = 7 a 9, onde as duas quase coincidem
espessura = {"estudante_sim": 4}
camada = {"limite": 5, "renda": 4, "pontuacao": 3, "estudante_sim": 2}
for coluna in X.columns:
    em_destaque = coluna in destaque
    ax_coef.plot(coef_pcr.index, coef_pcr[coluna],
                 color=destaque.get(coluna, "0.45"),
                 linewidth=espessura.get(coluna, 2 if em_destaque else 1),
                 linestyle=tracejado if coluna == "limite" else "-",
                 zorder=camada.get(coluna, 1))
ax_coef.axhline(0, color="0.45", linewidth=0.8)
ax_coef.set_ylim(-320, 480)
ax_coef.set_ylabel("coeficiente padronizado")
ax_cv.plot(cv_pcr.index, cv_pcr / 1000, color="0.45", linewidth=2,
           marker="o", markersize=5)
ax_cv.set_ylim(0, 95)
ax_cv.set_ylabel("erro de validação\ncruzada (dólares² ÷ 1.000)")
ax_cv.set_xticks(list(Ms_credit))
ax_cv.set_xlabel("número de componentes, M")
entradas = [Line2D([], [], color=c, linewidth=espessura.get(nome, 2),
                   linestyle=tracejado if nome == "limite" else "-")
            for nome, c in destaque.items()]
entradas.append(Line2D([], [], color="0.45", linewidth=1))
fig.legend(entradas, [*destaque, "outras sete"],
           loc="outside lower center", ncols=3, frameon=False,
           fontsize="small")
plt.show()

Na tabela de coeficientes, acima da figura, com $M = 1$ e com $M = 9$, os coeficientes de `renda`, `limite` e `pontuacao` são positivos e parecidos, de 113,2 a 134,4. Com $M = 10$, o de `renda` troca de sinal e vai a −275,0, quase o −274,7 dos mínimos quadrados. `estudante_sim` fica em destaque porque, com $M = 10$ e $M = 11$, o coeficiente dela, 126,9 e 127,7, passa longe do das outras sete colunas, que ficam entre −60,3 e 50,4 em todos os $M$. A troca de sinal de `renda` vem da 10.ª componente.

In [ ]:
pca_credit = modelos[11][1]
escores_credit = modelos[11][:2].transform(X)
componentes = pd.DataFrame({
    "variância de X": pca_credit.explained_variance_ratio_,
    "|corr. com saldo|": [
        abs(np.corrcoef(escores_credit[:, m], y)[0, 1])
        for m in range(X.shape[1])
    ],
}, index=range(1, X.shape[1] + 1))
componentes.index.name = "componente"
cargas10 = pd.Series(pca_credit.components_[9], index=X.columns)
ordem = componentes["|corr. com saldo|"].sort_values(ascending=False)
print("mais correlacionadas:", *ordem.index[:2])
print("|cargas| da 10.ª:")
for coluna in ["renda", "limite", "pontuacao"]:
    print(f"  {coluna}: {numero_br(abs(cargas10[coluna]), 3)}")
print("maior |carga| nas outras:",
      numero_br(cargas10.drop(["renda", "limite", "pontuacao"])
                .abs().max(), 3))
print("renda contra limite:",
      bool(np.sign(cargas10["renda"]) != np.sign(cargas10["limite"])))
print("renda contra pontuacao:",
      bool(np.sign(cargas10["renda"])
           != np.sign(cargas10["pontuacao"])))
cargas11 = pd.Series(pca_credit.components_[10], index=X.columns)
print("|cargas| da 11.ª:",
      numero_br(abs(cargas11["limite"]), 3),
      numero_br(abs(cargas11["pontuacao"]), 3))
print("limite contra pontuacao:",
      bool(np.sign(cargas11["limite"])
           != np.sign(cargas11["pontuacao"])))
componentes.style.set_uuid("componentes").format(
    lambda v: numero_br(v, 3))

> **🔧 Função**
>
> **`modelo[:2]`**: um `Pipeline` só com os dois primeiros passos, já ajustados, o `StandardScaler` e a `PCA`; com `.transform(X)`, dá os escores das componentes.

As componentes vêm em ordem decrescente de variância, e a 10.ª carrega só 0,024 da variância de $X$. Mesmo assim, ela é a segunda mais correlacionada com `saldo`, com 0,517 em valor absoluto, atrás só da 1.ª, com 0,762. As cargas dela põem `renda`, com 0,836, de um lado, e `limite` e `pontuacao`, com 0,379 e 0,374, do outro, com o sinal oposto; nas outras oito colunas, nenhuma carga passa de 0,103 em valor absoluto. O escore mede o quanto a renda do cliente se afasta do que o `limite` e a `pontuacao` dele sugeririam.

É essa direção que dá à `renda` o coeficiente negativo dos mínimos quadrados: entre clientes com o mesmo limite, a mesma pontuação e as demais colunas iguais, mais renda vai com menos `saldo`. Ela fica no fim da ordem porque `renda`, `limite` e `pontuacao` andam juntas (a correlação entre `renda` e `limite` é 0,792), e o afastamento entre elas tem pouca variância, os 0,024. Pouca variância, mas muita relação com `saldo`.

A 11.ª componente é a diferença entre `limite` e `pontuacao`, com cargas de 0,705 e 0,708 e sinais opostos, e carrega uma fração da variância que a tabela arredonda para 0,000, porque as duas colunas, com correlação de 0,997, quase se repetem. Com ela, os coeficientes das duas, 308,3 e 307,9 com $M = 10$, se separam em 440,1 e 175,6, e o erro desce os 22,00 da conta acima.

Trocar $p$ colunas por $M$ combinações corta a variância, mas a PCR escolhe as direções pela variância de $X$, sem olhar $Y$. Com três fatores escondidos, as direções de maior variância eram as da $f$; com colunas independentes e em `Credit`, não eram.

Lado a lado, os quatro métodos para colunas que repetem informação:

| Método | O que faz com as colunas |
|---|---|
| seleção de subconjuntos | escolhe algumas e descarta as outras |
| ridge | mantém todas e encolhe os coeficientes |
| lasso | encolhe os coeficientes e leva alguns a zero, o que descarta colunas |
| PCR | troca as colunas por $M$ combinações de todas elas |

Como a ridge, a PCR mantém todas as colunas originais no modelo: cada componente é uma combinação delas, e em `Credit` nenhum $\beta_j$ da PCR é zero, nem com $M = 1$. Ela funciona quando as direções em que $X$ mais varia são as que carregam a $f$; quando a $f$ depende de uma direção de pouca variância, ela precisa de quase todas as componentes, e a redução de dimensão se perde.

## O que Muda em Alta Dimensão

> **📌 Nota**
>
> Esta seção corresponde à seção 6.4 de James et al. (2023).

Para prever a pressão arterial de um paciente, a idade, o sexo e o índice de massa corporal são três preditores, e um hospital pode ter essas medidas de milhares de pacientes. Um exame genético mede muito mais: meio milhão de marcadores no DNA de cada pessoa, num estudo que talvez tenha 200 pacientes. Aí há mais colunas que observações. O que acontece com os mínimos quadrados quando o número de preditores $p$ passa do número de observações $n$, ou chega perto dele? E o que a regularização ainda consegue fazer nesse caso?

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.linear_model import Lasso, LinearRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

plt.style.use("estilo-figuras.mplstyle")

numero_br = lambda valor, casas=2: (
    f"{valor:,.{casas}f}".replace(",", "_").replace(".", ",").replace("_", ".")
    .replace("-", "−")
)
virgula = lambda casas: (
    lambda valor, _: f"{valor:.{casas}f}".replace(".", ",").replace("-", "−")
)
contagem = lambda valor: f"{valor:g}".replace(".", ",")   # 98 ou 67,5

### Mais colunas que observações

No exame genético, $p$ passa de $n$. Outro caso comum são os termos de busca: cada termo que alguém digitou num buscador vira uma coluna, com 1 se a pessoa o usou e 0 se não, e as colunas chegam a dezenas de milhares com os dados de poucos milhares de usuários.

> **🔷 Conceito**
>
> Um conjunto de dados é de **alta dimensão** quando tem mais preditores que observações, $p > n$; a dimensão, aqui, é o número de preditores $p$. Quando $n$ é bem maior que $p$, o conjunto é de **baixa dimensão**. Os efeitos medidos a seguir já aparecem quando $p$ chega perto de $n$.

### Mínimos quadrados com muitas colunas

#### Dois pontos, uma reta

Com um preditor, a regressão por mínimos quadrados estima dois coeficientes, o intercepto e a inclinação. Com 20 pontos, a reta passa entre eles. Com dois pontos de $x$ diferentes, sempre existe uma reta que passa exatamente pelos dois, e o RSS dela é zero: aí $p = 1$ e $n = 2$, e há $p + 1 = n$ coeficientes para $n$ pontos. Na célula, os 20 pontos vêm de $f(x) = 1 + 2x$ mais um ruído de desvio padrão 2, e a segunda reta é ajustada só com os dois primeiros. Antes de rodar a célula: qual das duas retas você espera que erre menos em pontos novos?

In [ ]:
rng = np.random.default_rng(11)
x = rng.normal(size=(20, 1))                  # 20 valores de X
# f(x) = 1 + 2x, com ruído de desvio padrão 2
y = 1 + 2 * x[:, 0] + rng.normal(0, 2, 20)
x_novo = rng.normal(size=(10_000, 1))         # pontos novos, para o teste
y_novo = 1 + 2 * x_novo[:, 0] + rng.normal(0, 2, 10_000)
reta20 = LinearRegression().fit(x, y)
reta2 = LinearRegression().fit(x[:2], y[:2])  # só os dois primeiros pontos
rss2 = ((y[:2] - reta2.predict(x[:2])) ** 2).sum()
print("RSS da reta de 2 pontos")
print("  abaixo de 10⁻⁶:", bool(rss2 < 1e-6))
for nome, reta in [("20 pontos", reta20), ("2 pontos", reta2)]:
    mse = ((y_novo - reta.predict(x_novo)) ** 2).mean()
    print(f"MSE de teste, {nome}:", numero_br(mse))

In [ ]:
# Figura: 'Mínimos quadrados com um preditor. Em cima, a reta laranja ajustada com os 20 pontos azuis. Embaixo, a reta ajustada só com os dois pontos azuis, que passa exatamente pelos dois; os círculos vazados cinza são os outros 18 pontos, que não entraram nesse ajuste. Os dois painéis têm os mesmos eixos.'
fig, (ax20, ax2) = plt.subplots(2, 1, figsize=(4.8, 5.2), sharex=True,
                                sharey=True, layout="constrained")
grade_x = np.array([[-2.2], [1.9]])
ax20.scatter(x[:, 0], y, s=30, color="C0", zorder=3)
ax20.plot(grade_x[:, 0], reta20.predict(grade_x), color="C1", linewidth=2)
ax2.scatter(x[2:, 0], y[2:], s=30, facecolors="none", edgecolors="0.45")
ax2.scatter(x[:2, 0], y[:2], s=45, color="C0", zorder=3)
ax2.plot(grade_x[:, 0], reta2.predict(grade_x), color="C1", linewidth=2)
ax20.set_title("ajustada com os 20 pontos", loc="left")
ax2.set_title("ajustada com os 2 pontos azuis", loc="left")
for ax in (ax20, ax2):
    ax.set_ylabel("Y")
    ax.xaxis.set_major_formatter(virgula(1))
    ax.yaxis.set_major_formatter(virgula(0))
ax2.set_xlim(-2.2, 1.9)
ax2.set_xlabel("X")
plt.show()

> **🔧 Função**
>
> **`ax.scatter(x, y, facecolors="none", edgecolors=cor)`**: pontos vazados, só com o contorno na cor dada.

O RSS da reta de dois pontos fica abaixo de um milionésimo: ela passa pelos dois. Em 10.000 pontos novos, o MSE de teste dela é 7,86, contra 4,45 da reta ajustada com os 20; o piso é a variância do ruído, $2^2 = 4$. Com dois pontos, a reta não tem como separar a $f$ do ruído, e segue o ruído dos dois pontos que recebeu. Com outros dois pontos, a reta seria outra.

#### Colunas de ruído puro

Com $p$ preditores, os mínimos quadrados estimam $p + 1$ coeficientes, os $p$ das colunas e o intercepto. Quando $p + 1 \ge n$, há coeficientes bastantes para o ajuste passar por todos os $n$ pontos, como a reta passou pelos dois.

Para ver o que isso faz sem uma $f$ no meio, a resposta agora é ruído puro. O desenho:

- $n = 20$ valores de $y$ sorteados da normal padrão, e 20 colunas sorteadas do mesmo jeito, sem relação nenhuma com $y$;
- o modelo com $p$ colunas usa as $p$ primeiras, de $p = 1$ a $p = 20$;
- o teste usa 10.000 pontos novos, sorteados da mesma forma;
- como $y$ não depende de nenhuma coluna, o melhor modelo prevê a média de $y$, zero, e o erro de teste dele é a variância do ruído, 1: é o piso.

Antes de rodar a célula: o que você espera do $R^2$ de treino com $p = 20$?

In [ ]:
rng = np.random.default_rng(11)
n = 20
X_ruido = rng.normal(size=(n, 20))      # 20 colunas de ruído
y_ruido = rng.normal(size=n)            # a resposta também é ruído
X_ruido_teste = rng.normal(size=(10_000, 20))
y_ruido_teste = rng.normal(size=10_000)
linhas = {}
with warnings.catch_warnings(record=True) as avisos:
    warnings.simplefilter("always")
    for p in range(1, 21):
        modelo = LinearRegression().fit(X_ruido[:, :p], y_ruido)
        previsto = modelo.predict(X_ruido[:, :p])
        previsto_teste = modelo.predict(X_ruido_teste[:, :p])
        linhas[p] = {
            "R² de treino": modelo.score(X_ruido[:, :p], y_ruido),
            "MSE de treino": ((y_ruido - previsto) ** 2).mean(),
            "MSE de teste": ((y_ruido_teste - previsto_teste) ** 2).mean(),
        }
ruido = pd.DataFrame(linhas).T
ruido.index.name = "p"
teste = ruido["MSE de teste"]
print("avisos nos 20 ajustes:", len(avisos))
print("R² = 1 com p = 19 e 20:",
      bool(np.allclose(ruido.loc[19:, "R² de treino"], 1)))
print("MSE de treino abaixo de 10⁻⁶")
print("  com p = 19 e 20:",
      bool((ruido.loc[19:, "MSE de treino"] < 1e-6).all()))
print("teste, p = 20 acima de p = 1:", bool(teste[20] > teste[1]))
print("teste só sobe com p:", bool((teste.diff().dropna() > 0).all()))
print("maior MSE de teste em p =", teste.idxmax())
ruido.loc[[1, 5, 10, 15, 17, 18, 19, 20]].style.set_uuid(
    "ruido-puro").format(lambda v: numero_br(v, 4))

> **🔧 Função**
>
> - **`warnings.catch_warnings(record=True)`**: dentro do bloco `with`, guarda numa lista, aqui `avisos`, os avisos que o código emitir, em vez de mostrá-los. **`warnings.simplefilter("always")`** faz todo aviso entrar na lista, mesmo repetido.
> - **`ruido.loc[19:, coluna]`**: as linhas de rótulo 19 em diante; com `.loc`, o rótulo final da fatia entra.

A tabela mostra oito dos 20 valores de $p$; a figura mostra todos.

In [ ]:
# Figura: 'Regressão por mínimos quadrados com $n = 20$ observações e $p$ colunas de ruído, de 1 a 20, sem relação nenhuma com a resposta. De cima para baixo, o $R^2$ de treino, o MSE de treino e o MSE de teste, medido em 10.000 pontos novos. A linha vertical pontilhada marca $p = 19$: dali em diante, $p + 1 \ge n$; nesta amostra, o $R^2$ de treino já é 0,9997 em $p = 18$. No painel de baixo, a linha horizontal tracejada é o piso, a variância do ruído, 1.'
fig, eixos = plt.subplots(3, 1, figsize=(4.8, 6.4), sharex=True,
                          layout="constrained")
formatos = {"R² de treino": 2, "MSE de treino": 1, "MSE de teste": 0}
for ax, (coluna, casas) in zip(eixos, formatos.items()):
    ax.axvline(19, color="0.45", linestyle=":", linewidth=1.2, zorder=1)
    ax.plot(ruido.index, ruido[coluna], color="C0", linewidth=2,
            marker="o", markersize=4, zorder=3, clip_on=False)
    ax.set_ylabel(coluna)
    ax.yaxis.set_major_formatter(virgula(casas))
eixos[2].axhline(1, color="0.45", linestyle="--", linewidth=1.2, zorder=1)
eixos[2].text(18.7, 1.15, "piso = 1", ha="right", va="bottom", color="0.45")
eixos[0].set_ylim(0, 1.05)
eixos[1].set_ylim(bottom=0)
eixos[2].set_ylim(0, 7)
eixos[-1].set_xlim(0, 21)
eixos[-1].set_xticks([1, 5, 10, 15, 20])
eixos[-1].set_xlabel("número de colunas de ruído, p")
plt.show()

O `LinearRegression` não reclama: os 20 ajustes terminam sem nenhum aviso, inclusive com $p = 20$, quando há 21 coeficientes para 20 observações. Nesse caso, infinitas combinações de coeficientes passam por todos os pontos, e o ajuste devolve uma delas.

O $R^2$ de treino sobe de 0,0113, com $p = 1$, até 1, com $p = 19$ e $p = 20$, e o MSE de treino cai de 0,5351 até ficar abaixo de um milionésimo. São 20 colunas de ruído, e quem olhasse só o treino concluiria que elas explicam $y$ por inteiro. O MSE de teste diz o contrário: 1,0099 com $p = 1$, perto do piso de 1, e 5,8985 com $p = 20$, cerca de 5,8 vezes mais (5,8985 ÷ 1,0099).

O teste não sobe a cada coluna, como mostra o `False` de "teste só sobe com p": o maior valor está em $p = 18$, 6,4075, e com $p = 20$ o erro é um pouco menor. A tabela vem de uma amostra só de 20 linhas, e o valor exato de cada $p$ depende dela. A próxima célula mede o motivo da subida: quanto os coeficientes mudam de uma amostra para outra.

#### Coeficientes que mudam com a amostra

Por que o teste piora? Cada coluna nova é mais um coeficiente estimado com os mesmos 20 pontos. Para medir o quanto uma estimativa muda de uma amostra para outra, a célula sorteia 200 amostras novas de treino, cada uma com 20 linhas e 19 colunas de ruído, e ajusta em cada uma os modelos de $p = 1$ a $p = 19$. O que se acompanha é $\hat\beta_1$, o coeficiente da primeira coluna, que está em todos os modelos e cujo valor verdadeiro é zero. A medida de quanto ele varia é a distância entre o primeiro e o terceiro quartil dos 200 valores: a faixa que guarda a metade central deles.

In [ ]:
rng = np.random.default_rng(12)
registros = []
for amostra in range(200):              # 200 amostras novas de treino
    X_rep = rng.normal(size=(n, 19))
    y_rep = rng.normal(size=n)
    coeficientes = {}
    for p in range(1, 20):
        modelo = LinearRegression().fit(X_rep[:, :p], y_rep)
        coeficientes[p] = modelo.coef_[0]      # β̂₁
    registros.append(coeficientes)
beta1 = pd.DataFrame(registros)         # uma linha por amostra, uma coluna por p
entre_quartis = beta1.quantile(0.75) - beta1.quantile(0.25)
print("distância entre os quartis de β̂₁:")
for p in [1, 10, 17, 18, 19]:
    print(f"  p = {p:>2}: {numero_br(entre_quartis[p], 3)}")
print("maior |β̂₁| nas 200 amostras:")
for p in [1, 19]:
    print(f"  p = {p:>2}: {numero_br(beta1[p].abs().max(), 1)}")
print("|β̂₁| acima de 5, p = 19:", int((beta1[19].abs() > 5).sum()))

> **🔧 Função**
>
> **`df.quantile(q)`**: para cada coluna de `df`, o valor abaixo do qual fica a fração `q` dos dados; `q = 0.25` dá o primeiro quartil, e `q = 0.75`, o terceiro.

A distância entre os quartis de $\hat\beta_1$ vai de 0,305 com $p = 1$ a 0,978 com $p = 17$ e 2,397 com $p = 19$, quando $p + 1 = n$: perto de oito vezes a de $p = 1$ nesta simulação (2,397 ÷ 0,305). Fora da metade central, há amostras em que o valor dispara: o maior $|\hat\beta_1|$ das 200 é 0,8 com $p = 1$ e 64,1 com $p = 19$. A distância entre quartis olha só a metade central, e esses valores extremos não a puxam.

In [ ]:
# Figura: 'O coeficiente $\hat\beta_1$ da primeira coluna de ruído nas 200 amostras de treino, com $n = 20$. Em cima, o modelo com $p = 1$; embaixo, com $p = 19$. As linhas verticais laranja marcam o primeiro e o terceiro quartil; entre elas fica a metade central dos 200 valores. O valor verdadeiro é zero. Os valores de $|\hat\beta_1|$ acima de 5, que só aparecem com $p = 19$, ficam fora do gráfico.'
fig, eixos = plt.subplots(2, 1, figsize=(4.8, 4.6), sharex=True,
                          sharey=True, layout="constrained")
faixas = np.arange(-5, 5.01, 0.25)
for ax, p in zip(eixos, [1, 19]):
    ax.hist(beta1[p], bins=faixas, color="C0", edgecolor="white",
            linewidth=0.5)
    for q in (0.25, 0.75):
        ax.axvline(beta1[p].quantile(q), color="C1", linewidth=2)
    ax.set_title(f"p = {p}", loc="left")
    ax.set_ylabel("amostras")
eixos[-1].set_xlim(-5, 5)
eixos[-1].xaxis.set_major_formatter(virgula(0))
eixos[-1].set_xlabel("β̂₁ (valor verdadeiro: 0)")
plt.show()

> **🔧 Função**
>
> **`ax.hist(valores, bins=faixas)`**: o histograma de `valores`, com uma barra para cada intervalo entre dois limites consecutivos de `faixas`; a altura é quantos valores caem nele. Valores fora do primeiro e do último limite ficam de fora.

Na figura, o coeficiente verdadeiro é zero nos dois painéis; o que cresce com $p$ é o quanto a estimativa depende da amostra. Com $p = 19$, 26 das 200 amostras dão $|\hat\beta_1|$ acima de 5 e nem cabem no gráfico. Coeficientes que mudam tanto dão previsões que mudam também: é a variância, um dos dois lados da troca entre viés e variância da seção 7.6, e aqui ela não traz nenhuma redução de viés em troca, porque nenhuma coluna tem relação com $y$.

#### O $C_p$, o BIC e o $R^2$ ajustado

O erro de treino não serve para comparar modelos com números diferentes de colunas. Os critérios da seção 11.2 corrigem isso: servem também quando $p$ chega perto de $n$? Os três se apoiam numa estimativa da variância do ruído feita com o próprio treino:

$$
\hat\sigma^2 = \frac{\mathrm{RSS}_p}{n - p - 1},
$$

em que $\mathrm{RSS}_p$ é o RSS do modelo com todas as $p$ colunas. O $C_p$ e o BIC de um modelo com $d$ colunas somam ao RSS dele uma penalidade proporcional a $\hat\sigma^2$. O $R^2$ ajustado é

$$
R^2_{\text{ajustado}} = 1 - \frac{\mathrm{RSS}/(n - d - 1)}{\mathrm{TSS}/(n - 1)},
$$

e no modelo com todas as colunas, $d = p$, o numerador da fração é o próprio $\hat\sigma^2$. Com $n = 20$ e $p = 19$, o denominador $n - p - 1$ é zero. Com $p = 18$, é 1, e $\hat\sigma^2$ é o próprio RSS.

A célula sorteia 200 conjuntos de ruído puro, cada um com $n = 20$ linhas e 18 colunas. Cada valor de $p$ usa as $p$ primeiras colunas de cada amostra: com $p = 1$, só a primeira; com $p = 18$, todas. Em cada caso, a célula ajusta o modelo com as $p$ colunas e calcula $\hat\sigma^2$ e o $R^2$ ajustado. A variância verdadeira do ruído é 1, e nenhuma coluna tem relação com $y$. A tabela dá a mediana dos 200 valores e os quantis de 10% e 90%, que cercam os 80% centrais. E de 18 colunas de ruído, que $R^2$ ajustado você espera?

In [ ]:
rng = np.random.default_rng(13)
registros = []
for amostra in range(200):              # 200 amostras de ruído puro, n = 20
    X_rep = rng.normal(size=(n, 18))
    y_rep = rng.normal(size=n)
    tss = ((y_rep - y_rep.mean()) ** 2).sum()
    for p in [1, 18]:                   # o modelo com todas as p colunas
        modelo = LinearRegression().fit(X_rep[:, :p], y_rep)
        rss = ((y_rep - modelo.predict(X_rep[:, :p])) ** 2).sum()
        sigma2 = rss / (n - p - 1)
        registros.append({"p": p, "σ̂²": sigma2,
                          "R² ajustado": 1 - sigma2 / (tss / (n - 1))})
criterios = pd.DataFrame(registros)
linhas_q = []
for p in [1, 18]:
    grupo = criterios[criterios["p"] == p]
    for nome, q in [("10%", 0.1), ("mediana", 0.5), ("90%", 0.9)]:
        linhas_q.append({"p": p, "quantil": nome,
                         "σ̂²": grupo["σ̂²"].quantile(q),
                         "R² ajustado": grupo["R² ajustado"].quantile(q)})
print("n − p − 1 com p = 19:", n - 19 - 1)
pd.DataFrame(linhas_q).set_index(["p", "quantil"]).style.set_uuid(
    "criterios").format(lambda v: numero_br(v, 3))

Com $p = 1$, os dois critérios fazem o que se espera: $\hat\sigma^2$ fica em torno da variância verdadeira, 1, e o $R^2$ ajustado em torno de zero, como deve ser para uma coluna de ruído.

Com $p = 18$, a mediana do $R^2$ ajustado é 0,430: em metade das amostras, 18 colunas de ruído passam de 0,430 no critério que devia descontar o número de colunas. Em um décimo delas, passam de 0,976.

A causa está em $\hat\sigma^2$, que erra para os dois lados: os 80% centrais vão de 0,028 a 3,101. A distribuição é assimétrica, e mais da metade das amostras fica abaixo de 1, com mediana 0,467. Pela fórmula, sempre que $\hat\sigma^2$ sai abaixo da variância amostral de $y$, $\mathrm{TSS}/(n - 1)$, o $R^2$ ajustado sai positivo. Com $p$ perto de $n$, a estimativa em que os critérios se apoiam muda tanto de uma amostra para outra que a correção deixa de valer, e com $p = 19$ nem se calcula. O que avalia o modelo, então, é o erro em observações que não entraram no ajuste, num conjunto de teste ou na validação cruzada.

### Regularizar com $p$ grande

A seleção passo a passo para frente, a ridge, o lasso e a PCR ajustam modelos menos flexíveis que os mínimos quadrados, e é isso que os torna úteis quando $p$ é grande. Até quantas colunas de ruído eles ainda ajudam? A célula mede só o lasso, como representante da família; a seleção, a ridge e a PCR não são testadas aqui. Ele aparece lado a lado com os mínimos quadrados e com o modelo nulo, que prevê a média de $y$ do treino. O desenho:

- $n = 100$ observações de treino e 1.000 de teste;
- as 20 primeiras colunas têm coeficiente 1, e as outras, zero; o ruído tem desvio padrão 1, e o piso do erro de teste é 1;
- $p$ vale 20, 50 ou 2.000: 0, 30 ou 1.980 colunas de ruído, sempre com os mesmos 20 sinais;
- para cada $p$, 50 amostras de treino, e em cada uma o lasso com $\lambda = 2$, 20 e 60 (com `alpha` $= \lambda/(2n)$, a conversão da seção 11.4).

Os três valores de $\lambda$ dão lassos cada vez mais rígidos; a tabela mostra quantos coeficientes cada um deixa diferentes de zero em cada $p$. Aqui, o melhor deles é apontado pelo erro de teste, só para comparar; num problema real, $\lambda$ se escolhe por validação cruzada, como na seção 11.5.

Com $p = 2.000$, a célula leva mais tempo que as outras. Pense antes de ver a saída: com 1.980 colunas de ruído, você espera que um $\lambda$ maior ajude ou atrapalhe?

In [ ]:
rng = np.random.default_rng(11)
n_l, n_teste, sinais = 100, 1000, 20
lambdas = [2, 20, 60]
linhas, referencias, interpola = [], [], []
for p in [20, 50, 2000]:
    beta = np.zeros(p)
    beta[:sinais] = 1.0                 # só as 20 primeiras importam
    for _ in range(50):                 # 50 amostras de treino
        X_l = rng.normal(size=(n_l, p))
        y_l = X_l @ beta + rng.normal(size=n_l)
        X_t = rng.normal(size=(n_teste, p))
        y_t = X_t @ beta + rng.normal(size=n_teste)
        mq = LinearRegression().fit(X_l, y_l)
        if p > n_l:                     # os mínimos quadrados passam pelos pontos?
            residuo = np.abs(y_l - mq.predict(X_l)).max()
            interpola.append(residuo < 1e-6)
        referencias.append({
            "p": p,
            "nulo": ((y_t - y_l.mean()) ** 2).mean(),
            "mínimos quadrados": ((y_t - mq.predict(X_t)) ** 2).mean(),
        })
        for lam in lambdas:
            modelo = make_pipeline(
                StandardScaler(),
                Lasso(alpha=lam / (2 * n_l), max_iter=100_000, tol=1e-6),
            ).fit(X_l, y_l)
            coef = modelo[-1].coef_
            linhas.append({
                "p": p, "λ": lam,
                "MSE de teste": ((y_t - modelo.predict(X_t)) ** 2).mean(),
                "não nulos": np.count_nonzero(coef),
                "sinais não nulos": np.count_nonzero(coef[:sinais]),
            })
lasso_p = pd.DataFrame(linhas)
referencias = pd.DataFrame(referencias)

A célula seguinte resume as 50 amostras de cada caso pela mediana.

In [ ]:
medianas = lasso_p.groupby(["p", "λ"]).median()
mse_lasso = medianas["MSE de teste"].unstack()   # uma linha por p, uma coluna por λ
medianas_ref = referencias.groupby("p").median()
print("mediana do MSE de teste, sem lasso:")
for p in medianas_ref.index:
    print(f"  p = {numero_br(p, 0)}:")
    print(f"    nulo {numero_br(medianas_ref.loc[p, 'nulo'])}")
    mq_p = medianas_ref.loc[p, "mínimos quadrados"]
    print(f"    mínimos quadrados {numero_br(mq_p)}")
print("λ de menor mediana, por p:")
for p in mse_lasso.index:
    print(f"  p = {numero_br(p, 0)}: λ = {mse_lasso.loc[p].idxmin()}")
lasso_2000, ref_2000 = mse_lasso.loc[2000], medianas_ref.loc[2000]
print("p = 2.000, mínimos quadrados passam")
print("  pelos pontos de treino nas 50 amostras:", all(interpola))
print("p = 2.000, lasso com os três λ:")
print("  abaixo dos mínimos quadrados:",
      bool((lasso_2000 < ref_2000["mínimos quadrados"]).all()))
print("  abaixo do nulo:", bool((lasso_2000 < ref_2000["nulo"]).all()))
print("  menor mediana acima de 10:", bool(lasso_2000.min() > 10))
medianas.rename(index=lambda v: numero_br(v, 0)).style.set_uuid(
    "lasso-p").format({"MSE de teste": numero_br, "não nulos": contagem,
                       "sinais não nulos": contagem})

> **🔧 Função**
>
> - **`np.count_nonzero(arr)`**: quantos elementos de `arr` são diferentes de zero.
> - **`serie.unstack()`**: numa série com índice de dois níveis, passa o segundo nível para as colunas; aqui, uma linha por $p$ e uma coluna por $\lambda$.
> - **`df.rename(index=funcao)`**: troca cada rótulo do índice pelo que `funcao` devolve para ele; aqui, só para exibir 2000 como 2.000.

In [ ]:
# Figura: 'O lasso e os mínimos quadrados com $n = 100$ observações, 20 sinais e $p$ igual a 20, 50 e 2.000 colunas, de cima para baixo. Cada caixa azul resume o MSE de teste nas 50 amostras de treino com um valor de $\lambda$, e o número embaixo dela é a mediana do número de coeficientes não nulos; a caixa cinza, à direita, é a dos mínimos quadrados. O traço laranja dentro de cada caixa é a mediana. A linha tracejada cinza, rotulada no painel de cima, é o piso, a variância do ruído, 1. As escalas verticais dos três painéis são diferentes.'
fig, eixos = plt.subplots(3, 1, figsize=(4.8, 7.4), layout="constrained")
nao_nulos = medianas["não nulos"].unstack()   # uma linha por p, uma coluna por λ
for ax, (p, grupo) in zip(eixos, lasso_p.groupby("p")):
    caixas = [grupo.loc[grupo["λ"] == lam, "MSE de teste"] for lam in lambdas]
    caixas.append(referencias.loc[referencias["p"] == p, "mínimos quadrados"])
    rotulos = [
        f"λ = {lam}\n{contagem(nao_nulos.loc[p, lam])}\nnão nulos"
        for lam in lambdas
    ] + ["mínimos\nquadrados"]
    desenho = ax.boxplot(
        caixas, tick_labels=rotulos, patch_artist=True, widths=0.5,
        boxprops={"facecolor": "C0", "edgecolor": "C0", "alpha": 0.6},
        medianprops={"color": "C1", "linewidth": 2},
        whiskerprops={"color": "0.45"}, capprops={"color": "0.45"},
        flierprops={"markeredgecolor": "0.45", "markersize": 4})
    desenho["boxes"][-1].set(facecolor="0.45", edgecolor="0.45")
    ax.axhline(1, color="0.45", linestyle="--", linewidth=1.2, zorder=1)
    ax.set_ylim(bottom=0)
    ax.set_title(f"p = {numero_br(p, 0)}", loc="left")
    ax.set_ylabel("MSE de teste")
    ax.yaxis.set_major_formatter(virgula(0))
eixos[0].text(3.45, 1.15, "piso = 1", ha="right", va="bottom", color="0.45")
plt.show()

> **🔧 Função**
>
> **`ax.boxplot(caixas, tick_labels, patch_artist=True)`**: um diagrama de caixa para cada série de `caixas`, com o rótulo de `tick_labels` embaixo. A caixa vai do primeiro ao terceiro quartil, o traço dentro dela é a mediana, as hastes vão até o valor mais afastado que ainda fica a menos de 1,5 vez a altura da caixa, e os pontos além delas aparecem soltos. `patch_artist=True` deixa pintar o interior das caixas com `boxprops`; o dicionário que a função devolve guarda as caixas em `"boxes"`, e `.set(facecolor=cor)` repinta uma delas.

Com $p = 20$, sem colunas de ruído, a menor mediana do MSE de teste do lasso é a do menor $\lambda$: 1,25, a 0,25 do piso, contra 1,66 com $\lambda = 20$ e 4,69 com $\lambda = 60$. Os mínimos quadrados, com 1,26, ficam praticamente empatados com $\lambda = 2$. Com 100 observações para 20 coeficientes, os mínimos quadrados já variam pouco de uma amostra para outra, e o encolhimento tem pouca variância a cortar.

Com $p = 50$, a menor mediana é a de $\lambda = 20$, 1,78, abaixo dos 2,04 dos mínimos quadrados e um pouco abaixo dos 1,83 de $\lambda = 2$, com 29 coeficientes não nulos em vez de 46; com os dois valores de $\lambda$, a mediana de sinais não nulos é 20. Com $\lambda = 60$, o erro sobe para 5,34: o $\lambda$ certo depende de quantas colunas de ruído há.

Com $p = 2.000$, há 20 vezes mais colunas que observações. Os mínimos quadrados passam por todos os pontos de treino nas 50 amostras, e o `LinearRegression` devolve uma das infinitas soluções. A mediana do erro de teste dos mínimos quadrados é 20,07, perto dos 20,84 do modelo nulo.

As medianas do lasso ficam abaixo das duas com os três valores de $\lambda$, e a menor é a de $\lambda = 60$, 16,88. O ganho sobre o modelo nulo, porém, é pequeno, 16,88 contra 20,84, e a menor mediana fica acima de 10, longe do piso de 1. Na mediana, o lasso fica com 13, 13 e 12 dos 20 sinais, com $\lambda = 2$, 20 e 60.

> **🔷 Conceito**
>
> A **maldição da dimensionalidade** (*curse of dimensionality*): acrescentar colunas não melhora o modelo por si só. Colunas com relação real com a resposta podem baixar o erro de teste; colunas de ruído dão ao ajuste mais chances de encontrar, por acaso, uma associação que só existe no treino, e fazem o erro de teste subir.

### Cuidado ao interpretar

Com $p \ge n$, uma coluna qualquer pode ser escrita exatamente como uma combinação linear das outras e do intercepto: é a colinearidade da seção 8.6 levada ao extremo. A célula confere isso nos dados de ruído puro, com 20 linhas e 20 colunas.

In [ ]:
outras, ultima = X_ruido[:, :19], X_ruido[:, 19]
r2_ultima = LinearRegression().fit(outras, ultima).score(outras, ultima)
print("coluna 20 a partir das outras 19")
print("  R² = 1:", bool(np.isclose(r2_ultima, 1)))

A 20.ª coluna de ruído é reproduzida sem erro pelas outras 19 e pelo intercepto; no VIF da seção 8.6, esse $R^2$ igual a 1 dá uma divisão por zero. Quando uma coluna qualquer é combinação das outras, os dados não dizem quais colunas de fato importam: o crédito de uma pode ir para qualquer combinação que a reproduza.

O lasso escolhe algumas colunas e zera as outras. A célula seguinte o ajusta com $p = 2.000$ e $\lambda = 60$, como acima, em duas amostras de treino novas, e compara as colunas de ruído que ele deixa diferentes de zero em cada uma. Se a escolha refletisse algo nos dados, quantas colunas de ruído você esperaria ver em comum nas duas listas?

In [ ]:
rng = np.random.default_rng(12)
p_a, sinais_a = 2000, 20
beta_a = np.zeros(p_a)
beta_a[:sinais_a] = 1.0
escolhidas = []
for amostra in range(2):                # duas amostras de treino
    X_a = rng.normal(size=(100, p_a))
    y_a = X_a @ beta_a + rng.normal(size=100)
    lasso = make_pipeline(
        StandardScaler(),
        Lasso(alpha=60 / (2 * 100), max_iter=100_000, tol=1e-6),
    ).fit(X_a, y_a)
    ruido_nao_nulo = np.flatnonzero(lasso[-1].coef_[sinais_a:])
    escolhidas.append(set(ruido_nao_nulo))
print("colunas de ruído escolhidas")
print("(p = 2.000, λ = 60):")
print("  1.ª amostra:", len(escolhidas[0]))
print("  2.ª amostra:", len(escolhidas[1]))
print("  nas duas:", len(escolhidas[0] & escolhidas[1]))   # & : a interseção

> **🔧 Função**
>
> **`np.flatnonzero(arr)`**: as posições dos elementos de `arr` diferentes de zero.

Com $p = 2.000$, o lasso ficou com 58 colunas de ruído na primeira amostra e 59 na segunda, e só 3 delas estão nas duas listas. Se as duas listas fossem sorteadas ao acaso entre as 1.980 colunas de ruído, a interseção esperada seria de 58 × 59 ÷ 1.980, cerca de 1,7 coluna: as 3 em comum estão perto do que o puro acaso daria. Um modelo escolhido em alta dimensão é um entre muitos possíveis, e outra amostra de treino escolhe outras colunas. Ele pode ser útil para prever, mas não prova que as colunas dele são as que importam.

### O que muda, então

Com $p + 1 \ge n$, os mínimos quadrados passam por todos os pontos de treino e erram muito fora deles; já perto disso, o erro de teste sobe. A regularização ainda ajuda, mas o erro dela também sobe com as colunas de ruído: com 1.980 delas, o lasso ficou longe do piso de 1.

## Leituras adicionais

- O capítulo 6 de James et al. (2023). O PDF gratuito e os dados estão no [site oficial do livro](https://www.statlearning.com).
- As seções 3.3 a 3.5 do capítulo 3 de Hastie et al. (2009), para quem quiser a seleção de subconjuntos, os métodos de encolhimento e a regressão sobre direções derivadas das colunas num tratamento mais técnico.
- [O guia de modelos lineares do scikit-learn](https://scikit-learn.org/stable/modules/linear_model.html), com `Ridge`, `Lasso` e as versões com validação cruzada embutida, `RidgeCV` e `LassoCV`.

## Referências

- **Hastie; Tibshirani; Friedman**. *The Elements of Statistical Learning*. 2nd ed. Springer. 2009.
- **James; Witten; Hastie; Tibshirani; Taylor**. *An Introduction to Statistical Learning with Applications in Python*. Springer. 2023.